# 03. MODELADO PREDICTIVO DE RETRASOS

Este notebook desarrolla la fase de modelado supervisado del TFM a partir del dataset de modelado preparado y persistido previamente. El objetivo es construir y evaluar modelos capaces de estimar, utilizando exclusivamente información disponible antes de la salida programada, si un vuelo llegará a su destino con un retraso igual o superior a 15 minutos (`ARR_DEL15 = 1`). Esta estimación se plantea como base de un sistema de alerta temprana orientado a anticipar posibles retrasos y proporcionar información útil al viajero antes de la salida del vuelo.

El diseño experimental seguirá una separación estrictamente temporal. Los vuelos correspondientes a 2022, 2023 y 2024 constituirán el período de desarrollo utilizado para el entrenamiento de los modelos, mientras que 2025 se reservará como período de validación temporal para comparar configuraciones, estudiar estrategias frente al desbalance, seleccionar hiperparámetros y establecer el criterio de decisión definitivo. Esta separación reproduce de forma más adecuada el escenario de aplicación, en el que las predicciones se realizan sobre observaciones futuras a partir de información histórica.

El período comprendido entre enero y mayo de 2026 permanecerá completamente aislado durante el proceso de desarrollo y selección. Una vez fijados el algoritmo, el procedimiento de preprocesamiento, la estrategia frente al desbalance, los hiperparámetros y el criterio de decisión, el sistema será reconstruido utilizando conjuntamente los datos correspondientes a 2022–2025. Solo entonces se realizará una única evaluación externa sobre 2026, destinada a estimar la capacidad de generalización temporal de la configuración seleccionada.

Debido al elevado volumen del dataset, el diseño experimental considerará explícitamente la eficiencia computacional. Se priorizará la lectura selectiva de columnas, el procesamiento incremental cuando resulte adecuado, el uso de representaciones dispersas para las variables categóricas y la reutilización de productos previamente calculados. Asimismo, los experimentos de mayor coste computacional se delimitarán previamente para evitar ejecuciones redundantes que no aporten nueva evidencia al proceso de selección.

El rendimiento se analizará mediante métricas adecuadas para un problema de clasificación desbalanceado, incluyendo Accuracy, Precision, Recall, F1, ROC-AUC y PR-AUC. La Accuracy no se interpretará de forma aislada, dado que la distribución de la variable objetivo presenta un predominio aproximado del 79 % de observaciones pertenecientes a la clase negativa. En el contexto del sistema de alerta temprana, Precision permitirá evaluar la fiabilidad de las alertas emitidas, mientras que Recall permitirá cuantificar la proporción de retrasos reales que el sistema consigue anticipar.

El proceso comenzará con la construcción de un baseline de referencia y continuará con una comparación inicial de diferentes familias de modelos. Posteriormente se analizará el efecto de distintas estrategias frente al desbalance y se realizará una optimización dirigida de los modelos finalistas. A partir de esta evidencia se seleccionará una configuración definitiva, se establecerá el criterio de decisión del sistema de alerta, se reconstruirá el sistema sobre el período 2022–2025 y se evaluará una única vez sobre el período externo de 2026.

De esta forma, el notebook mantiene una separación explícita entre desarrollo, selección y evaluación externa, evitando que la información correspondiente al período de test intervenga en las decisiones de modelado.

## 1. Configuración y recuperación del dataset de modelado

El inicio del proceso de modelado requiere establecer un entorno reproducible desde el que recuperar los productos generados durante la preparación de los datos. Esta separación evita repetir operaciones ya ejecutadas y permite que el notebook concentre sus recursos en las etapas propiamente experimentales.

Las rutas del proyecto proporcionarán acceso al dataset completo de modelado y a los productos persistidos correspondientes al período 2022–2025 y al test externo de enero a mayo de 2026. Sobre esta estructura se recuperará también el esquema de variables definido previamente, diferenciando la variable temporal, la variable objetivo y los predictores categóricos y numéricos seleccionados bajo el criterio de disponibilidad previa a la salida programada.

Antes de iniciar cualquier experimento se comprobará de forma consolidada que los productos necesarios se encuentran disponibles y que su estructura es compatible con la configuración esperada. Estas comprobaciones permiten detectar posibles inconsistencias antes de introducir decisiones de modelado o transformaciones aprendidas a partir de los datos.

Este bloque tiene exclusivamente una función de configuración y recuperación. No se realizarán transformaciones ni se entrenarán modelos, y la partición cronológica utilizada durante el desarrollo experimental se establecerá posteriormente.

En este bloque se abordarán progresivamente:

1. Configuración del entorno de modelado.
2. Recuperación del esquema y datasets persistidos.
3. Validación consolidada de la configuración.

Como resultado se dispondrá de una configuración validada y reutilizable sobre la que construir el protocolo experimental del notebook.

### 1.1 Configuración del entorno de modelado

La primera etapa establece las rutas y parámetros generales necesarios para acceder a los productos persistidos durante la preparación de los datos. El notebook trabajará directamente sobre estos productos, evitando repetir operaciones de limpieza, selección de variables y construcción del dataset de modelado ya realizadas y validadas previamente.

Se conservará una referencia al dataset completo de modelado, una ruta específica al conjunto persistido correspondiente a 2022–2025 y otra al período externo de enero a mayo de 2026. En esta etapa las rutas representan únicamente la organización física de los datos; la función experimental de cada período se establecerá posteriormente mediante el protocolo temporal del modelado.

Esta separación entre almacenamiento y diseño experimental resulta necesaria porque el producto 2022–2025 contiene tanto las observaciones utilizadas durante el desarrollo de los modelos como la validación temporal de 2025. Mantener ambas dimensiones diferenciadas evita confundir la estructura física de los datasets con la función metodológica asignada a cada período.

In [23]:
# ---------------------------------------------------------
# 1. Importación de librerías generales
# ---------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 2. Configuración de las rutas principales
# ---------------------------------------------------------

project_root = Path(
    r"G:\My Drive\MASTER Big Data\TFM"
)

modeling_data_path = (
    project_root
    / "data"
    / "processed"
    / "flights"
    / "modeling"
)

complete_data_path = (
    modeling_data_path
    / "complete"
)

development_data_path = (
    modeling_data_path
    / "train"
)

external_test_data_path = (
    modeling_data_path
    / "test"
)


# ---------------------------------------------------------
# 3. Presentación de la configuración
# ---------------------------------------------------------

path_configuration = pd.DataFrame(
    {
        "dataset": [
            "Complete modeling dataset",
            "Development dataset",
            "External test dataset",
        ],
        "path": [
            str(complete_data_path),
            str(development_data_path),
            str(external_test_data_path),
        ],
        "exists": [
            complete_data_path.exists(),
            development_data_path.exists(),
            external_test_data_path.exists(),
        ],
    }
)

display(path_configuration)

,dataset,path,exists
0,Complete modeling dataset,G:\My Drive\MASTER Big Data\TFM\data\processed...,True
1,Development dataset,G:\My Drive\MASTER Big Data\TFM\data\processed...,True
2,External test dataset,G:\My Drive\MASTER Big Data\TFM\data\processed...,True


### 1.2 Recuperación del esquema y datasets persistidos

Se recupera el esquema de variables definido durante la preparación del dataset de modelado. Este esquema identifica la variable temporal necesaria para establecer las particiones cronológicas, la variable objetivo del problema de clasificación y los predictores categóricos y numéricos seleccionados previamente bajo el criterio de disponibilidad antes de la salida programada del vuelo.

La variable `FL_DATE` se conservará como referencia temporal para asignar cada observación al período experimental correspondiente, mientras que `ARR_DEL15` representa la variable objetivo. Los diez predictores restantes constituyen el conjunto de información disponible para el entrenamiento y la evaluación de los modelos.

También se recuperará el inventario de archivos Parquet correspondiente al dataset completo de modelado, al producto persistido de 2022–2025 y al test externo de enero a mayo de 2026. En esta etapa únicamente se identifican los archivos disponibles, sin cargar de forma conjunta su contenido en memoria.

Esta recuperación permite reconstruir explícitamente el esquema de entrada del Notebook 03 y disponer de las referencias necesarias para realizar posteriormente lecturas selectivas y establecer las particiones temporales definidas por el protocolo experimental.

In [24]:
# ---------------------------------------------------------
# 1. Recuperación del esquema de variables
# ---------------------------------------------------------

temporal_variable = "FL_DATE"

target_variable = "ARR_DEL15"

categorical_features = [
    "MONTH",
    "DAY_OF_WEEK",
    "MKT_UNIQUE_CARRIER",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
]

numerical_features = [
    "CRS_ELAPSED_TIME",
    "DISTANCE",
]

modeling_features = (
    categorical_features
    + numerical_features
)

modeling_columns = [
    temporal_variable,
    *modeling_features,
    target_variable,
]


# ---------------------------------------------------------
# 2. Recuperación de los archivos Parquet persistidos
# ---------------------------------------------------------

complete_parquet_files = sorted(
    complete_data_path.rglob("*.parquet")
)

development_parquet_files = sorted(
    development_data_path.rglob("*.parquet")
)

external_test_parquet_files = sorted(
    external_test_data_path.rglob("*.parquet")
)


# ---------------------------------------------------------
# 3. Resumen del esquema de modelado
# ---------------------------------------------------------

schema_summary = pd.DataFrame(
    {
        "component": [
            "Variable temporal",
            "Predictores categóricos",
            "Predictores numéricos",
            "Total de predictores",
            "Variable objetivo",
            "Columnas de modelado",
        ],
        "value": [
            temporal_variable,
            len(categorical_features),
            len(numerical_features),
            len(modeling_features),
            target_variable,
            len(modeling_columns),
        ],
    }
)


# ---------------------------------------------------------
# 4. Resumen de los datasets persistidos
# ---------------------------------------------------------

persisted_datasets_summary = pd.DataFrame(
    {
        "dataset": [
            "Completo",
            "Período 2022–2025",
            "Test externo 2026",
        ],
        "parquet_files": [
            len(complete_parquet_files),
            len(development_parquet_files),
            len(external_test_parquet_files),
        ],
    }
)


# ---------------------------------------------------------
# 5. Presentación consolidada
# ---------------------------------------------------------

display(schema_summary)

display(persisted_datasets_summary)

,component,value
0,Variable temporal,FL_DATE
1,Predictores categóricos,8
2,Predictores numéricos,2
3,Total de predictores,10
4,Variable objetivo,ARR_DEL15
5,Columnas de modelado,12


,dataset,parquet_files
0,Completo,354
1,Período 2022–2025,320
2,Test externo 2026,34


#### Interpretación

El contrato de entrada del modelado se ha recuperado correctamente. El esquema mantiene `FL_DATE` como variable auxiliar para la construcción de las particiones temporales y `ARR_DEL15` como variable objetivo del problema de clasificación.

El conjunto de predictores está compuesto por diez variables: ocho categóricas y dos numéricas. Al incorporar la variable temporal y la variable objetivo, el dataset de modelado conserva un total de doce columnas, de acuerdo con el esquema definido durante la fase de preparación de los datos.

El inventario de almacenamiento también coincide con los productos persistidos previamente: 354 archivos Parquet para el dataset completo, 320 para el período de desarrollo y 34 para el test externo. Esta correspondencia permite reutilizar directamente los productos existentes sin repetir su generación.

La recuperación se ha realizado únicamente mediante el esquema y el inventario de archivos, evitando cargar innecesariamente en memoria el conjunto completo de observaciones. El siguiente paso consistirá en validar de forma consolidada que las rutas, archivos y columnas recuperados son coherentes con el contrato esperado antes de definir el nuevo protocolo experimental.

### 1.3 Validación consolidada de la configuración

Antes de definir el protocolo experimental se realiza una validación consolidada del entorno de modelado recuperado. El objetivo es comprobar que las rutas, los archivos Parquet y el contrato de variables son coherentes con los productos persistidos previamente.

La validación comprobará la disponibilidad de los tres datasets, el número esperado de archivos, la composición del conjunto de predictores y la presencia de las doce columnas requeridas en los esquemas de los conjuntos completo, de desarrollo y de test externo.

Para evitar una lectura innecesaria de millones de observaciones, la comprobación del esquema se realizará sobre un archivo representativo de cada conjunto. Dado que los productos fueron generados previamente bajo un esquema común, esta verificación resulta suficiente en esta etapa para detectar incompatibilidades estructurales antes de comenzar el diseño experimental.

El resultado será una única tabla de validación que permitirá cerrar la configuración inicial del Notebook 03.

In [25]:
# ---------------------------------------------------------
# 1. Recuperación de esquemas representativos
# ---------------------------------------------------------

complete_sample_columns = pd.read_parquet(
    complete_parquet_files[0]
).columns.tolist()

development_sample_columns = pd.read_parquet(
    development_parquet_files[0]
).columns.tolist()

external_test_sample_columns = pd.read_parquet(
    external_test_parquet_files[0]
).columns.tolist()


# ---------------------------------------------------------
# 2. Validación consolidada de la configuración
# ---------------------------------------------------------

configuration_validation = pd.DataFrame(
    {
        "validation": [
            "Complete dataset available",
            "Development dataset available",
            "External test dataset available",
            "Complete Parquet count correct",
            "Development Parquet count correct",
            "External test Parquet count correct",
            "Ten predictors defined",
            "Eight categorical predictors defined",
            "Two numerical predictors defined",
            "Twelve modeling columns defined",
            "Complete schema valid",
            "Development schema valid",
            "External test schema valid",
        ],
        "result": [
            complete_data_path.exists(),
            development_data_path.exists(),
            external_test_data_path.exists(),
            len(complete_parquet_files) == 354,
            len(development_parquet_files) == 320,
            len(external_test_parquet_files) == 34,
            len(modeling_features) == 10,
            len(categorical_features) == 8,
            len(numerical_features) == 2,
            len(modeling_columns) == 12,
            set(modeling_columns).issubset(
                complete_sample_columns
            ),
            set(modeling_columns).issubset(
                development_sample_columns
            ),
            set(modeling_columns).issubset(
                external_test_sample_columns
            ),
        ],
    }
)


# ---------------------------------------------------------
# 3. Presentación de resultados
# ---------------------------------------------------------

display(configuration_validation)

print(
    "All configuration checks valid:",
    configuration_validation["result"].all(),
)

,validation,result
0,Complete dataset available,True
1,Development dataset available,True
2,External test dataset available,True
3,Complete Parquet count correct,True
4,Development Parquet count correct,True
5,External test Parquet count correct,True
6,Ten predictors defined,True
7,Eight categorical predictors defined,True
8,Two numerical predictors defined,True
9,Twelve modeling columns defined,True


All configuration checks valid: True


#### Interpretación

La validación consolidada confirma que el entorno de modelado se encuentra correctamente configurado para iniciar el nuevo protocolo experimental. Los datasets completo, de desarrollo y de test externo están disponibles y contienen el número esperado de archivos Parquet.

El contrato de variables también resulta consistente: se mantienen diez predictores, distribuidos entre ocho variables categóricas y dos numéricas, junto con `FL_DATE` como variable temporal auxiliar y `ARR_DEL15` como variable objetivo. En consecuencia, el esquema de modelado conserva las doce columnas previstas.

La comprobación de los esquemas representativos confirma además que las variables requeridas están disponibles en los tres productos persistidos. El resultado global de la validación es satisfactorio, sin detectarse incompatibilidades estructurales que requieran modificar los datasets antes de iniciar el modelado.

Con esta comprobación queda cerrado el proceso de configuración y recuperación de los datos. El siguiente bloque establecerá formalmente la separación temporal entre entrenamiento, validación y test externo que regirá todos los experimentos posteriores.

## 2. Diseño del protocolo experimental

La naturaleza temporal del problema predictivo requiere que la evaluación respete el orden cronológico de las observaciones. Una partición aleatoria permitiría mezclar vuelos pertenecientes a distintos momentos del período analizado y no reproduciría adecuadamente el escenario de aplicación, en el que el sistema debe generar estimaciones sobre observaciones futuras a partir de información histórica.

Por este motivo, el desarrollo experimental se estructurará mediante una separación temporal. Los vuelos correspondientes a 2022–2024 se utilizarán para el entrenamiento durante la fase de selección, mientras que 2025 constituirá una validación temporal posterior sobre la que se compararán las configuraciones consideradas. El período comprendido entre enero y mayo de 2026 permanecerá aislado y se reservará exclusivamente para la evaluación externa final.

Una vez concluidas las decisiones de selección utilizando la información disponible hasta 2025, el sistema definitivo será reconstruido sobre el período completo 2022–2025. Esta reconstrucción permitirá aprovechar toda la información histórica disponible antes de realizar una única evaluación sobre el período externo de 2026.

En este bloque se abordarán progresivamente:

1. Definición de los períodos de desarrollo y test externo.
2. Definición de la partición temporal train-validation.
3. Caracterización de los conjuntos de entrenamiento y validación.
4. Validación consolidada del protocolo experimental.

Como resultado se establecerá un protocolo temporal explícito y reproducible que mantendrá separado el proceso de selección de la evaluación externa y será utilizado de forma consistente en los experimentos posteriores.

### 2.1 Definición de los períodos de desarrollo y test externo

El primer nivel de separación temporal distingue las observaciones disponibles para el proceso de desarrollo y selección de aquellas reservadas exclusivamente para evaluar la configuración definitiva.

El período comprendido entre enero de 2022 y diciembre de 2025 constituye la información histórica disponible antes del test externo. Dentro de este intervalo se diferenciarán posteriormente dos funciones experimentales: los años 2022–2024 se utilizarán para el entrenamiento durante la fase de selección, mientras que 2025 se reservará como validación temporal. Una vez finalizada esta fase y fijada la configuración definitiva, ambos períodos se integrarán para reconstruir el sistema sobre el conjunto completo 2022–2025.

El período comprendido entre enero y mayo de 2026 se reserva exclusivamente como test externo. Ninguna observación de este período podrá intervenir en el aprendizaje del preprocesamiento, la comparación de algoritmos, el tratamiento del desbalance, la selección de hiperparámetros ni el establecimiento del criterio de decisión.

Esta separación garantiza que las decisiones de modelado se adopten utilizando únicamente información anterior al período externo. De esta forma, la evaluación de 2026 permitirá analizar el comportamiento de la configuración seleccionada sobre observaciones temporalmente posteriores que no participaron en su construcción.

In [26]:
# ---------------------------------------------------------
# 1. Definición del período histórico previo al test externo
# ---------------------------------------------------------

development_start_date = pd.Timestamp(
    "2022-01-01"
)

development_end_date = pd.Timestamp(
    "2025-12-31"
)


# ---------------------------------------------------------
# 2. Definición del período de test externo
# ---------------------------------------------------------

external_test_start_date = pd.Timestamp(
    "2026-01-01"
)

external_test_end_date = pd.Timestamp(
    "2026-05-31"
)


# ---------------------------------------------------------
# 3. Resumen de los períodos experimentales generales
# ---------------------------------------------------------

experimental_periods = pd.DataFrame(
    {
        "period": [
            "Período histórico 2022–2025",
            "Test externo 2026",
        ],
        "start_date": [
            development_start_date,
            external_test_start_date,
        ],
        "end_date": [
            development_end_date,
            external_test_end_date,
        ],
        "role": [
            "Desarrollo, selección y posterior reentrenamiento definitivo",
            "Evaluación externa final exclusivamente",
        ],
    }
)


# ---------------------------------------------------------
# 4. Comprobación temporal básica
# ---------------------------------------------------------

development_precedes_external_test = (
    development_end_date
    < external_test_start_date
)


# ---------------------------------------------------------
# 5. Presentación de resultados
# ---------------------------------------------------------

display(experimental_periods)

print(
    "El período histórico precede al test externo:",
    development_precedes_external_test,
)

,period,start_date,end_date,role
0,Período histórico 2022–2025,2022-01-01,2025-12-31,"Desarrollo, selección y posterior reentrenamie..."
1,Test externo 2026,2026-01-01,2026-05-31,Evaluación externa final exclusivamente


El período histórico precede al test externo: True


#### Interpretación

La configuración temporal distingue correctamente el período histórico comprendido entre 2022 y 2025 del test externo correspondiente a enero–mayo de 2026. La comprobación devuelve `True`, confirmando que ambos intervalos son cronológicamente disjuntos.

Esta separación garantiza que las observaciones de 2026 permanezcan excluidas de las decisiones de desarrollo y selección, incluyendo el aprendizaje del preprocesamiento, el tratamiento del desbalance, la comparación de modelos, la selección de hiperparámetros y el establecimiento del criterio de decisión.

Dentro del período histórico 2022–2025 se definirá a continuación la partición experimental interna, utilizando 2022–2024 para entrenamiento y 2025 como validación temporal.

### 2.2 Definición de la partición temporal train-validation

Una vez delimitado el período histórico disponible antes del test externo, se establece la partición temporal utilizada durante el proceso de desarrollo y selección de los modelos.

Las observaciones comprendidas entre enero de 2022 y diciembre de 2024 constituirán el conjunto de entrenamiento. Durante esta fase, este período será el único utilizado para aprender los parámetros de los modelos y cualquier componente del preprocesamiento que requiera estimación a partir de los datos, evitando incorporar información procedente de observaciones temporalmente posteriores.

El año 2025 se reservará como conjunto de validación temporal. Sobre este período se evaluarán las configuraciones candidatas, incluyendo la comparación de modelos, las estrategias frente al desbalance, la selección dirigida de hiperparámetros y, posteriormente, el establecimiento del criterio de decisión definitivo. Las transformaciones aprendidas sobre 2022–2024 se aplicarán a 2025 sin reajustar sus parámetros utilizando información de este período.

Esta estructura reproduce un escenario temporal en el que el sistema se construye utilizando información histórica y se evalúa sobre observaciones posteriores. La validación de 2025 forma parte del proceso de selección y, por tanto, no constituye la evaluación final del proyecto.

Una vez fijada la configuración definitiva, el preprocesamiento y el modelo serán reconstruidos utilizando conjuntamente las observaciones de 2022–2025. El período comprendido entre enero y mayo de 2026 permanecerá reservado exclusivamente para la evaluación externa final.

De esta forma, se mantiene una separación cronológica explícita entre entrenamiento, validación temporal y test externo, reduciendo el riesgo de incorporar información futura durante el proceso de aprendizaje y selección.

In [27]:
# ---------------------------------------------------------
# 1. Definición del período de entrenamiento
# ---------------------------------------------------------

training_start_date = pd.Timestamp(
    "2022-01-01"
)

training_end_date = pd.Timestamp(
    "2024-12-31"
)


# ---------------------------------------------------------
# 2. Definición del período de validación temporal
# ---------------------------------------------------------

validation_start_date = pd.Timestamp(
    "2025-01-01"
)

validation_end_date = pd.Timestamp(
    "2025-12-31"
)


# ---------------------------------------------------------
# 3. Construcción del protocolo temporal
# ---------------------------------------------------------

temporal_protocol = pd.DataFrame(
    {
        "dataset": [
            "Training",
            "Temporal validation",
            "External test",
        ],
        "start_date": [
            training_start_date,
            validation_start_date,
            external_test_start_date,
        ],
        "end_date": [
            training_end_date,
            validation_end_date,
            external_test_end_date,
        ],
        "experimental_role": [
            "Model fitting",
            "Model selection",
            "Final external evaluation",
        ],
    }
)


# ---------------------------------------------------------
# 4. Comprobación del orden cronológico
# ---------------------------------------------------------

training_precedes_validation = (
    training_end_date
    < validation_start_date
)

validation_precedes_external_test = (
    validation_end_date
    < external_test_start_date
)

temporal_order_valid = (
    training_precedes_validation
    and validation_precedes_external_test
)


# ---------------------------------------------------------
# 5. Presentación de resultados
# ---------------------------------------------------------

display(temporal_protocol)

print(
    "Temporal order valid:",
    temporal_order_valid,
)

,dataset,start_date,end_date,experimental_role
0,Training,2022-01-01,2024-12-31,Model fitting
1,Temporal validation,2025-01-01,2025-12-31,Model selection
2,External test,2026-01-01,2026-05-31,Final external evaluation


Temporal order valid: True


#### Interpretación

El protocolo experimental queda estructurado en tres períodos cronológicamente diferenciados. Las observaciones comprendidas entre 2022 y 2024 constituyen el conjunto de entrenamiento, el año 2025 se reserva para la validación temporal y el período entre enero y mayo de 2026 permanece destinado exclusivamente al test externo final.

La comprobación realizada confirma que el conjunto de entrenamiento precede temporalmente a la validación y que, a su vez, la validación precede al test externo. Por tanto, el orden cronológico definido para el experimento es consistente.

Esta separación determina también el papel metodológico de cada conjunto. Durante la fase de desarrollo, los parámetros aprendidos por los modelos y por las transformaciones dependientes de los datos deberán estimarse utilizando exclusivamente el período de entrenamiento. Los resultados obtenidos sobre 2025 podrán utilizarse para comparar y seleccionar configuraciones, mientras que las observaciones de 2026 permanecerán excluidas de estas decisiones.

El siguiente paso será caracterizar cuantitativamente los conjuntos definidos, comprobando su volumen de observaciones y la distribución de la variable objetivo antes de comenzar los experimentos de modelado.

### 2.3 Caracterización de los conjuntos de entrenamiento y validación

Una vez definidos los períodos experimentales, se caracterizan cuantitativamente los conjuntos de entrenamiento y validación temporal. Esta comprobación permite conocer el volumen efectivo de observaciones disponible en cada etapa y verificar la distribución de la variable objetivo antes de iniciar el modelado.

Para cada conjunto se calcularán el número total de observaciones, el número de vuelos pertenecientes a las clases `ARR_DEL15 = 0` y `ARR_DEL15 = 1`, y la proporción de vuelos con retraso de llegada igual o superior a 15 minutos.

La caracterización se realizará mediante lecturas selectivas de `FL_DATE` y `ARR_DEL15` sobre los archivos Parquet del período de desarrollo. No es necesario cargar los predictores en esta etapa, ya que todavía no se realizará ningún entrenamiento ni transformación. Este procedimiento reduce el consumo de memoria y mantiene el análisis ajustado al objetivo específico del subbloque.

La comparación entre entrenamiento y validación permitirá además identificar posibles diferencias en la prevalencia de la clase positiva entre ambos períodos. Estas diferencias serán relevantes posteriormente para interpretar las métricas de clasificación y el tratamiento del desbalance.

In [28]:
# ---------------------------------------------------------
# 1. Inicialización de acumuladores
# ---------------------------------------------------------

period_counts = {
    "Training": {
        "total": 0,
        "class_0": 0,
        "class_1": 0,
    },
    "Temporal validation": {
        "total": 0,
        "class_0": 0,
        "class_1": 0,
    },
}


# ---------------------------------------------------------
# 2. Lectura incremental del período de desarrollo
# ---------------------------------------------------------

for parquet_file in development_parquet_files:

    period_data = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            target_variable,
        ],
    )

    period_data[temporal_variable] = pd.to_datetime(
        period_data[temporal_variable]
    )

    training_mask = (
        (period_data[temporal_variable] >= training_start_date)
        & (period_data[temporal_variable] <= training_end_date)
    )

    validation_mask = (
        (period_data[temporal_variable] >= validation_start_date)
        & (period_data[temporal_variable] <= validation_end_date)
    )

    training_target = period_data.loc[
        training_mask,
        target_variable,
    ]

    validation_target = period_data.loc[
        validation_mask,
        target_variable,
    ]

    period_counts["Training"]["total"] += len(
        training_target
    )
    period_counts["Training"]["class_0"] += (
        training_target == 0
    ).sum()
    period_counts["Training"]["class_1"] += (
        training_target == 1
    ).sum()

    period_counts["Temporal validation"]["total"] += len(
        validation_target
    )
    period_counts["Temporal validation"]["class_0"] += (
        validation_target == 0
    ).sum()
    period_counts["Temporal validation"]["class_1"] += (
        validation_target == 1
    ).sum()


# ---------------------------------------------------------
# 3. Construcción del resumen experimental
# ---------------------------------------------------------

experimental_characterization = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "total_rows": counts["total"],
            "class_0": counts["class_0"],
            "class_1": counts["class_1"],
            "positive_rate_pct": (
                counts["class_1"]
                / counts["total"]
                * 100
            ),
        }
        for dataset_name, counts in period_counts.items()
    ]
)


# ---------------------------------------------------------
# 4. Formato de resultados
# ---------------------------------------------------------

experimental_characterization[
    "positive_rate_pct"
] = experimental_characterization[
    "positive_rate_pct"
].round(2)


# ---------------------------------------------------------
# 5. Presentación de resultados
# ---------------------------------------------------------

display(experimental_characterization)

,dataset,total_rows,class_0,class_1,positive_rate_pct
0,Training,21399093,16977169,4421924,20.66
1,Temporal validation,7597494,5912161,1685333,22.18


#### Interpretación

La partición temporal proporciona 21,399,093 observaciones para el entrenamiento correspondiente al período 2022–2024 y 7,597,494 observaciones para la validación temporal de 2025. La suma de ambos conjuntos coincide con las 28,996,587 observaciones del dataset de desarrollo, por lo que la separación temporal conserva íntegramente la población disponible para esta fase.

En el conjunto de entrenamiento se registran 4,421,924 vuelos con `ARR_DEL15 = 1`, equivalentes al 20.66 % de las observaciones, mientras que en la validación temporal se registran 1,685,333 casos positivos, correspondientes al 22.18 %. La prevalencia de la clase positiva aumenta, por tanto, 1.52 puntos porcentuales en 2025 respecto al período 2022–2024.

En ambos conjuntos continúa predominando la clase negativa, lo que confirma que el problema de desbalance identificado durante el análisis exploratorio también está presente en la partición utilizada para el modelado. No obstante, la diferencia observada entre las tasas positivas muestra que la distribución de la variable objetivo no es completamente constante entre ambos períodos.

Esta variación no permite establecer por sí sola la existencia ni la causa de un cambio estructural, pero refuerza el interés de utilizar una validación cronológica: los modelos serán evaluados sobre un período posterior cuya prevalencia de retrasos difiere ligeramente de la observada durante el entrenamiento.

### 2.4 Validación consolidada del protocolo experimental

Antes de iniciar la construcción del baseline y los modelos predictivos se realiza una validación consolidada del protocolo experimental definido en los subbloques anteriores.

La comprobación integra las fronteras temporales, el volumen de observaciones y la asignación metodológica de cada período. Se verificará que el entrenamiento finaliza antes del comienzo de la validación temporal, que la validación finaliza antes del inicio del test externo y que la suma de las observaciones de entrenamiento y validación coincide con el volumen total del dataset persistido para desarrollo.

También se comprobará que las dos particiones utilizadas durante el desarrollo poseen observaciones y contienen ejemplos de ambas clases de la variable objetivo. Esta condición es necesaria para que las métricas de clasificación previstas puedan calcularse e interpretarse adecuadamente.

La validación se realizará reutilizando los resultados ya calculados, evitando nuevas lecturas sobre los archivos Parquet. El objetivo es cerrar el diseño experimental con un conjunto compacto de comprobaciones antes de comenzar la evaluación predictiva.

In [29]:
# ---------------------------------------------------------
# 1. Recuperación de los conteos experimentales
# ---------------------------------------------------------

training_summary = experimental_characterization.loc[
    experimental_characterization["dataset"] == "Training"
].iloc[0]

validation_summary = experimental_characterization.loc[
    experimental_characterization["dataset"] == "Temporal validation"
].iloc[0]

development_total_rows = (
    int(training_summary["total_rows"])
    + int(validation_summary["total_rows"])
)


# ---------------------------------------------------------
# 2. Definición de las comprobaciones del protocolo
# ---------------------------------------------------------

protocol_validation = pd.DataFrame(
    {
        "validation": [
            "Training precedes temporal validation",
            "Temporal validation precedes external test",
            "Training contains observations",
            "Temporal validation contains observations",
            "Training contains both target classes",
            "Temporal validation contains both target classes",
            "Development partition preserves all rows",
            "External test reserved after development",
        ],
        "result": [
            training_end_date < validation_start_date,
            validation_end_date < external_test_start_date,
            int(training_summary["total_rows"]) > 0,
            int(validation_summary["total_rows"]) > 0,
            (
                int(training_summary["class_0"]) > 0
                and int(training_summary["class_1"]) > 0
            ),
            (
                int(validation_summary["class_0"]) > 0
                and int(validation_summary["class_1"]) > 0
            ),
            development_total_rows == 28_996_587,
            development_end_date < external_test_start_date,
        ],
    }
)


# ---------------------------------------------------------
# 3. Resumen cuantitativo del protocolo
# ---------------------------------------------------------

protocol_summary = pd.DataFrame(
    {
        "dataset": [
            "Training",
            "Temporal validation",
            "Development total",
        ],
        "period": [
            "2022-2024",
            "2025",
            "2022-2025",
        ],
        "total_rows": [
            int(training_summary["total_rows"]),
            int(validation_summary["total_rows"]),
            development_total_rows,
        ],
        "positive_rate_pct": [
            float(training_summary["positive_rate_pct"]),
            float(validation_summary["positive_rate_pct"]),
            np.nan,
        ],
    }
)


# ---------------------------------------------------------
# 4. Presentación consolidada
# ---------------------------------------------------------

display(protocol_summary)

display(protocol_validation)

print(
    "All experimental protocol checks valid:",
    protocol_validation["result"].all(),
)

,dataset,period,total_rows,positive_rate_pct
0,Training,2022-2024,21399093,20.66
1,Temporal validation,2025,7597494,22.18
2,Development total,2022-2025,28996587,NaN


,validation,result
0,Training precedes temporal validation,True
1,Temporal validation precedes external test,True
2,Training contains observations,True
3,Temporal validation contains observations,True
4,Training contains both target classes,True
5,Temporal validation contains both target classes,True
6,Development partition preserves all rows,True
7,External test reserved after development,True


All experimental protocol checks valid: True


## 3. Baseline de referencia

Antes de entrenar los modelos predictivos se establece un baseline que represente el nivel mínimo de rendimiento que debería superar cualquier modelo que pretenda aportar capacidad predictiva útil.

Dado el desbalance observado en `ARR_DEL15`, se utilizará como referencia un clasificador trivial que predice siempre la clase mayoritaria del conjunto de entrenamiento. En este caso, la clase mayoritaria corresponde a `ARR_DEL15 = 0`, es decir, vuelos que no presentan un retraso de llegada igual o superior a 15 minutos.

El baseline se evaluará sobre la validación temporal de 2025, manteniendo exactamente la misma población que posteriormente se utilizará para comparar los modelos candidatos. Se calcularán Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (PR-AUC), de forma que la referencia sea comparable con las métricas utilizadas durante el resto del proceso experimental.

El bloque contiene un único subbloque:

- 1. Definición y evaluación del baseline.

Como resultado se obtendrá una referencia cuantitativa mínima frente a la cual podrán evaluarse los modelos supervisados posteriores.

### 3.1 Definición y evaluación del baseline

El baseline se define mediante la predicción constante de la clase mayoritaria observada en el conjunto de entrenamiento. Esta estrategia no utiliza ninguna de las variables predictoras y, por tanto, no pretende resolver el problema de clasificación, sino establecer una referencia mínima de rendimiento.

La clase mayoritaria se determinará exclusivamente a partir del período de entrenamiento 2022–2024, respetando la separación temporal del protocolo experimental. Posteriormente, esta predicción constante se evaluará sobre la distribución real de `ARR_DEL15` correspondiente a la validación temporal de 2025.

Además de `Accuracy`, se calcularán `Precision`, `Recall`, `F1`, `ROC-AUC` y `Average Precision`. Esta combinación es especialmente importante ante el desbalance de clases, ya que una elevada exactitud global puede coexistir con una incapacidad completa para identificar vuelos retrasados.

Para evitar nuevas lecturas sobre los archivos Parquet, la evaluación reutilizará los conteos de clases obtenidos durante la caracterización del protocolo experimental.

In [8]:
# ---------------------------------------------------------
# 1. Importación de las métricas necesarias
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


# ---------------------------------------------------------
# 2. Determinación de la clase mayoritaria en training
# ---------------------------------------------------------

training_class_counts = {
    0: int(training_summary["class_0"]),
    1: int(training_summary["class_1"]),
}

majority_class = max(
    training_class_counts,
    key=training_class_counts.get,
)


# ---------------------------------------------------------
# 3. Reconstrucción compacta de la validación
# ---------------------------------------------------------

validation_class_0 = int(
    validation_summary["class_0"]
)

validation_class_1 = int(
    validation_summary["class_1"]
)

y_validation_baseline = np.concatenate(
    [
        np.zeros(
            validation_class_0,
            dtype=np.uint8,
        ),
        np.ones(
            validation_class_1,
            dtype=np.uint8,
        ),
    ]
)

baseline_predictions = np.full(
    y_validation_baseline.shape,
    majority_class,
    dtype=np.uint8,
)

baseline_scores = np.full(
    y_validation_baseline.shape,
    float(majority_class),
    dtype=np.float32,
)


# ---------------------------------------------------------
# 4. Evaluación del baseline
# ---------------------------------------------------------

baseline_results = pd.DataFrame(
    {
        "model": [
            "Majority class baseline"
        ],
        "validation_period": [
            "2025"
        ],
        "predicted_class": [
            majority_class
        ],
        "accuracy": [
            accuracy_score(
                y_validation_baseline,
                baseline_predictions,
            )
        ],
        "precision": [
            precision_score(
                y_validation_baseline,
                baseline_predictions,
                zero_division=0,
            )
        ],
        "recall": [
            recall_score(
                y_validation_baseline,
                baseline_predictions,
                zero_division=0,
            )
        ],
        "f1": [
            f1_score(
                y_validation_baseline,
                baseline_predictions,
                zero_division=0,
            )
        ],
        "roc_auc": [
            roc_auc_score(
                y_validation_baseline,
                baseline_scores,
            )
        ],
        "pr_auc": [
            average_precision_score(
                y_validation_baseline,
                baseline_scores,
            )
        ],
    }
)


# ---------------------------------------------------------
# 5. Formato y presentación
# ---------------------------------------------------------

metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

baseline_results[
    metric_columns
] = baseline_results[
    metric_columns
].round(4)

display(baseline_results)

,model,validation_period,predicted_class,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Majority class baseline,2025,0,0.7782,0.0,0.0,0.0,0.5,0.2218


#### Interpretación

El baseline de clase mayoritaria alcanza una exactitud del 77.82 % sobre la validación temporal de 2025. Este valor refleja principalmente el predominio de la clase `ARR_DEL15 = 0` y no una capacidad real para identificar vuelos con retraso.

La precisión, el recall y el F1 de la clase positiva son iguales a cero, ya que el clasificador no genera ninguna predicción de `ARR_DEL15 = 1`. Por tanto, aunque la exactitud global pueda parecer elevada, el modelo trivial no resulta útil para el objetivo principal del proyecto, centrado en detectar vuelos que llegarán con al menos 15 minutos de retraso.

El valor de `ROC-AUC = 0.5` confirma que el baseline no posee capacidad discriminativa entre ambas clases. Asimismo, el `PR-AUC = 0.2218` coincide con la prevalencia aproximada de la clase positiva en la validación temporal, comportamiento esperado para una referencia sin capacidad predictiva.

En consecuencia, los modelos supervisados posteriores deberán evaluarse no solo por su exactitud global, sino por su capacidad para mejorar la identificación de la clase positiva y superar este nivel mínimo de discriminación.

## 4. Preparación del preprocesamiento para modelado

Este bloque define el preprocesamiento que transformará las variables seleccionadas en una representación compatible con los algoritmos de clasificación, manteniendo la separación temporal establecida en el protocolo experimental.

El preprocesamiento debe adaptarse a la naturaleza de los predictores. Las variables categóricas requieren una representación numérica que permita trabajar con categorías nominales sin imponer relaciones ordinales artificiales, mientras que las variables numéricas pueden requerir estandarización en aquellos algoritmos sensibles a la escala.

Un aspecto metodológico fundamental es evitar leakage temporal. Cualquier parámetro aprendido a partir de los datos, como las categorías reconocidas por el codificador o los parámetros de escalado, deberá estimarse exclusivamente utilizando el período de entrenamiento 2022–2024. La validación temporal de 2025 se transformará posteriormente utilizando esos parámetros sin intervenir en su aprendizaje.

Debido al elevado volumen del conjunto de entrenamiento, el procedimiento también deberá limitar las lecturas innecesarias y evitar la materialización prematura de matrices transformadas de gran tamaño.

El bloque se divide en:

1. Recuperación y clasificación de los predictores.
2. Tratamiento de variables categóricas.
3. Tratamiento de variables numéricas.
4. Construcción del preprocesamiento reproducible.
5. Validación consolidada del preprocesamiento.

Como resultado se obtendrá un contrato de preprocesamiento reproducible, compatible con las distintas familias de modelos y aprendido exclusivamente a partir de la información disponible durante el período de entrenamiento.

### 4.1 Recuperación y clasificación de los predictores

El primer paso del preprocesamiento consiste en recuperar y clasificar los predictores definidos durante la preparación del dataset de modelado.

Los diez predictores se separan según su naturaleza y el tratamiento que requerirán posteriormente. Las ocho variables categóricas serán procesadas mediante codificación categórica, mientras que `CRS_ELAPSED_TIME` y `DISTANCE` constituyen las dos variables numéricas.

Esta clasificación permite construir posteriormente transformaciones diferenciadas según las necesidades de cada familia de modelos. En particular, los algoritmos sensibles a la escala podrán utilizar versiones estandarizadas de las variables numéricas, mientras que los modelos basados en árboles no requieren necesariamente esta transformación.

También se comprobará que `FL_DATE` y `ARR_DEL15` permanecen fuera del conjunto de predictores. La primera se utiliza exclusivamente para controlar las particiones temporales y la segunda representa la variable objetivo, por lo que su incorporación como entrada del modelo sería metodológicamente incorrecta.

In [9]:
# ---------------------------------------------------------
# 1. Recuperación de los grupos de predictores
# ---------------------------------------------------------

categorical_modeling_features = categorical_features.copy()

numerical_modeling_features = numerical_features.copy()

all_modeling_features = (
    categorical_modeling_features
    + numerical_modeling_features
)


# ---------------------------------------------------------
# 2. Clasificación del tratamiento previsto
# ---------------------------------------------------------

predictor_classification = pd.DataFrame(
    {
        "feature": all_modeling_features,
        "feature_type": (
            ["Categorical"] * len(categorical_modeling_features)
            + ["Numerical"] * len(numerical_modeling_features)
        ),
        "planned_treatment": (
            ["One-hot encoding"] * len(categorical_modeling_features)
            + ["Model-dependent"] * len(numerical_modeling_features)
        ),
    }
)


# ---------------------------------------------------------
# 3. Comprobaciones del contrato de predictores
# ---------------------------------------------------------

predictor_contract_validation = pd.DataFrame(
    {
        "validation": [
            "Ten predictors recovered",
            "Eight categorical predictors recovered",
            "Two numerical predictors recovered",
            "No duplicated predictors",
            "Temporal variable excluded from predictors",
            "Target variable excluded from predictors",
            "Categorical and numerical groups are disjoint",
        ],
        "result": [
            len(all_modeling_features) == 10,
            len(categorical_modeling_features) == 8,
            len(numerical_modeling_features) == 2,
            len(all_modeling_features)
            == len(set(all_modeling_features)),
            temporal_variable not in all_modeling_features,
            target_variable not in all_modeling_features,
            set(categorical_modeling_features).isdisjoint(
                numerical_modeling_features
            ),
        ],
    }
)


# ---------------------------------------------------------
# 4. Presentación de resultados
# ---------------------------------------------------------

display(predictor_classification)

display(predictor_contract_validation)

print(
    "Predictor contract valid:",
    predictor_contract_validation["result"].all(),
)

,feature,feature_type,planned_treatment
0,MONTH,Categorical,One-hot encoding
1,DAY_OF_WEEK,Categorical,One-hot encoding
2,MKT_UNIQUE_CARRIER,Categorical,One-hot encoding
3,OP_UNIQUE_CARRIER,Categorical,One-hot encoding
4,ORIGIN,Categorical,One-hot encoding
5,DEST,Categorical,One-hot encoding
6,DEP_TIME_BLK,Categorical,One-hot encoding
7,ARR_TIME_BLK,Categorical,One-hot encoding
8,CRS_ELAPSED_TIME,Numerical,Model-dependent
9,DISTANCE,Numerical,Model-dependent


,validation,result
0,Ten predictors recovered,True
1,Eight categorical predictors recovered,True
2,Two numerical predictors recovered,True
3,No duplicated predictors,True
4,Temporal variable excluded from predictors,True
5,Target variable excluded from predictors,True
6,Categorical and numerical groups are disjoint,True


Predictor contract valid: True


#### Interpretación

El contrato de predictores recuperado para el modelado está compuesto por diez variables, distribuidas en ocho predictores categóricos y dos predictores numéricos. La clasificación obtenida coincide con el esquema definido durante la preparación del dataset.

Las variables categóricas se destinarán a una representación mediante one-hot encoding, adecuada para variables nominales en las que no se desea introducir una relación ordinal artificial. Las variables `CRS_ELAPSED_TIME` y `DISTANCE` se mantienen como predictores numéricos y su transformación dependerá de las características del algoritmo utilizado.

Las siete comprobaciones realizadas resultan satisfactorias. No existen predictores duplicados, los grupos categórico y numérico son disjuntos y tanto `FL_DATE` como `ARR_DEL15` permanecen excluidos del conjunto de variables explicativas.

Esta última comprobación resulta especialmente relevante desde el punto de vista metodológico: `FL_DATE` se utilizará únicamente para controlar la separación cronológica de los datos, mientras que `ARR_DEL15` constituye la variable objetivo. En consecuencia, el contrato obtenido permite continuar con la construcción del preprocesamiento sin incorporar información incompatible con el diseño predictivo establecido.

El siguiente paso consistirá en determinar las categorías observadas durante el período de entrenamiento 2022–2024, que constituirán el vocabulario utilizado posteriormente por el codificador categórico.

### 4.2 Tratamiento de variables categóricas

Las ocho variables categóricas serán representadas mediante one-hot encoding. Esta transformación permite convertir cada categoría en una variable indicadora sin introducir relaciones ordinales artificiales entre sus valores.

Para preservar la independencia temporal de la validación, el vocabulario de categorías debe obtenerse exclusivamente a partir de las observaciones correspondientes al período de entrenamiento 2022–2024. Las categorías que aparezcan posteriormente en 2025 no podrán intervenir en la construcción del codificador.

Debido al elevado volumen del conjunto de entrenamiento y a que `OneHotEncoder` no permite un aprendizaje incremental mediante `partial_fit`, se utilizará una estrategia en dos etapas. Primero se recorrerán los archivos Parquet del dataset de desarrollo leyendo únicamente `FL_DATE` y las ocho variables categóricas. Para cada archivo se conservarán exclusivamente las observaciones pertenecientes a 2022–2024 y se acumularán sus categorías únicas. Posteriormente, este vocabulario se utilizará para configurar explícitamente el codificador.

Esta estrategia evita materializar simultáneamente los más de 21 millones de registros de entrenamiento y reduce la lectura a las columnas estrictamente necesarias. Además, se incorporará una barra de progreso para visualizar el porcentaje completado, el tiempo transcurrido y la estimación del tiempo restante durante el procesamiento incremental.

Las categorías no observadas durante el entrenamiento serán tratadas posteriormente mediante `handle_unknown="ignore"`, permitiendo transformar la validación temporal y el test externo sin incorporar información futura al aprendizaje del preprocesamiento.

In [10]:
# ---------------------------------------------------------
# 1. Importación de herramientas necesarias
# ---------------------------------------------------------

from tqdm.auto import tqdm
from sklearn.preprocessing import OneHotEncoder


# ---------------------------------------------------------
# 2. Inicialización del vocabulario categórico
# ---------------------------------------------------------

training_category_sets = {
    feature: set()
    for feature in categorical_modeling_features
}


# ---------------------------------------------------------
# 3. Recuperación incremental de categorías de training
# ---------------------------------------------------------

for parquet_file in tqdm(
    development_parquet_files,
    desc="Recovering training categories",
    unit="file",
):

    categorical_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            *categorical_modeling_features,
        ],
    )

    categorical_chunk[temporal_variable] = pd.to_datetime(
        categorical_chunk[temporal_variable]
    )

    training_mask = (
        (categorical_chunk[temporal_variable] >= training_start_date)
        & (categorical_chunk[temporal_variable] <= training_end_date)
    )

    training_chunk = categorical_chunk.loc[
        training_mask,
        categorical_modeling_features,
    ]

    for feature in categorical_modeling_features:
        training_category_sets[feature].update(
            training_chunk[feature]
            .dropna()
            .unique()
            .tolist()
        )

    del categorical_chunk
    del training_chunk


# ---------------------------------------------------------
# 4. Construcción ordenada del vocabulario
# ---------------------------------------------------------

training_categories = {
    feature: sorted(training_category_sets[feature])
    for feature in categorical_modeling_features
}

categorical_vocabulary = pd.DataFrame(
    {
        "feature": categorical_modeling_features,
        "training_categories": [
            len(training_categories[feature])
            for feature in categorical_modeling_features
        ],
    }
)

categorical_vocabulary["ohe_columns"] = (
    categorical_vocabulary["training_categories"]
)


# ---------------------------------------------------------
# 5. Configuración del codificador categórico
# ---------------------------------------------------------

categorical_encoder = OneHotEncoder(
    categories=[
        training_categories[feature]
        for feature in categorical_modeling_features
    ],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)


# ---------------------------------------------------------
# 6. Resumen del vocabulario categórico
# ---------------------------------------------------------

total_categorical_categories = int(
    categorical_vocabulary["training_categories"].sum()
)

display(categorical_vocabulary)

print(
    "Total categorical OHE columns:",
    total_categorical_categories,
)

print(
    "Vocabulary learned from training only:",
    True,
)

Recovering training categories:   0%|          | 0/320 [00:00<?, ?file/s]

,feature,training_categories,ohe_columns
0,MONTH,12,12
1,DAY_OF_WEEK,7,7
2,MKT_UNIQUE_CARRIER,10,10
3,OP_UNIQUE_CARRIER,21,21
4,ORIGIN,376,376
5,DEST,377,377
6,DEP_TIME_BLK,19,19
7,ARR_TIME_BLK,19,19


Total categorical OHE columns: 841
Vocabulary learned from training only: True


#### Interpretación

El vocabulario categórico construido exclusivamente a partir del período de entrenamiento 2022–2024 contiene un total de 841 categorías distribuidas entre los ocho predictores categóricos.

Las variables de mayor cardinalidad son `ORIGIN` y `DEST`, con 376 y 377 categorías respectivamente, mientras que las variables temporales y de compañía presentan cardinalidades considerablemente menores. Como consecuencia, la mayor parte de la dimensionalidad introducida por el one-hot encoding procede de la información aeroportuaria.

El codificador categórico queda configurado utilizando únicamente las categorías observadas durante el entrenamiento y mediante `handle_unknown="ignore"`. De este modo, cualquier categoría que aparezca por primera vez durante la validación temporal de 2025 o en el test externo de 2026 podrá ser procesada sin modificar el vocabulario aprendido ni incorporar información futura.

La representación categórica resultante tendrá 841 columnas. Al incorporar posteriormente las dos variables numéricas, se espera una representación de 843 predictores transformados, aunque esta dimensionalidad total deberá confirmarse una vez construido y validado el preprocesamiento completo.

La recuperación incremental del vocabulario permite además evitar la materialización simultánea de los 21.4 millones de registros de entrenamiento, manteniendo el procedimiento compatible con las restricciones de memoria del proyecto.

### 4.3 Tratamiento de variables numéricas

Las dos variables numéricas seleccionadas, `CRS_ELAPSED_TIME` y `DISTANCE`, requieren un tratamiento diferenciado según la familia de modelos utilizada.

Para los modelos sensibles a la escala, como la regresión logística, se utilizará una estandarización mediante `StandardScaler`. Esta transformación centra cada variable respecto a la media observada durante el entrenamiento y la expresa en función de su desviación estándar. De esta forma se evita que diferencias en las unidades o magnitudes de los predictores numéricos condicionen de manera desproporcionada el proceso de optimización.

Los parámetros de escalado deberán aprenderse exclusivamente a partir del período 2022–2024. Para evitar cargar simultáneamente los más de 21 millones de registros de entrenamiento, se utilizará `partial_fit`, actualizando incrementalmente las estadísticas del escalador a partir de cada archivo Parquet.

Los modelos basados en árboles no requieren estandarización para construir sus reglas de partición. Por este motivo, se conservará también una rama de preprocesamiento en la que las variables numéricas permanezcan sin transformar.

Este diseño permitirá construir posteriormente dos variantes compatibles del preprocesamiento: una destinada a modelos sensibles a la escala y otra para modelos basados en árboles, manteniendo en ambos casos el mismo tratamiento categórico.

In [50]:
# ---------------------------------------------------------
# 1. Importación del escalador numérico
# ---------------------------------------------------------

from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 2. Inicialización del escalador
# ---------------------------------------------------------

numerical_scaler = StandardScaler()

numerical_training_rows_processed = 0


# ---------------------------------------------------------
# 3. Aprendizaje incremental sobre training 2022-2024
# ---------------------------------------------------------

for parquet_file in tqdm(
    development_parquet_files,
    desc="Fitting numerical scaler",
    unit="file",
):

    numerical_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            *numerical_modeling_features,
        ],
    )

    numerical_chunk[temporal_variable] = pd.to_datetime(
        numerical_chunk[temporal_variable]
    )

    training_mask = (
        (numerical_chunk[temporal_variable] >= training_start_date)
        & (numerical_chunk[temporal_variable] <= training_end_date)
    )

    training_numerical_chunk = numerical_chunk.loc[
        training_mask,
        numerical_modeling_features,
    ]

    if not training_numerical_chunk.empty:
        numerical_scaler.partial_fit(
            training_numerical_chunk
        )

        numerical_training_rows_processed += len(
            training_numerical_chunk
        )

    del numerical_chunk
    del training_numerical_chunk


# ---------------------------------------------------------
# 4. Recuperación de los parámetros aprendidos
# ---------------------------------------------------------

numerical_scaler_summary = pd.DataFrame(
    {
        "feature": numerical_modeling_features,
        "mean": numerical_scaler.mean_,
        "scale": numerical_scaler.scale_,
        "variance": numerical_scaler.var_,
    }
)

numerical_scaler_summary[
    [
        "mean",
        "scale",
        "variance",
    ]
] = numerical_scaler_summary[
    [
        "mean",
        "scale",
        "variance",
    ]
].round(4)


# ---------------------------------------------------------
# 5. Validación del aprendizaje incremental
# ---------------------------------------------------------

numerical_scaler_validation = pd.DataFrame(
    {
        "validation": [
            "Scaler fitted",
            "Two numerical features learned",
            "Means are finite",
            "Scales are finite",
            "Scales are positive",
            "Training observations processed",
        ],
        "result": [
            hasattr(numerical_scaler, "mean_"),
            len(numerical_scaler.mean_) == 2,
            np.isfinite(numerical_scaler.mean_).all(),
            np.isfinite(numerical_scaler.scale_).all(),
            (numerical_scaler.scale_ > 0).all(),
            numerical_training_rows_processed
            == int(training_summary["total_rows"]),
        ],
    }
)


# ---------------------------------------------------------
# 6. Presentación de resultados
# ---------------------------------------------------------

display(numerical_scaler_summary)

display(numerical_scaler_validation)

print(
    "Numerical scaler valid:",
    numerical_scaler_validation["result"].all(),
)

Fitting numerical scaler:   0%|          | 0/320 [00:00<?, ?file/s]

,feature,mean,scale,variance
0,CRS_ELAPSED_TIME,142.5795,71.9046,5170.2725
1,DISTANCE,803.8786,592.7956,351406.6306


,validation,result
0,Scaler fitted,True
1,Two numerical features learned,True
2,Means are finite,True
3,Scales are finite,True
4,Scales are positive,True
5,Training observations processed,True


Numerical scaler valid: True


#### Interpretación

El escalador numérico se ha ajustado correctamente utilizando exclusivamente las observaciones correspondientes al período de entrenamiento 2022–2024. Las seis comprobaciones realizadas resultan satisfactorias, incluida la verificación de que el aprendizaje incremental procesó las 21,399,093 observaciones del conjunto de entrenamiento.

Para `CRS_ELAPSED_TIME` se obtiene una media de 142.5795 y una escala de 71.9046, mientras que `DISTANCE` presenta una media de 803.8786 y una escala de 592.7956. Todos los parámetros estimados son finitos y las escalas son estrictamente positivas, por lo que ambas variables pueden ser estandarizadas sin detectar problemas numéricos en esta etapa.

Estos parámetros se utilizarán en los modelos sensibles a la escala y permanecerán fijos al transformar la validación temporal de 2025. De este modo, las estadísticas de la validación no intervienen en la definición de la transformación y se mantiene la separación temporal establecida por el protocolo experimental.

Para los modelos basados en árboles se conservarán los valores numéricos originales, ya que su mecanismo de construcción mediante reglas de partición no requiere que las variables se encuentren en una escala común.

El siguiente paso consistirá en integrar el codificador categórico y las dos alternativas de tratamiento numérico en preprocesadores reproducibles adaptados a las diferentes familias de modelos.

### 4.4 Construcción del preprocesamiento reproducible

A partir del vocabulario categórico definido exclusivamente durante 2022–2024, se construyen las configuraciones de preprocesamiento que utilizarán posteriormente los modelos candidatos.

Se mantendrán dos variantes. La primera estará destinada a modelos sensibles a la escala y combinará one-hot encoding para las variables categóricas con estandarización para las variables numéricas. La segunda estará orientada a modelos basados en árboles y combinará el mismo tratamiento categórico con las variables numéricas en su escala original.

Las categorías del OneHotEncoder se especificarán explícitamente utilizando el vocabulario recuperado del conjunto de entrenamiento. Los parámetros necesarios para el tratamiento numérico deberán obtenerse exclusivamente a partir del período de entrenamiento 2022–2024 antes de transformar la validación temporal. De esta forma se mantiene un contrato de transformación independiente de la validación temporal.

La separación entre ambos preprocesadores evita aplicar transformaciones innecesarias de forma indiscriminada y permite mantener condiciones comparables entre modelos, modificando únicamente aquellos aspectos del preprocesamiento que dependen de las características de cada familia algorítmica.

En esta etapa se construirá el contrato reproducible de transformación. Su funcionamiento y dimensionalidad se comprobarán de forma consolidada en el siguiente subbloque antes de iniciar los entrenamientos.

In [12]:
# ---------------------------------------------------------
# 1. Importación de componentes del preprocesamiento
# ---------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer


# ---------------------------------------------------------
# 2. Reconstrucción del codificador categórico
# ---------------------------------------------------------

categorical_encoder = OneHotEncoder(
    categories=[
        training_categories[feature]
        for feature in categorical_modeling_features
    ],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)


# ---------------------------------------------------------
# 3. Configuración del tratamiento numérico para árboles
# ---------------------------------------------------------

numerical_passthrough = FunctionTransformer(
    func=None,
    validate=False,
)


# ---------------------------------------------------------
# 4. Construcción del preprocesador sensible a escala
# ---------------------------------------------------------

scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder,
            categorical_modeling_features,
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_modeling_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)


# ---------------------------------------------------------
# 5. Construcción del preprocesador para árboles
# ---------------------------------------------------------

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder,
            categorical_modeling_features,
        ),
        (
            "numerical",
            numerical_passthrough,
            numerical_modeling_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)


# ---------------------------------------------------------
# 6. Definición del contrato de aprendizaje
# ---------------------------------------------------------

preprocessing_contract = pd.DataFrame(
    {
        "preprocessor": [
            "Scaled",
            "Tree",
        ],
        "categorical_treatment": [
            "OneHotEncoder",
            "OneHotEncoder",
        ],
        "numerical_treatment": [
            "StandardScaler",
            "Passthrough",
        ],
        "categorical_vocabulary_source": [
            "Training 2022-2024",
            "Training 2022-2024",
        ],
        "fit_period": [
            "Training 2022-2024",
            "Training 2022-2024",
        ],
        "intended_models": [
            "Logistic Regression",
            "Decision Tree / Random Forest",
        ],
    }
)


# ---------------------------------------------------------
# 7. Presentación del contrato
# ---------------------------------------------------------

display(preprocessing_contract)

print(
    "Expected transformed features:",
    total_categorical_categories
    + len(numerical_modeling_features),
)

,preprocessor,categorical_treatment,numerical_treatment,categorical_vocabulary_source,fit_period,intended_models
0,Scaled,OneHotEncoder,StandardScaler,Training 2022-2024,Training 2022-2024,Logistic Regression
1,Tree,OneHotEncoder,Passthrough,Training 2022-2024,Training 2022-2024,Decision Tree / Random Forest


Expected transformed features: 843


#### Interpretación

Se han construido dos configuraciones de preprocesamiento adaptadas a las características de las familias de modelos consideradas.

La primera configuración combina one-hot encoding para los predictores categóricos con estandarización de las variables numéricas y está destinada a la regresión logística, cuyo proceso de optimización puede verse afectado por diferencias de escala entre los predictores.

La segunda configuración utiliza el mismo tratamiento categórico, pero conserva las variables numéricas en su escala original, siendo adecuada para Decision Tree y Random Forest, cuyos mecanismos de partición no requieren estandarización.

En ambos casos, el vocabulario categórico procede exclusivamente del período de entrenamiento 2022–2024 y cualquier ajuste posterior del preprocesamiento deberá realizarse únicamente sobre esta misma población. La validación temporal de 2025 permanecerá, por tanto, fuera del aprendizaje de las transformaciones.

A partir de las 841 variables generadas por el one-hot encoding y de las dos variables numéricas, la representación transformada se espera que contenga 843 predictores. Esta dimensionalidad deberá confirmarse mediante una transformación real en el siguiente subbloque antes de iniciar los experimentos de modelado.

### 4.5 Validación consolidada del preprocesamiento

Antes de iniciar el entrenamiento de los modelos se realizará una validación funcional de la configuración de preprocesamiento definida en los subbloques anteriores.

La comprobación utilizará una muestra real perteneciente exclusivamente al período de entrenamiento 2022–2024. Sobre esta muestra se ajustarán y aplicarán las dos configuraciones definidas: el preprocesamiento con estandarización numérica destinado a la regresión logística y el preprocesamiento sin escalado destinado a los modelos basados en árboles.

El objetivo es verificar que ambos procedimientos procesan correctamente los diez predictores originales, generan una representación dispersa y producen la dimensionalidad esperada a partir del vocabulario categórico definido con los datos de entrenamiento.

Esta comprobación se realizará sobre una muestra limitada, ya que su finalidad es validar el funcionamiento técnico del preprocesamiento antes de su aplicación sobre el conjunto completo de entrenamiento. Los parámetros utilizados posteriormente durante el modelado deberán ajustarse exclusivamente con los datos correspondientes al período 2022–2024.

La validación consolidada permitirá confirmar que las dos configuraciones son técnicamente compatibles con los modelos candidatos antes de comenzar los experimentos de entrenamiento y evaluación.

In [13]:
# ---------------------------------------------------------
# 1. Recuperación de una muestra real de training
# ---------------------------------------------------------

preprocessing_sample = None

for parquet_file in development_parquet_files:

    sample_chunk = pd.read_parquet(
        parquet_file,
        columns=modeling_columns,
    )

    sample_chunk[temporal_variable] = pd.to_datetime(
        sample_chunk[temporal_variable]
    )

    training_mask = (
        (sample_chunk[temporal_variable] >= training_start_date)
        & (sample_chunk[temporal_variable] <= training_end_date)
    )

    training_sample_chunk = sample_chunk.loc[
        training_mask,
        modeling_columns,
    ]

    if not training_sample_chunk.empty:
        preprocessing_sample = (
            training_sample_chunk
            .head(10_000)
            .copy()
        )
        break

    del sample_chunk
    del training_sample_chunk


if preprocessing_sample is None:
    raise ValueError(
        "No training observations were found for preprocessing validation."
    )


# ---------------------------------------------------------
# 2. Separación de los predictores
# ---------------------------------------------------------

X_preprocessing_sample = preprocessing_sample[
    all_modeling_features
].copy()


# ---------------------------------------------------------
# 3. Ajuste y transformación de prueba
# ---------------------------------------------------------

X_scaled_sample = scaled_preprocessor.fit_transform(
    X_preprocessing_sample
)

X_tree_sample = tree_preprocessor.fit_transform(
    X_preprocessing_sample
)


# ---------------------------------------------------------
# 4. Recuperación de la dimensionalidad esperada
# ---------------------------------------------------------

expected_transformed_features = (
    total_categorical_categories
    + len(numerical_modeling_features)
)


# ---------------------------------------------------------
# 5. Construcción del resumen de salida
# ---------------------------------------------------------

preprocessing_output_summary = pd.DataFrame(
    {
        "preprocessor": [
            "Scaled",
            "Tree",
        ],
        "sample_rows": [
            X_scaled_sample.shape[0],
            X_tree_sample.shape[0],
        ],
        "original_features": [
            X_preprocessing_sample.shape[1],
            X_preprocessing_sample.shape[1],
        ],
        "transformed_features": [
            X_scaled_sample.shape[1],
            X_tree_sample.shape[1],
        ],
        "output_type": [
            type(X_scaled_sample).__name__,
            type(X_tree_sample).__name__,
        ],
    }
)


# ---------------------------------------------------------
# 6. Validación consolidada del preprocesamiento
# ---------------------------------------------------------

preprocessing_validation = pd.DataFrame(
    {
        "validation": [
            "Training sample available",
            "Ten original predictors available",
            "Scaled preprocessing successful",
            "Tree preprocessing successful",
            "Scaled output has expected features",
            "Tree output has expected features",
            "Both preprocessors have equal dimensionality",
            "Scaled output is sparse",
            "Tree output is sparse",
        ],
        "result": [
            len(X_preprocessing_sample) > 0,
            X_preprocessing_sample.shape[1] == 10,
            X_scaled_sample.shape[0] == len(X_preprocessing_sample),
            X_tree_sample.shape[0] == len(X_preprocessing_sample),
            X_scaled_sample.shape[1] == expected_transformed_features,
            X_tree_sample.shape[1] == expected_transformed_features,
            X_scaled_sample.shape[1] == X_tree_sample.shape[1],
            hasattr(X_scaled_sample, "tocsr"),
            hasattr(X_tree_sample, "tocsr"),
        ],
    }
)


# ---------------------------------------------------------
# 7. Presentación de resultados
# ---------------------------------------------------------

display(preprocessing_output_summary)

display(preprocessing_validation)

print(
    "All preprocessing checks valid:",
    preprocessing_validation["result"].all(),
)

,preprocessor,sample_rows,original_features,transformed_features,output_type
0,Scaled,10000,10,843,csr_matrix
1,Tree,10000,10,843,csr_matrix


,validation,result
0,Training sample available,True
1,Ten original predictors available,True
2,Scaled preprocessing successful,True
3,Tree preprocessing successful,True
4,Scaled output has expected features,True
5,Tree output has expected features,True
6,Both preprocessors have equal dimensionality,True
7,Scaled output is sparse,True
8,Tree output is sparse,True


All preprocessing checks valid: True


#### Interpretación

La validación funcional del preprocesamiento se ha completado satisfactoriamente. Las dos configuraciones evaluadas procesan correctamente una muestra real de 10,000 observaciones pertenecientes al período de entrenamiento 2022–2024 y superan las nueve comprobaciones establecidas.

Los diez predictores originales se transforman en 843 variables en ambas configuraciones. Esta dimensionalidad corresponde a las 841 columnas generadas mediante one-hot encoding y a las dos variables numéricas incorporadas posteriormente. De este modo, se confirma empíricamente la dimensionalidad que había sido anticipada a partir del vocabulario categórico.

Tanto la configuración destinada a la regresión logística como la destinada a los modelos basados en árboles producen matrices dispersas de tipo `csr_matrix`. Este formato resulta especialmente adecuado para la representación generada mediante one-hot encoding, ya que evita almacenar explícitamente la elevada proporción de valores cero introducida por la codificación categórica.

La diferencia entre ambas configuraciones se mantiene exclusivamente en el tratamiento de las variables numéricas: la regresión logística utilizará estandarización, mientras que Decision Tree y Random Forest conservarán sus valores originales.

La prueba realizada en este subbloque tiene carácter exclusivamente funcional. Los ajustes efectuados sobre la muestra de 10,000 observaciones no se utilizarán para el entrenamiento de los modelos. Durante la fase experimental, el preprocesamiento correspondiente deberá ajustarse utilizando exclusivamente el conjunto de entrenamiento 2022–2024 antes de transformar la validación temporal de 2025.

Con estos resultados se considera validada la configuración de preprocesamiento y puede iniciarse la evaluación de los modelos candidatos.

## 5. Evaluación inicial de modelos candidatos

Una vez validado el procedimiento de preprocesamiento, se inicia la evaluación de los algoritmos de clasificación seleccionados para predecir `ARR_DEL15`.

Esta primera evaluación tiene como objetivo establecer el comportamiento inicial de diferentes familias de modelos bajo unas condiciones experimentales comunes y sin aplicar todavía estrategias específicas de optimización. De esta forma será posible comparar posteriormente estos resultados con el baseline de clase mayoritaria e identificar qué modelos presentan mayor potencial para el sistema de alerta temprana.

Se evaluarán tres algoritmos supervisados: Logistic Regression, Decision Tree y Random Forest. Su selección permite comparar un modelo lineal con dos métodos basados en árboles, manteniendo además la experimentación dentro de las técnicas estudiadas durante el máster.

Todos los modelos utilizarán como entrenamiento el período 2022–2024 y serán evaluados sobre la validación temporal de 2025. El conjunto correspondiente a enero–mayo de 2026 permanecerá completamente aislado durante esta etapa.

La evaluación se organizará en cuatro subbloques:

1. Definición de los algoritmos candidatos: definición de los modelos y justificación de sus configuraciones iniciales.
2. Configuración experimental: establecimiento de las condiciones comunes de entrenamiento, transformación y evaluación.
3. Entrenamiento y evaluación sobre la validación temporal: ajuste de los modelos con 2022–2024 y evaluación sobre 2025.
4. Comparación consolidada de resultados: comparación de las métricas obtenidas frente al baseline y entre los diferentes algoritmos.

Las métricas principales serán Precision, Recall, F1, ROC-AUC y Average Precision (PR-AUC), complementadas con Accuracy y la matriz de confusión cuando resulte necesario. Esta combinación permitirá evaluar tanto la capacidad discriminativa de los modelos como su comportamiento específico sobre la clase positiva `ARR_DEL15 = 1`.

Dado que el objetivo aplicado del TFM es utilizar el modelo como sistema de alerta temprana de retrasos, se prestará especial atención a Recall, buscando posteriormente una configuración capaz de identificar aproximadamente el 80 % de los vuelos retrasados sin deteriorar de forma excesiva Precision y el resto de métricas.

El resultado de este bloque proporcionará una referencia experimental inicial para identificar los modelos con mayor potencial y orientar las decisiones de optimización posteriores.

### 5.1 Definición de los algoritmos candidatos

La evaluación inicial utilizará tres algoritmos de clasificación supervisada: Logistic Regression, Decision Tree y Random Forest.

`LogisticRegression` proporciona un modelo lineal que permitirá establecer una referencia predictiva interpretable y evaluar hasta qué punto la combinación de los predictores disponibles permite separar ambas clases mediante una frontera lineal. Debido a la representación dispersa generada por el one-hot encoding y al elevado volumen de observaciones, se utilizará el solver `saga`. Las variables numéricas serán previamente estandarizadas.

`DecisionTreeClassifier` permitirá introducir relaciones no lineales e interacciones entre los predictores sin necesidad de especificarlas explícitamente. Se limitará inicialmente la profundidad del árbol y se establecerá un número mínimo de observaciones por hoja para evitar el crecimiento excesivo del modelo sobre un conjunto de entrenamiento de gran tamaño.

`RandomForestClassifier` permitirá evaluar un método de ensamblado basado en múltiples árboles. Se utilizará paralelización mediante los recursos disponibles del equipo y una configuración inicial moderada, evitando realizar todavía una búsqueda de hiperparámetros.

Las configuraciones de esta etapa no pretenden ser óptimas. Su finalidad es obtener una primera referencia comparable entre algoritmos. La optimización sistemática de los modelos finalistas se realizará posteriormente utilizando exclusivamente los datos destinados al desarrollo y validación.

No se aplicarán todavía `class_weight`, undersampling ni otras estrategias específicas de tratamiento del desbalance. Esto permitirá observar el comportamiento inicial de cada algoritmo frente a la distribución natural de la variable objetivo y disponer de una referencia para orientar las decisiones de optimización posteriores.

In [14]:
# ---------------------------------------------------------
# 1. Importación de los algoritmos candidatos
# ---------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# ---------------------------------------------------------
# 2. Configuración reproducible
# ---------------------------------------------------------

random_state = 42


# ---------------------------------------------------------
# 3. Definición de Logistic Regression
# ---------------------------------------------------------

logistic_regression = LogisticRegression(
    solver="saga",
    penalty="l2",
    max_iter=500,
    random_state=random_state,
)


# ---------------------------------------------------------
# 4. Definición de Decision Tree
# ---------------------------------------------------------

decision_tree = DecisionTreeClassifier(
    max_depth=15,
    min_samples_leaf=50,
    random_state=random_state,
)


# ---------------------------------------------------------
# 5. Definición de Random Forest
# ---------------------------------------------------------

random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=random_state,
)


# ---------------------------------------------------------
# 6. Asociación entre modelos y preprocesamiento
# ---------------------------------------------------------

candidate_models = {
    "Logistic Regression": {
        "estimator": logistic_regression,
        "preprocessing": "scaled",
    },
    "Decision Tree": {
        "estimator": decision_tree,
        "preprocessing": "tree",
    },
    "Random Forest": {
        "estimator": random_forest,
        "preprocessing": "tree",
    },
}


# ---------------------------------------------------------
# 7. Resumen de las configuraciones iniciales
# ---------------------------------------------------------

candidate_models_summary = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Decision Tree",
            "Random Forest",
        ],
        "family": [
            "Linear",
            "Tree",
            "Ensemble",
        ],
        "preprocessing": [
            "OHE + StandardScaler",
            "OHE + numerical passthrough",
            "OHE + numerical passthrough",
        ],
        "main_configuration": [
            "solver=saga, penalty=l2, max_iter=500",
            "max_depth=15, min_samples_leaf=50",
            "n_estimators=100, max_depth=15, min_samples_leaf=50",
        ],
        "imbalance_treatment": [
            "None",
            "None",
            "None",
        ],
    }
)

display(candidate_models_summary)

,model,family,preprocessing,main_configuration,imbalance_treatment
0,Logistic Regression,Linear,OHE + StandardScaler,"solver=saga, penalty=l2, max_iter=500",None
1,Decision Tree,Tree,OHE + numerical passthrough,"max_depth=15, min_samples_leaf=50",None
2,Random Forest,Ensemble,OHE + numerical passthrough,"n_estimators=100, max_depth=15, min_samples_le...",None


#### Interpretación

Se han definido tres algoritmos candidatos pertenecientes a familias diferentes: Logistic Regression como modelo lineal, Decision Tree como modelo basado en un único árbol y Random Forest como método de ensamblado.

La configuración de preprocesamiento se adapta a las características de cada familia. Logistic Regression utilizará one-hot encoding junto con estandarización de las variables numéricas, mientras que Decision Tree y Random Forest utilizarán la misma codificación categórica conservando las variables numéricas en su escala original.

Las parametrizaciones establecidas corresponden a configuraciones iniciales y no a configuraciones optimizadas. En Logistic Regression se utiliza el solver `saga`, regularización L2 y un máximo de 500 iteraciones. Para Decision Tree se limita inicialmente la complejidad mediante `max_depth=15` y `min_samples_leaf=50`. Random Forest utiliza 100 árboles y mantiene las mismas restricciones estructurales en cada estimador, incorporando además paralelización mediante `n_jobs=-1`.

Ninguno de los tres modelos incorpora en esta etapa ponderación de clases ni técnicas de remuestreo. Esto permitirá evaluar inicialmente su comportamiento frente a la distribución natural de `ARR_DEL15` y utilizar los resultados obtenidos como referencia para orientar las decisiones de optimización posteriores.

La definición de estos modelos establece, por tanto, las condiciones iniciales de la comparación experimental antes de proceder a su entrenamiento y evaluación temporal.

### 5.2 Configuración experimental

La comparación inicial de los modelos candidatos se realizará bajo unas condiciones experimentales comunes con el objetivo de garantizar que las diferencias observadas posteriormente sean atribuibles a los algoritmos evaluados y no a cambios en los datos o en el procedimiento de evaluación.

El período 2022–2024 constituirá el conjunto de entrenamiento, mientras que el año 2025 se utilizará como validación temporal. Esta separación se mantendrá constante para Logistic Regression, Decision Tree y Random Forest. El conjunto externo correspondiente a enero–mayo de 2026 permanecerá aislado durante toda esta etapa.

Los modelos utilizarán los mismos diez predictores y la misma variable objetivo, `ARR_DEL15`. Se mantendrán dos configuraciones de preprocesamiento: una con estandarización numérica para Logistic Regression y otra sin escalado para Decision Tree y Random Forest. Los dos modelos basados en árboles compartirán, por tanto, la misma representación transformada de los datos.

La clasificación inicial utilizará el umbral convencional de probabilidad 0.5. Este valor no se optimizará en esta etapa, ya que el objetivo es obtener una referencia inicial comparable entre algoritmos. Cualquier modificación posterior del umbral deberá realizarse utilizando exclusivamente los datos destinados al desarrollo y validación, sin utilizar el test externo de 2026.

La evaluación incluirá Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (PR-AUC). Accuracy se conservará como medida descriptiva, pero no será utilizada de forma aislada para seleccionar modelos debido al desbalance existente en la variable objetivo.

Debido al elevado volumen del conjunto de entrenamiento, se evitará repetir innecesariamente las transformaciones. Logistic Regression utilizará la representación escalada, mientras que Decision Tree y Random Forest compartirán la representación destinada a modelos basados en árboles. Durante la ejecución se registrarán también los tiempos de entrenamiento y evaluación para incorporar el coste computacional a la comparación experimental.

In [15]:
# ---------------------------------------------------------
# 1. Definición de los conjuntos experimentales
# ---------------------------------------------------------

experimental_sets = pd.DataFrame(
    {
        "dataset": [
            "Training",
            "Temporal validation",
        ],
        "period": [
            "2022-2024",
            "2025",
        ],
        "start_date": [
            training_start_date,
            validation_start_date,
        ],
        "end_date": [
            training_end_date,
            validation_end_date,
        ],
        "rows": [
            int(training_summary["total_rows"]),
            int(validation_summary["total_rows"]),
        ],
        "purpose": [
            "Model fitting",
            "Model evaluation and selection",
        ],
    }
)


# ---------------------------------------------------------
# 2. Definición de las métricas de evaluación
# ---------------------------------------------------------

evaluation_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]


# ---------------------------------------------------------
# 3. Configuración del umbral inicial
# ---------------------------------------------------------

initial_classification_threshold = 0.50


# ---------------------------------------------------------
# 4. Asociación entre modelos y representación de datos
# ---------------------------------------------------------

model_experimental_configuration = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Decision Tree",
            "Random Forest",
        ],
        "training_period": [
            "2022-2024",
            "2022-2024",
            "2022-2024",
        ],
        "validation_period": [
            "2025",
            "2025",
            "2025",
        ],
        "preprocessing": [
            "Scaled",
            "Tree",
            "Tree",
        ],
        "classification_threshold": [
            initial_classification_threshold,
            initial_classification_threshold,
            initial_classification_threshold,
        ],
        "imbalance_treatment": [
            "None",
            "None",
            "None",
        ],
    }
)


# ---------------------------------------------------------
# 5. Validación consolidada de la configuración
# ---------------------------------------------------------

experimental_configuration_validation = pd.DataFrame(
    {
        "validation": [
            "Training precedes validation",
            "Training rows match protocol",
            "Validation rows match protocol",
            "Same predictors for all models",
            "Same target for all models",
            "Same classification threshold",
            "No imbalance treatment applied",
            "External test remains excluded",
        ],
        "result": [
            training_end_date < validation_start_date,
            int(training_summary["total_rows"]) == 21_399_093,
            int(validation_summary["total_rows"]) == 7_597_494,
            len(all_modeling_features) == 10,
            target_variable == "ARR_DEL15",
            (
                model_experimental_configuration[
                    "classification_threshold"
                ].nunique()
                == 1
            ),
            (
                model_experimental_configuration[
                    "imbalance_treatment"
                ].eq("None").all()
            ),
            validation_end_date < external_test_start_date,
        ],
    }
)


# ---------------------------------------------------------
# 6. Presentación de la configuración experimental
# ---------------------------------------------------------

display(experimental_sets)

display(model_experimental_configuration)

display(experimental_configuration_validation)

print(
    "Evaluation metrics:",
    ", ".join(evaluation_metrics),
)

print(
    "Experimental configuration valid:",
    experimental_configuration_validation["result"].all(),
)

,dataset,period,start_date,end_date,rows,purpose
0,Training,2022-2024,2022-01-01,2024-12-31,21399093,Model fitting
1,Temporal validation,2025,2025-01-01,2025-12-31,7597494,Model evaluation and selection


,model,training_period,validation_period,preprocessing,classification_threshold,imbalance_treatment
0,Logistic Regression,2022-2024,2025,Scaled,0.5,None
1,Decision Tree,2022-2024,2025,Tree,0.5,None
2,Random Forest,2022-2024,2025,Tree,0.5,None


,validation,result
0,Training precedes validation,True
1,Training rows match protocol,True
2,Validation rows match protocol,True
3,Same predictors for all models,True
4,Same target for all models,True
5,Same classification threshold,True
6,No imbalance treatment applied,True
7,External test remains excluded,True


Evaluation metrics: accuracy, precision, recall, f1, roc_auc, pr_auc
Experimental configuration valid: True


#### Interpretación

La configuración experimental queda validada para la evaluación inicial de los modelos candidatos. Los tres algoritmos utilizarán exactamente el mismo período de entrenamiento, compuesto por 21,399,093 observaciones correspondientes a 2022–2024, y serán evaluados sobre las 7,597,494 observaciones de la validación temporal de 2025.

Las ocho comprobaciones realizadas resultan satisfactorias. Se confirma la correcta separación cronológica entre entrenamiento y validación, la utilización de los mismos diez predictores y de `ARR_DEL15` como variable objetivo, así como la aplicación de un umbral inicial común de 0.5.

Ninguno de los modelos incorpora todavía estrategias específicas de tratamiento del desbalance, por lo que los resultados obtenidos constituirán una referencia inicial del comportamiento de cada algoritmo frente a la distribución natural de la variable objetivo.

La evaluación utilizará Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (PR-AUC). La consideración conjunta de estas métricas permitirá analizar tanto el comportamiento de la clasificación a un umbral de 0.5 como la capacidad discriminativa de los modelos.

Finalmente, se confirma que el test externo correspondiente a enero–mayo de 2026 permanece completamente excluido de esta fase. Por tanto, puede iniciarse el entrenamiento y evaluación de los modelos sobre la validación temporal de 2025.

### 5.3 Entrenamiento y evaluación sobre la validación temporal

En este subbloque se realizará el primer entrenamiento de los modelos candidatos utilizando el período 2022–2024 y su evaluación sobre la validación temporal de 2025.

Debido al elevado volumen de datos, la preparación se realizará de forma incremental a partir de los archivos Parquet. Cada archivo se leerá únicamente con las columnas necesarias, se asignarán sus observaciones al conjunto de entrenamiento o validación según `FL_DATE` y se aplicará el preprocesamiento correspondiente.

Para controlar el consumo de memoria, las dos representaciones necesarias no se mantendrán simultáneamente. En primer lugar se construirá la representación con estandarización numérica, se entrenará y evaluará Logistic Regression y posteriormente se liberará esta matriz. A continuación se construirá la representación destinada a modelos basados en árboles, que será reutilizada por Decision Tree y Random Forest.

Las matrices transformadas se almacenarán en formato disperso CSR. Esta decisión resulta especialmente importante debido a las 843 variables generadas después del one-hot encoding y al elevado número de observaciones utilizadas en el experimento.

Los modelos serán evaluados mediante Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (PR-AUC), utilizando inicialmente un umbral de clasificación de 0.5. También se conservarán los componentes de la matriz de confusión para analizar posteriormente el comportamiento sobre la clase positiva.

Se registrará el tiempo empleado por cada modelo y los resultados se persistirán después de cada entrenamiento. De esta forma se mejora la trazabilidad del experimento y se reduce el riesgo de perder resultados ya calculados ante una interrupción de la sesión.

Las barras de progreso permitirán observar el avance durante la lectura y transformación de los archivos y durante la secuencia de modelos. Sin embargo, los estimadores de scikit-learn utilizados no proporcionan progreso interno mediante `tqdm` durante una llamada individual a `fit()`, por lo que la barra correspondiente permanecerá en el modelo actual hasta que finalice su entrenamiento.

In [ ]:
# ---------------------------------------------------------
# 1. Importación de herramientas para la ejecución
# ---------------------------------------------------------

import gc
import time

from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


# ---------------------------------------------------------
# 2. Configuración de la persistencia de resultados
# ---------------------------------------------------------

modeling_results_path = project_root / "results" / "modeling"
modeling_results_path.mkdir(
    parents=True,
    exist_ok=True,
)

initial_models_results_path = (
    modeling_results_path
    / "initial_models_results.csv"
)


# ---------------------------------------------------------
# 3. Ajuste funcional del codificador con categorías fijadas
# ---------------------------------------------------------

categorical_fit_sample = pd.DataFrame(
    {
        feature: [training_categories[feature][0]]
        for feature in categorical_modeling_features
    }
)

categorical_encoder.fit(
    categorical_fit_sample
)


# ---------------------------------------------------------
# 4. Función para construir incrementalmente una representación
# ---------------------------------------------------------

def build_modeling_matrices(numerical_mode):

    training_matrix_chunks = []
    validation_matrix_chunks = []

    training_target_chunks = []
    validation_target_chunks = []

    for parquet_file in tqdm(
        development_parquet_files,
        desc=f"Preparing {numerical_mode} matrices",
        unit="file",
    ):

        modeling_chunk = pd.read_parquet(
            parquet_file,
            columns=modeling_columns,
        )

        modeling_chunk[temporal_variable] = pd.to_datetime(
            modeling_chunk[temporal_variable]
        )

        training_mask = (
            (modeling_chunk[temporal_variable] >= training_start_date)
            & (modeling_chunk[temporal_variable] <= training_end_date)
        )

        validation_mask = (
            (modeling_chunk[temporal_variable] >= validation_start_date)
            & (modeling_chunk[temporal_variable] <= validation_end_date)
        )

        for dataset_name, dataset_mask in [
            ("training", training_mask),
            ("validation", validation_mask),
        ]:

            dataset_chunk = modeling_chunk.loc[
                dataset_mask,
                all_modeling_features + [target_variable],
            ]

            if dataset_chunk.empty:
                continue

            categorical_matrix = categorical_encoder.transform(
                dataset_chunk[categorical_modeling_features]
            )

            numerical_values = dataset_chunk[
                numerical_modeling_features
            ].to_numpy(
                dtype=np.float32,
                copy=True,
            )

            if numerical_mode == "scaled":
                numerical_values = numerical_scaler.transform(
                    numerical_values
                ).astype(
                    np.float32,
                    copy=False,
                )

            numerical_matrix = csr_matrix(
                numerical_values
            )

            transformed_chunk = hstack(
                [
                    categorical_matrix,
                    numerical_matrix,
                ],
                format="csr",
                dtype=np.float32,
            )

            target_chunk = dataset_chunk[
                target_variable
            ].to_numpy(
                dtype=np.int8,
                copy=True,
            )

            if dataset_name == "training":
                training_matrix_chunks.append(
                    transformed_chunk
                )
                training_target_chunks.append(
                    target_chunk
                )
            else:
                validation_matrix_chunks.append(
                    transformed_chunk
                )
                validation_target_chunks.append(
                    target_chunk
                )

        del modeling_chunk
        gc.collect()

    X_training = vstack(
        training_matrix_chunks,
        format="csr",
        dtype=np.float32,
    )

    X_validation = vstack(
        validation_matrix_chunks,
        format="csr",
        dtype=np.float32,
    )

    y_training = np.concatenate(
        training_target_chunks
    )

    y_validation = np.concatenate(
        validation_target_chunks
    )

    return (
        X_training,
        y_training,
        X_validation,
        y_validation,
    )


# ---------------------------------------------------------
# 5. Función de evaluación de un modelo
# ---------------------------------------------------------

def evaluate_initial_model(
    model_name,
    estimator,
    X_training,
    y_training,
    X_validation,
    y_validation,
):

    training_start = time.perf_counter()

    estimator.fit(
        X_training,
        y_training,
    )

    training_seconds = (
        time.perf_counter()
        - training_start
    )

    evaluation_start = time.perf_counter()

    validation_probability = estimator.predict_proba(
        X_validation
    )[:, 1]

    validation_prediction = (
        validation_probability
        >= initial_classification_threshold
    ).astype(np.int8)

    evaluation_seconds = (
        time.perf_counter()
        - evaluation_start
    )

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        validation_prediction,
        labels=[0, 1],
    ).ravel()

    result = {
        "model": model_name,
        "training_period": "2022-2024",
        "validation_period": "2025",
        "threshold": initial_classification_threshold,
        "imbalance_treatment": "None",
        "accuracy": accuracy_score(
            y_validation,
            validation_prediction,
        ),
        "precision": precision_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probability,
        ),
        "pr_auc": average_precision_score(
            y_validation,
            validation_probability,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "training_minutes": training_seconds / 60,
        "evaluation_minutes": evaluation_seconds / 60,
    }

    return result


# ---------------------------------------------------------
# 6. Inicialización de los resultados
# ---------------------------------------------------------

initial_models_results = []


# ---------------------------------------------------------
# 7. Preparación, entrenamiento y evaluación de Logistic Regression
# ---------------------------------------------------------

scaled_preparation_start = time.perf_counter()

(
    X_training_scaled,
    y_training_scaled,
    X_validation_scaled,
    y_validation_scaled,
) = build_modeling_matrices(
    numerical_mode="scaled"
)

scaled_preparation_minutes = (
    time.perf_counter()
    - scaled_preparation_start
) / 60

print(
    "Scaled training matrix:",
    X_training_scaled.shape,
)

print(
    "Scaled validation matrix:",
    X_validation_scaled.shape,
)

print(
    "Scaled preparation minutes:",
    round(scaled_preparation_minutes, 2),
)


for model_name in tqdm(
    ["Logistic Regression"],
    desc="Training scaled models",
    unit="model",
):

    result = evaluate_initial_model(
        model_name=model_name,
        estimator=logistic_regression,
        X_training=X_training_scaled,
        y_training=y_training_scaled,
        X_validation=X_validation_scaled,
        y_validation=y_validation_scaled,
    )

    result["preprocessing_minutes"] = (
        scaled_preparation_minutes
    )

    initial_models_results.append(
        result
    )

    pd.DataFrame(
        initial_models_results
    ).to_csv(
        initial_models_results_path,
        index=False,
    )


# ---------------------------------------------------------
# 8. Liberación de la representación escalada
# ---------------------------------------------------------

del X_training_scaled
del X_validation_scaled
del y_training_scaled
del y_validation_scaled

gc.collect()


# ---------------------------------------------------------
# 9. Preparación de la representación para árboles
# ---------------------------------------------------------

tree_preparation_start = time.perf_counter()

(
    X_training_tree,
    y_training_tree,
    X_validation_tree,
    y_validation_tree,
) = build_modeling_matrices(
    numerical_mode="tree"
)

tree_preparation_minutes = (
    time.perf_counter()
    - tree_preparation_start
) / 60

print(
    "Tree training matrix:",
    X_training_tree.shape,
)

print(
    "Tree validation matrix:",
    X_validation_tree.shape,
)

print(
    "Tree preparation minutes:",
    round(tree_preparation_minutes, 2),
)


# ---------------------------------------------------------
# 10. Entrenamiento y evaluación de los modelos de árboles
# ---------------------------------------------------------

tree_models = {
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
}

for model_name, estimator in tqdm(
    tree_models.items(),
    total=len(tree_models),
    desc="Training tree models",
    unit="model",
):

    result = evaluate_initial_model(
        model_name=model_name,
        estimator=estimator,
        X_training=X_training_tree,
        y_training=y_training_tree,
        X_validation=X_validation_tree,
        y_validation=y_validation_tree,
    )

    result["preprocessing_minutes"] = (
        tree_preparation_minutes
    )

    initial_models_results.append(
        result
    )

    pd.DataFrame(
        initial_models_results
    ).to_csv(
        initial_models_results_path,
        index=False,
    )


# ---------------------------------------------------------
# 11. Construcción del resultado consolidado
# ---------------------------------------------------------

initial_models_results = pd.DataFrame(
    initial_models_results
)

metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

initial_models_results[
    metric_columns
] = initial_models_results[
    metric_columns
].round(4)

initial_models_results[
    [
        "training_minutes",
        "evaluation_minutes",
        "preprocessing_minutes",
    ]
] = initial_models_results[
    [
        "training_minutes",
        "evaluation_minutes",
        "preprocessing_minutes",
    ]
].round(2)


# ---------------------------------------------------------
# 12. Persistencia definitiva de los resultados
# ---------------------------------------------------------

initial_models_results.to_csv(
    initial_models_results_path,
    index=False,
)


# ---------------------------------------------------------
# 13. Presentación de resultados
# ---------------------------------------------------------

display(
    initial_models_results[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "training_minutes",
            "evaluation_minutes",
            "preprocessing_minutes",
        ]
    ]
)

print(
    "Results persisted at:",
    initial_models_results_path,
)

#### Interpretación de los resultados

La evaluación inicial sobre la validación temporal de 2025 muestra que los tres modelos alcanzan valores de accuracy próximos al baseline de clase mayoritaria, situado en 0.7782. Este resultado confirma que la accuracy, considerada de forma aislada, resulta poco informativa para este problema debido al desbalance de la variable objetivo.

Las métricas basadas en capacidad discriminativa muestran, sin embargo, que los modelos han aprendido información predictiva. La regresión logística presenta el mejor comportamiento inicial, con un ROC-AUC de 0.6453 y un PR-AUC de 0.3325, seguida por Random Forest, con valores de 0.6412 y 0.3250 respectivamente. El árbol de decisión obtiene un rendimiento ligeramente inferior, con ROC-AUC de 0.6319 y PR-AUC de 0.3116. Todos estos resultados superan los valores del baseline, cuyo ROC-AUC es 0.5000 y cuyo PR-AUC es 0.2218.

A pesar de esta capacidad discriminativa, el comportamiento con el umbral de clasificación estándar de 0.5 es claramente insuficiente para la clase positiva. La regresión logística alcanza un recall de 0.0044, el árbol de decisión un recall de 0.0009 y Random Forest no genera predicciones positivas. Por tanto, los modelos presentan capacidad para ordenar observaciones según su riesgo de retraso, pero dicha información no se traduce todavía en una detección adecuada de los vuelos retrasados bajo la configuración inicial.

La regresión logística alcanzó además el límite de 500 iteraciones sin satisfacer el criterio de convergencia. Su resultado debe interpretarse como una referencia experimental válida para esta fase inicial, pero no como una configuración optimizada definitiva. El elevado coste computacional observado desaconseja incrementar sucesivamente el número de iteraciones sobre el conjunto completo sin una estrategia de optimización más dirigida.

Desde el punto de vista computacional, Random Forest presenta una relación especialmente favorable entre tiempo de entrenamiento y capacidad discriminativa, mientras que la regresión logística requiere un coste sustancialmente mayor. Estas diferencias deberán considerarse conjuntamente con las métricas predictivas en las fases posteriores de selección.

Los resultados justifican avanzar hacia el análisis del tratamiento del desbalance de clases. El objetivo será determinar si estrategias aplicadas exclusivamente sobre los datos de entrenamiento permiten mejorar la detección de ARR_DEL15 = 1 sin comprometer de forma desproporcionada la precisión ni la capacidad global de discriminación.

### 5.4 Comparación consolidada de resultados

Una vez evaluados los modelos candidatos sobre la validación temporal de 2025, se realiza una comparación consolidada frente al baseline de clase mayoritaria. El objetivo es determinar si los algoritmos entrenados incorporan capacidad predictiva respecto a la referencia mínima y caracterizar las diferencias existentes entre ellos antes de realizar las decisiones de optimización posteriores.

La comparación considera conjuntamente accuracy, precision, recall, F1, ROC-AUC y PR-AUC. Se presta especial atención a ROC-AUC y PR-AUC como medidas de capacidad discriminativa, así como a recall y F1 para evaluar el comportamiento efectivo sobre la clase positiva con el umbral inicial de 0.5.

Este análisis utiliza exclusivamente los resultados ya obtenidos y persistidos, por lo que no requiere repetir el entrenamiento de los modelos.

In [17]:
# ---------------------------------------------------------
# 1. Recuperación de los resultados persistidos de los modelos iniciales
# ---------------------------------------------------------

initial_models_results = pd.read_csv(initial_models_results_path)

comparison_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

required_model_columns = ["model", *comparison_metrics]

missing_model_columns = [
    column
    for column in required_model_columns
    if column not in initial_models_results.columns
]

if missing_model_columns:
    raise ValueError(
        f"Missing columns in initial model results: {missing_model_columns}"
    )


# ---------------------------------------------------------
# 2. Recuperación de los resultados del baseline
# ---------------------------------------------------------

baseline_comparison = baseline_results[
    comparison_metrics
].copy()

baseline_comparison.insert(
    0,
    "model",
    "Majority Class Baseline",
)


# ---------------------------------------------------------
# 3. Construcción de la comparación consolidada
# ---------------------------------------------------------

initial_model_comparison = initial_models_results[
    required_model_columns
].copy()

initial_models_comparison = pd.concat(
    [
        baseline_comparison,
        initial_model_comparison,
    ],
    ignore_index=True,
)

initial_models_comparison["roc_auc_gain_vs_baseline"] = (
    initial_models_comparison["roc_auc"]
    - float(baseline_results["roc_auc"].iloc[0])
)

initial_models_comparison["pr_auc_gain_vs_baseline"] = (
    initial_models_comparison["pr_auc"]
    - float(baseline_results["pr_auc"].iloc[0])
)


# ---------------------------------------------------------
# 4. Ordenación de los modelos candidatos por capacidad discriminativa
# ---------------------------------------------------------

candidate_ranking = (
    initial_models_comparison[
        initial_models_comparison["model"] != "Majority Class Baseline"
    ]
    .sort_values(
        by=["pr_auc", "roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

candidate_ranking.insert(
    0,
    "discrimination_rank",
    np.arange(1, len(candidate_ranking) + 1),
)


# ---------------------------------------------------------
# 5. Validación consolidada
# ---------------------------------------------------------

comparison_validation = pd.DataFrame(
    {
        "check": [
            "Baseline included",
            "Three candidate models included",
            "Four configurations compared",
            "All comparison metrics available",
            "All metric values finite",
            "All ROC-AUC values within valid range",
            "All PR-AUC values within valid range",
            "Candidate ranking contains three models",
        ],
        "valid": [
            "Majority Class Baseline"
            in initial_models_comparison["model"].values,
            len(initial_model_comparison) == 3,
            len(initial_models_comparison) == 4,
            not initial_models_comparison[
                comparison_metrics
            ].isna().any().any(),
            np.isfinite(
                initial_models_comparison[comparison_metrics].to_numpy()
            ).all(),
            initial_models_comparison["roc_auc"].between(0, 1).all(),
            initial_models_comparison["pr_auc"].between(0, 1).all(),
            len(candidate_ranking) == 3,
        ],
    }
)


# ---------------------------------------------------------
# 6. Presentación de los resultados consolidados
# ---------------------------------------------------------

display(
    initial_models_comparison.round(4)
)

display(
    candidate_ranking[
        [
            "discrimination_rank",
            "model",
            "roc_auc",
            "pr_auc",
            "precision",
            "recall",
            "f1",
        ]
    ].round(4)
)

display(comparison_validation)

print(
    "Initial model comparison valid:",
    comparison_validation["valid"].all(),
)

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,roc_auc_gain_vs_baseline,pr_auc_gain_vs_baseline
0,Majority Class Baseline,0.7782,0.0000,0.0000,0.0000,0.5000,0.2218,0.0000,0.0000
1,Logistic Regression,0.7781,0.4803,0.0044,0.0087,0.6453,0.3325,0.1453,0.1107
2,Decision Tree,0.7780,0.3555,0.0009,0.0018,0.6319,0.3116,0.1319,0.0898
3,Random Forest,0.7782,0.0000,0.0000,0.0000,0.6412,0.3250,0.1412,0.1032


,discrimination_rank,model,roc_auc,pr_auc,precision,recall,f1
0,1,Logistic Regression,0.6453,0.3325,0.4803,0.0044,0.0087
1,2,Random Forest,0.6412,0.3250,0.0000,0.0000,0.0000
2,3,Decision Tree,0.6319,0.3116,0.3555,0.0009,0.0018


,check,valid
0,Baseline included,True
1,Three candidate models included,True
2,Four configurations compared,True
3,All comparison metrics available,True
4,All metric values finite,True
5,All ROC-AUC values within valid range,True
6,All PR-AUC values within valid range,True
7,Candidate ranking contains three models,True


Initial model comparison valid: True


#### Interpretación

La comparación consolidada confirma que los tres modelos candidatos superan al baseline de clase mayoritaria en capacidad discriminativa, aunque sus valores de accuracy permanezcan prácticamente idénticos a la referencia. Este comportamiento evidencia que la accuracy no resulta adecuada como criterio principal de selección en un problema con una distribución de clases desbalanceada.

La regresión logística obtiene el mejor resultado inicial, con un ROC-AUC de 0.6453 y un PR-AUC de 0.3325. Respecto al baseline, esto representa incrementos de 0.1453 y 0.1107 respectivamente. Random Forest presenta un comportamiento próximo, con ROC-AUC de 0.6412 y PR-AUC de 0.3250, mientras que el árbol de decisión ocupa la tercera posición con valores de 0.6319 y 0.3116.

Los resultados muestran, por tanto, que los modelos contienen información útil para ordenar los vuelos según su probabilidad de retraso. Sin embargo, esta capacidad discriminativa no se traduce todavía en una detección efectiva de la clase positiva utilizando el umbral estándar de 0.5. La regresión logística presenta el mayor recall entre los candidatos, pero su valor de 0.0044 sigue siendo insuficiente desde el punto de vista predictivo. Random Forest no genera predicciones positivas bajo esta configuración inicial, pese a mantener métricas de ranking claramente superiores al baseline.

En consecuencia, la fase inicial permite identificar a la regresión logística y Random Forest como los modelos con mayor capacidad discriminativa. Los siguientes experimentos deberán orientarse a mejorar la detección de `ARR_DEL15 = 1`, prestando especial atención al recall y al criterio de decisión, manteniendo al mismo tiempo una capacidad discriminativa y una precisión aceptables para su utilización como sistema de alerta temprana.

## 6. Tratamiento del desbalance de clases

Los experimentos iniciales mostraron que los modelos candidatos presentan una capacidad discriminativa superior al baseline, especialmente en términos de ROC-AUC y PR-AUC. Sin embargo, esta capacidad no se traduce en una detección adecuada de los vuelos retrasados cuando se utiliza la configuración inicial sin tratamiento del desbalance y un umbral de clasificación de 0.5.

Este bloque estudia el efecto de diferentes estrategias de tratamiento del desbalance aplicadas exclusivamente sobre el conjunto de entrenamiento 2022–2024. La validación temporal de 2025 conservará en todo momento su distribución natural, evitando modificar artificialmente el escenario sobre el que se evalúa la capacidad de generalización.

Se compararán tres escenarios: ausencia de tratamiento como referencia, ponderación de clases mediante `class_weight` y submuestreo controlado de la clase mayoritaria. El objetivo no es construir artificialmente un conjunto perfectamente balanceado, sino determinar si una modificación razonable del proceso de aprendizaje permite aumentar la sensibilidad hacia ARR_DEL15 = 1 sin deteriorar de forma desproporcionada la precisión ni la capacidad discriminativa.

El bloque se organiza en los siguientes subbloques:

- 6.1 Definición de las estrategias de tratamiento.
- 6.2 Evaluación sobre la validación temporal.
- 6.3 Comparación consolidada del efecto del desbalance.

Como resultado se espera identificar qué estrategia ofrece el compromiso más adecuado entre detección de la clase positiva, falsos positivos y capacidad discriminativa, proporcionando una configuración justificada para las posteriores fases de optimización.

### 6.1 Definición de las estrategias de tratamiento

Se definen las estrategias que serán evaluadas para estudiar el efecto del desbalance de clases sobre el comportamiento de los modelos candidatos. La configuración sin tratamiento se mantiene como referencia experimental, permitiendo cuantificar posteriormente el efecto introducido por cada alternativa.

Como primera estrategia se considera la ponderación automática de clases mediante `class_weight="balanced"`. Este mecanismo modifica el peso de las observaciones durante el aprendizaje de acuerdo con la frecuencia de cada clase, sin eliminar registros del conjunto de entrenamiento.

Como segunda alternativa se plantea un submuestreo controlado de la clase mayoritaria. Dado el elevado volumen del conjunto de entrenamiento, no se propone construir automáticamente una distribución 50/50, ya que ello implicaría descartar una cantidad considerable de observaciones de la clase mayoritaria. En su lugar, se utilizará una relación de dos observaciones de la clase 0 por cada observación de la clase 1, conservando todos los ejemplos positivos y una muestra reproducible de la clase mayoritaria.

Las estrategias se aplicarán exclusivamente sobre el período de entrenamiento 2022–2024. La validación temporal de 2025 conservará su distribución original, garantizando que las métricas se calculen sobre una población no alterada.

In [19]:
# ---------------------------------------------------------
# 1. Recover training class distribution
# ---------------------------------------------------------

training_distribution = experimental_characterization.loc[
    experimental_characterization["dataset"] == "Training"
].iloc[0]

training_class_0 = int(training_distribution["class_0"])
training_class_1 = int(training_distribution["class_1"])

training_total = training_class_0 + training_class_1

original_training_ratio = (
    training_class_0 / training_class_1
)


# ---------------------------------------------------------
# 2. Define imbalance strategies
# ---------------------------------------------------------

undersampling_majority_to_minority_ratio = 2.0

undersampling_class_1 = training_class_1

undersampling_class_0 = min(
    training_class_0,
    int(
        undersampling_majority_to_minority_ratio
        * training_class_1
    ),
)

undersampling_total = (
    undersampling_class_0
    + undersampling_class_1
)

undersampling_positive_rate = (
    undersampling_class_1
    / undersampling_total
)

undersampling_retained_rate = (
    undersampling_total
    / training_total
)


# ---------------------------------------------------------
# 3. Build experimental strategy definition
# ---------------------------------------------------------

imbalance_strategies = pd.DataFrame(
    [
        {
            "strategy": "No treatment",
            "training_action": "Original distribution",
            "class_weight": None,
            "majority_to_minority_ratio": original_training_ratio,
            "expected_training_rows": training_total,
            "validation_modified": False,
        },
        {
            "strategy": "Class weight",
            "training_action": "Balanced class weights",
            "class_weight": "balanced",
            "majority_to_minority_ratio": original_training_ratio,
            "expected_training_rows": training_total,
            "validation_modified": False,
        },
        {
            "strategy": "Controlled undersampling",
            "training_action": "Majority class undersampling",
            "class_weight": None,
            "majority_to_minority_ratio": (
                undersampling_majority_to_minority_ratio
            ),
            "expected_training_rows": undersampling_total,
            "validation_modified": False,
        },
    ]
)


# ---------------------------------------------------------
# 4. Characterize controlled undersampling
# ---------------------------------------------------------

undersampling_summary = pd.DataFrame(
    {
        "metric": [
            "Original class 0 rows",
            "Original class 1 rows",
            "Original majority/minority ratio",
            "Selected class 0 rows",
            "Selected class 1 rows",
            "Undersampled training rows",
            "Undersampled positive rate",
            "Rows retained",
        ],
        "value": [
            training_class_0,
            training_class_1,
            original_training_ratio,
            undersampling_class_0,
            undersampling_class_1,
            undersampling_total,
            undersampling_positive_rate,
            undersampling_retained_rate,
        ],
    }
)


# ---------------------------------------------------------
# 5. Consolidated validation
# ---------------------------------------------------------

imbalance_strategy_validation = pd.DataFrame(
    {
        "check": [
            "Training rows match protocol",
            "Three strategies defined",
            "No-treatment reference included",
            "Class-weight strategy included",
            "Controlled undersampling included",
            "All minority observations retained",
            "Undersampling ratio below original ratio",
            "Undersampling does not force 1:1 balance",
            "Validation remains unmodified",
        ],
        "valid": [
            training_total == 21_399_093,
            len(imbalance_strategies) == 3,
            "No treatment"
            in imbalance_strategies["strategy"].values,
            "Class weight"
            in imbalance_strategies["strategy"].values,
            "Controlled undersampling"
            in imbalance_strategies["strategy"].values,
            undersampling_class_1 == training_class_1,
            (
                undersampling_majority_to_minority_ratio
                < original_training_ratio
            ),
            undersampling_majority_to_minority_ratio != 1.0,
            (
                imbalance_strategies["validation_modified"]
                == False
            ).all(),
        ],
    }
)


# ---------------------------------------------------------
# 6. Display strategy definition
# ---------------------------------------------------------

display(
    imbalance_strategies.round(4)
)

display(
    undersampling_summary.round(4)
)

display(
    imbalance_strategy_validation
)

print(
    "Imbalance strategies valid:",
    imbalance_strategy_validation["valid"].all(),
)

,strategy,training_action,class_weight,majority_to_minority_ratio,expected_training_rows,validation_modified
0,No treatment,Original distribution,None,3.8393,21399093,False
1,Class weight,Balanced class weights,balanced,3.8393,21399093,False
2,Controlled undersampling,Majority class undersampling,None,2.0000,13265772,False


,metric,value
0,Original class 0 rows,1.697717e+07
1,Original class 1 rows,4.421924e+06
2,Original majority/minority ratio,3.839300e+00
3,Selected class 0 rows,8.843848e+06
4,Selected class 1 rows,4.421924e+06
5,Undersampled training rows,1.326577e+07
6,Undersampled positive rate,3.333000e-01
7,Rows retained,6.199000e-01


,check,valid
0,Training rows match protocol,True
1,Three strategies defined,True
2,No-treatment reference included,True
3,Class-weight strategy included,True
4,Controlled undersampling included,True
5,All minority observations retained,True
6,Undersampling ratio below original ratio,True
7,Undersampling does not force 1:1 balance,True
8,Validation remains unmodified,True


Imbalance strategies valid: True


#### Interpretación de las estrategias de tratamiento del desbalance

La distribución original del conjunto de entrenamiento 2022–2024 presenta una relación aproximada de 3.84 observaciones de la clase 0 por cada observación de la clase 1. Sobre esta base se han definido tres escenarios experimentales: ausencia de tratamiento, ponderación automática de clases y submuestreo controlado de la clase mayoritaria.

La estrategia basada en `class_weight="balanced"` mantiene las 21,399,093 observaciones originales y modifica únicamente la importancia relativa de las clases durante el aprendizaje. Por el contrario, el submuestreo controlado reduce la relación mayoritaria/minoritaria hasta 2:1, conservando las 4,421,924 observaciones positivas y seleccionando 8,843,848 observaciones negativas.

Como resultado, el conjunto submuestreado contendría 13,265,772 observaciones, aproximadamente el 61.99 % del volumen original, y una prevalencia de la clase positiva cercana al 33.33 %. Esta configuración reduce la dominancia de la clase mayoritaria sin imponer una distribución artificialmente equilibrada al 50 %.

Las tres estrategias se aplicarán exclusivamente sobre el período de entrenamiento. La validación temporal de 2025 conservará su distribución natural, garantizando que las diferencias observadas en las métricas puedan atribuirse al procedimiento de aprendizaje y no a una alteración del conjunto de evaluación.

### 6.2 Evaluación sobre la validación temporal

Las estrategias definidas en el subbloque anterior se evalúan manteniendo constantes el conjunto de entrenamiento, la validación temporal, los predictores, el preprocesamiento y el umbral inicial de clasificación. De esta forma, las diferencias observadas pueden asociarse al tratamiento del desbalance y no a modificaciones simultáneas de otros componentes experimentales.

Los resultados obtenidos previamente sin tratamiento se reutilizan como referencia, evitando repetir entrenamientos ya realizados. Se entrenarán únicamente las configuraciones correspondientes a `class_weight="balanced"` y al submuestreo controlado 2:1 para Logistic Regression, Decision Tree y Random Forest.

La ponderación de clases utilizará el conjunto completo de entrenamiento 2022–2024. Para el submuestreo se conservarán todas las observaciones positivas y se seleccionará de forma reproducible, mediante `random_state=42`, una muestra de la clase mayoritaria que produzca la relación 2:1 definida previamente. La validación de 2025 permanecerá inalterada en todos los experimentos.

Debido al elevado volumen de datos y al coste observado en los entrenamientos iniciales, cada resultado se persistirá inmediatamente después de finalizar. Esta estrategia permite recuperar experimentos completados en caso de interrupción y evita repetir cálculos costosos.

In [20]:
# ---------------------------------------------------------
# 1. Import required utilities
# ---------------------------------------------------------

import gc
import time

from pathlib import Path

from scipy.sparse import hstack, vstack
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Configure experiment persistence
# ---------------------------------------------------------

imbalance_results_path = (
    modeling_results_path
    / "imbalance_results.csv"
)

imbalance_random_state = 42

imbalance_result_columns = [
    "model",
    "strategy",
    "training_rows",
    "validation_rows",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "training_minutes",
    "evaluation_minutes",
    "preprocessing_minutes",
]


# ---------------------------------------------------------
# 3. Recover completed imbalance experiments
# ---------------------------------------------------------

if imbalance_results_path.exists():
    imbalance_results = pd.read_csv(
        imbalance_results_path
    )

    print(
        "Recovered persisted imbalance experiments:",
        len(imbalance_results),
    )

else:
    imbalance_results = pd.DataFrame(
        columns=imbalance_result_columns
    )

    print(
        "No previous imbalance experiments found."
    )


# ---------------------------------------------------------
# 4. Add no-treatment results if not already persisted
# ---------------------------------------------------------

for _, initial_result in initial_models_results.iterrows():

    model_name = initial_result["model"]

    already_available = (
        (
            imbalance_results["model"] == model_name
        )
        & (
            imbalance_results["strategy"]
            == "No treatment"
        )
    ).any()

    if not already_available:

        baseline_model_result = {
            "model": model_name,
            "strategy": "No treatment",
            "training_rows": int(
                training_total
            ),
            "validation_rows": int(
                validation_summary["total_rows"]
            ),
            "accuracy": initial_result["accuracy"],
            "precision": initial_result["precision"],
            "recall": initial_result["recall"],
            "f1": initial_result["f1"],
            "roc_auc": initial_result["roc_auc"],
            "pr_auc": initial_result["pr_auc"],
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan,
            "training_minutes": initial_result[
                "training_minutes"
            ],
            "evaluation_minutes": initial_result[
                "evaluation_minutes"
            ],
            "preprocessing_minutes": initial_result[
                "preprocessing_minutes"
            ],
        }

        imbalance_results = pd.concat(
            [
                imbalance_results,
                pd.DataFrame(
                    [baseline_model_result]
                ),
            ],
            ignore_index=True,
        )

imbalance_results.to_csv(
    imbalance_results_path,
    index=False,
)


# ---------------------------------------------------------
# 5. Define cleaned matrix preparation procedure
# ---------------------------------------------------------

def build_imbalance_matrices(numerical_mode):

    preparation_start = time.perf_counter()

    training_X_chunks = []
    training_y_chunks = []

    validation_X_chunks = []
    validation_y_chunks = []

    for parquet_file in tqdm(
        development_parquet_files,
        desc=f"Preparing {numerical_mode} matrices",
        unit="file",
    ):

        dataset_chunk = pd.read_parquet(
            parquet_file,
            columns=modeling_columns,
        )

        dataset_chunk[temporal_variable] = pd.to_datetime(
            dataset_chunk[temporal_variable]
        )

        training_mask = (
            (
                dataset_chunk[temporal_variable]
                >= training_start_date
            )
            & (
                dataset_chunk[temporal_variable]
                <= training_end_date
            )
        )

        validation_mask = (
            (
                dataset_chunk[temporal_variable]
                >= validation_start_date
            )
            & (
                dataset_chunk[temporal_variable]
                <= validation_end_date
            )
        )

        for subset_name, subset_mask in [
            ("training", training_mask),
            ("validation", validation_mask),
        ]:

            if not subset_mask.any():
                continue

            subset = dataset_chunk.loc[
                subset_mask,
                all_modeling_features
                + [target_variable],
            ]

            categorical_matrix = (
                categorical_encoder.transform(
                    subset[
                        categorical_modeling_features
                    ]
                )
            )

            numerical_data = subset[
                numerical_modeling_features
            ]

            if numerical_mode == "scaled":

                numerical_matrix = (
                    numerical_scaler.transform(
                        numerical_data
                    )
                    .astype(np.float32)
                )

            elif numerical_mode == "tree":

                numerical_matrix = (
                    numerical_data
                    .to_numpy(dtype=np.float32)
                )

            else:
                raise ValueError(
                    "Unknown numerical mode: "
                    f"{numerical_mode}"
                )

            transformed_chunk = hstack(
                [
                    categorical_matrix,
                    numerical_matrix,
                ],
                format="csr",
                dtype=np.float32,
            )

            target_chunk = (
                subset[target_variable]
                .to_numpy(dtype=np.int8)
            )

            if subset_name == "training":

                training_X_chunks.append(
                    transformed_chunk
                )

                training_y_chunks.append(
                    target_chunk
                )

            else:

                validation_X_chunks.append(
                    transformed_chunk
                )

                validation_y_chunks.append(
                    target_chunk
                )

        del dataset_chunk
        gc.collect()

    X_train = vstack(
        training_X_chunks,
        format="csr",
    )

    y_train = np.concatenate(
        training_y_chunks
    )

    X_validation = vstack(
        validation_X_chunks,
        format="csr",
    )

    y_validation = np.concatenate(
        validation_y_chunks
    )

    preparation_minutes = (
        time.perf_counter()
        - preparation_start
    ) / 60

    return (
        X_train,
        y_train,
        X_validation,
        y_validation,
        preparation_minutes,
    )


# ---------------------------------------------------------
# 6. Define reproducible controlled undersampling
# ---------------------------------------------------------

def build_undersampled_training_set(
    X_train,
    y_train,
):

    random_generator = np.random.default_rng(
        imbalance_random_state
    )

    positive_indices = np.flatnonzero(
        y_train == 1
    )

    negative_indices = np.flatnonzero(
        y_train == 0
    )

    selected_negative_indices = (
        random_generator.choice(
            negative_indices,
            size=undersampling_class_0,
            replace=False,
        )
    )

    selected_indices = np.concatenate(
        [
            positive_indices,
            selected_negative_indices,
        ]
    )

    random_generator.shuffle(
        selected_indices
    )

    X_undersampled = X_train[
        selected_indices
    ]

    y_undersampled = y_train[
        selected_indices
    ]

    return (
        X_undersampled,
        y_undersampled,
    )


# ---------------------------------------------------------
# 7. Define model evaluation procedure
# ---------------------------------------------------------

def evaluate_imbalance_model(
    model_name,
    estimator,
    strategy_name,
    X_train,
    y_train,
    X_validation,
    y_validation,
    preprocessing_minutes,
):

    training_start = time.perf_counter()

    estimator.fit(
        X_train,
        y_train,
    )

    training_minutes = (
        time.perf_counter()
        - training_start
    ) / 60

    evaluation_start = time.perf_counter()

    validation_probability = (
        estimator.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_prediction = (
        validation_probability >= 0.5
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        validation_prediction,
        labels=[0, 1],
    ).ravel()

    result = {
        "model": model_name,
        "strategy": strategy_name,
        "training_rows": len(y_train),
        "validation_rows": len(y_validation),
        "accuracy": accuracy_score(
            y_validation,
            validation_prediction,
        ),
        "precision": precision_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probability,
        ),
        "pr_auc": average_precision_score(
            y_validation,
            validation_probability,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "training_minutes": training_minutes,
        "evaluation_minutes": (
            time.perf_counter()
            - evaluation_start
        ) / 60,
        "preprocessing_minutes": (
            preprocessing_minutes
        ),
    }

    return result


# ---------------------------------------------------------
# 8. Define experiment execution with checkpointing
# ---------------------------------------------------------

def run_imbalance_experiment(
    model_name,
    base_estimator,
    strategy_name,
    X_train,
    y_train,
    X_validation,
    y_validation,
    preprocessing_minutes,
):

    global imbalance_results

    experiment_completed = (
        (
            imbalance_results["model"]
            == model_name
        )
        & (
            imbalance_results["strategy"]
            == strategy_name
        )
    ).any()

    if experiment_completed:

        print(
            f"Skipping completed experiment: "
            f"{model_name} | {strategy_name}"
        )

        return

    estimator = clone(
        base_estimator
    )

    if strategy_name == "Class weight":

        estimator.set_params(
            class_weight="balanced"
        )

        experiment_X_train = X_train
        experiment_y_train = y_train

    elif strategy_name == "Controlled undersampling":

        estimator.set_params(
            class_weight=None
        )

        (
            experiment_X_train,
            experiment_y_train,
        ) = build_undersampled_training_set(
            X_train,
            y_train,
        )

    else:
        raise ValueError(
            f"Unknown strategy: {strategy_name}"
        )

    result = evaluate_imbalance_model(
        model_name=model_name,
        estimator=estimator,
        strategy_name=strategy_name,
        X_train=experiment_X_train,
        y_train=experiment_y_train,
        X_validation=X_validation,
        y_validation=y_validation,
        preprocessing_minutes=preprocessing_minutes,
    )

    imbalance_results = pd.concat(
        [
            imbalance_results,
            pd.DataFrame([result]),
        ],
        ignore_index=True,
    )

    imbalance_results.to_csv(
        imbalance_results_path,
        index=False,
    )

    print(
        f"Persisted: "
        f"{model_name} | {strategy_name}"
    )

    del estimator

    if strategy_name == "Controlled undersampling":
        del experiment_X_train
        del experiment_y_train

    gc.collect()


# ---------------------------------------------------------
# 9. Evaluate Logistic Regression strategies
# ---------------------------------------------------------

(
    X_train_scaled,
    y_train_scaled,
    X_validation_scaled,
    y_validation_scaled,
    scaled_preparation_minutes,
) = build_imbalance_matrices(
    numerical_mode="scaled"
)

print(
    "Scaled training matrix:",
    X_train_scaled.shape,
)

print(
    "Scaled validation matrix:",
    X_validation_scaled.shape,
)

print(
    "Scaled preparation minutes:",
    round(
        scaled_preparation_minutes,
        2,
    ),
)

for strategy_name in tqdm(
    [
        "Class weight",
        "Controlled undersampling",
    ],
    desc="Logistic Regression imbalance strategies",
    unit="strategy",
):

    run_imbalance_experiment(
        model_name="Logistic Regression",
        base_estimator=logistic_regression,
        strategy_name=strategy_name,
        X_train=X_train_scaled,
        y_train=y_train_scaled,
        X_validation=X_validation_scaled,
        y_validation=y_validation_scaled,
        preprocessing_minutes=(
            scaled_preparation_minutes
        ),
    )


del X_train_scaled
del y_train_scaled
del X_validation_scaled
del y_validation_scaled

gc.collect()


# ---------------------------------------------------------
# 10. Evaluate tree-based strategies
# ---------------------------------------------------------

(
    X_train_tree,
    y_train_tree,
    X_validation_tree,
    y_validation_tree,
    tree_preparation_minutes,
) = build_imbalance_matrices(
    numerical_mode="tree"
)

print(
    "Tree training matrix:",
    X_train_tree.shape,
)

print(
    "Tree validation matrix:",
    X_validation_tree.shape,
)

print(
    "Tree preparation minutes:",
    round(
        tree_preparation_minutes,
        2,
    ),
)

tree_models = {
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
}

for model_name, base_estimator in tqdm(
    tree_models.items(),
    total=len(tree_models),
    desc="Tree models",
    unit="model",
):

    for strategy_name in [
        "Class weight",
        "Controlled undersampling",
    ]:

        run_imbalance_experiment(
            model_name=model_name,
            base_estimator=base_estimator,
            strategy_name=strategy_name,
            X_train=X_train_tree,
            y_train=y_train_tree,
            X_validation=X_validation_tree,
            y_validation=y_validation_tree,
            preprocessing_minutes=(
                tree_preparation_minutes
            ),
        )


# ---------------------------------------------------------
# 11. Release large matrices
# ---------------------------------------------------------

del X_train_tree
del y_train_tree
del X_validation_tree
del y_validation_tree

gc.collect()


# ---------------------------------------------------------
# 12. Consolidate and validate results
# ---------------------------------------------------------

imbalance_results = pd.read_csv(
    imbalance_results_path
)

expected_experiments = (
    len(candidate_models)
    * len(imbalance_strategies)
)

imbalance_results_validation = pd.DataFrame(
    {
        "check": [
            "Nine experiments available",
            "Three models available",
            "Three strategies available",
            "Validation size constant",
            "All predictive metrics available",
            "All metric values finite",
        ],
        "valid": [
            len(imbalance_results)
            == expected_experiments,
            imbalance_results[
                "model"
            ].nunique() == 3,
            imbalance_results[
                "strategy"
            ].nunique() == 3,
            (
                imbalance_results[
                    "validation_rows"
                ]
                == int(
                    validation_summary[
                        "total_rows"
                    ]
                )
            ).all(),
            not imbalance_results[
                [
                    "accuracy",
                    "precision",
                    "recall",
                    "f1",
                    "roc_auc",
                    "pr_auc",
                ]
            ].isna().any().any(),
            np.isfinite(
                imbalance_results[
                    [
                        "accuracy",
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy()
            ).all(),
        ],
    }
)


# ---------------------------------------------------------
# 13. Display experiment results
# ---------------------------------------------------------

display(
    imbalance_results[
        [
            "model",
            "strategy",
            "training_rows",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "training_minutes",
        ]
    ]
    .sort_values(
        ["model", "strategy"]
    )
    .round(4)
)

display(
    imbalance_results_validation
)

print(
    "Imbalance evaluation valid:",
    imbalance_results_validation[
        "valid"
    ].all(),
)

print(
    "Results persisted at:",
    imbalance_results_path,
)

No previous imbalance experiments found.


C:\Users\ranie\AppData\Local\Temp\ipykernel_16608\3264874423.py:130: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  imbalance_results = pd.concat(


Preparing scaled matrices:   0%|          | 0/320 [00:00<?, ?file/s]

Scaled training matrix: (21399093, 843)
Scaled validation matrix: (7597494, 843)
Scaled preparation minutes: 2.27


Logistic Regression imbalance strategies:   0%|          | 0/2 [00:00<?, ?strategy/s]

...

Tree models:   0%|          | 0/2 [00:00<?, ?model/s]

Persisted: Decision Tree | Class weight
Persisted: Decision Tree | Controlled undersampling
Persisted: Random Forest | Class weight
Persisted: Random Forest | Controlled undersampling


,model,strategy,training_rows,accuracy,precision,recall,f1,roc_auc,pr_auc,training_minutes
5,Decision Tree,Class weight,21399093,0.5864,0.2928,0.6108,0.3959,0.6331,0.3145,69.5688
6,Decision Tree,Controlled undersampling,13265772,0.7671,0.4204,0.1314,0.2003,0.6327,0.3143,55.0111
1,Decision Tree,No treatment,21399093,0.7780,0.3555,0.0009,0.0018,0.6319,0.3116,29.7000
3,Logistic Regression,Class weight,21399093,0.6137,0.3079,0.5944,0.4057,0.6456,0.3307,329.7436
4,Logistic Regression,Controlled undersampling,13265772,0.7638,0.4099,0.1474,0.2168,0.6454,0.3317,267.0962
0,Logistic Regression,No treatment,21399093,0.7781,0.4803,0.0044,0.0087,0.6453,0.3325,326.5800
7,Random Forest,Class weight,21399093,0.5948,0.2983,0.6112,0.4009,0.6410,0.3244,23.0682
8,Random Forest,Controlled undersampling,13265772,0.7782,0.5045,0.0017,0.0035,0.6411,0.3248,14.8795
2,Random Forest,No treatment,21399093,0.7782,0.0000,0.0000,0.0000,0.6412,0.3250,17.3800


,check,valid
0,Nine experiments available,True
1,Three models available,True
2,Three strategies available,True
3,Validation size constant,True
4,All predictive metrics available,True
5,All metric values finite,True


Imbalance evaluation valid: True
Results persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\imbalance_results.csv


#### Interpretación de la evaluación del tratamiento del desbalance

La evaluación sobre la validación temporal de 2025 evidencia que el tratamiento del desbalance modifica sustancialmente el comportamiento de clasificación de los modelos, especialmente en relación con la detección de ARR_DEL15 = 1.

La ponderación automática de clases mediante `class_weight="balanced"` produce el incremento más acusado del recall. Logistic Regression pasa de 0.0044 a 0.5944, Decision Tree de 0.0009 a 0.6108 y Random Forest de 0.0000 a 0.6112. Esta mejora de sensibilidad está acompañada por una reducción de precision, situada entre 0.2928 y 0.3079, y por una disminución de la accuracy respecto a las configuraciones sin tratamiento. Este comportamiento refleja el aumento del número de observaciones clasificadas como positivas y, por tanto, un compromiso entre detección de retrasos y generación de falsos positivos.

El submuestreo controlado 2:1 presenta un comportamiento más moderado y dependiente del algoritmo. Logistic Regression alcanza un recall de 0.1474 y un F1 de 0.2168, mientras que Decision Tree obtiene un recall de 0.1314 y un F1 de 0.2003. En Random Forest, por el contrario, el recall permanece prácticamente nulo, con un valor de 0.0017.

Las métricas de discriminación muestran una estabilidad notable entre las distintas estrategias. Los valores de ROC-AUC y PR-AUC varían solo ligeramente dentro de cada familia de modelos. Este resultado indica que el tratamiento del desbalance afecta principalmente al comportamiento de clasificación asociado al umbral de 0.5, mientras que la capacidad relativa de ordenar las observaciones según su riesgo de retraso permanece aproximadamente estable.

Desde el punto de vista computacional, el submuestreo reduce el conjunto de entrenamiento a 13,265,772 observaciones y disminuye los tiempos de entrenamiento, aunque este ahorro no se corresponde con la mejora de recall alcanzada mediante ponderación de clases.

Las configuraciones de Logistic Regression volvieron a alcanzar el máximo de 500 iteraciones sin satisfacer el criterio de convergencia. Por este motivo, sus resultados son válidos como referencia experimental de las configuraciones evaluadas, pero la convergencia deberá considerarse específicamente durante la posterior optimización del modelo.

En conjunto, los resultados muestran que `class_weight="balanced"` constituye la estrategia con mayor capacidad para recuperar observaciones de la clase positiva entre las alternativas evaluadas, mientras que el submuestreo 2:1 proporciona un compromiso más conservador para Logistic Regression y Decision Tree. La comparación definitiva entre estas estrategias se realizará considerando conjuntamente recall, precision, F1, capacidad discriminativa y coste computacional.

### 6.3 Comparación consolidada del efecto del desbalance

Una vez evaluadas las estrategias de tratamiento del desbalance sobre la validación temporal de 2025, se realiza una comparación consolidada para identificar su efecto sobre la detección de la clase positiva y sobre la capacidad discriminativa de cada modelo.

El análisis considera conjuntamente precision, recall, F1, ROC-AUC y PR-AUC, además de las variaciones respecto a la configuración sin tratamiento. El objetivo es determinar qué estrategia mejora de forma más consistente la identificación de ARR_DEL15 = 1 sin deteriorar de manera desproporcionada el comportamiento global del modelo.

La comparación se realiza dentro de cada familia de modelos para aislar el efecto del tratamiento del desbalance y evitar atribuir a la estrategia diferencias que puedan deberse al algoritmo utilizado.

In [21]:
# ---------------------------------------------------------
# 1. Recover imbalance experiment results
# ---------------------------------------------------------

imbalance_results = pd.read_csv(
    imbalance_results_path
)


# ---------------------------------------------------------
# 2. Recover no-treatment reference by model
# ---------------------------------------------------------

no_treatment_reference = (
    imbalance_results[
        imbalance_results["strategy"] == "No treatment"
    ]
    .set_index("model")
)


# ---------------------------------------------------------
# 3. Compute changes versus no treatment
# ---------------------------------------------------------

imbalance_comparison = (
    imbalance_results.copy()
)

for metric in [
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]:

    imbalance_comparison[
        f"{metric}_gain_vs_no_treatment"
    ] = imbalance_comparison.apply(
        lambda row: (
            row[metric]
            - no_treatment_reference.loc[
                row["model"],
                metric,
            ]
        ),
        axis=1,
    )


# ---------------------------------------------------------
# 4. Rank strategies within each model
# ---------------------------------------------------------

strategy_ranking = (
    imbalance_comparison[
        imbalance_comparison["strategy"] != "No treatment"
    ]
    .sort_values(
        by=[
            "model",
            "f1",
            "recall",
            "pr_auc",
        ],
        ascending=[
            True,
            False,
            False,
            False,
        ],
    )
    .copy()
)

strategy_ranking[
    "strategy_rank"
] = (
    strategy_ranking
    .groupby("model")
    .cumcount()
    + 1
)


# ---------------------------------------------------------
# 5. Identify best treatment by model
# ---------------------------------------------------------

best_strategy_by_model = (
    strategy_ranking[
        strategy_ranking["strategy_rank"] == 1
    ][
        [
            "model",
            "strategy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "precision_gain_vs_no_treatment",
            "recall_gain_vs_no_treatment",
            "f1_gain_vs_no_treatment",
            "roc_auc_gain_vs_no_treatment",
            "pr_auc_gain_vs_no_treatment",
            "training_minutes",
        ]
    ]
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Consolidated validation
# ---------------------------------------------------------

imbalance_comparison_validation = pd.DataFrame(
    {
        "check": [
            "Nine experiments available",
            "Three models represented",
            "Three strategies represented",
            "Reference available for every model",
            "Metric gains computed",
            "Best strategy identified for every model",
            "All comparison values finite",
        ],
        "valid": [
            len(imbalance_comparison) == 9,
            imbalance_comparison["model"].nunique() == 3,
            imbalance_comparison["strategy"].nunique() == 3,
            len(no_treatment_reference) == 3,
            all(
                f"{metric}_gain_vs_no_treatment"
                in imbalance_comparison.columns
                for metric in [
                    "precision",
                    "recall",
                    "f1",
                    "roc_auc",
                    "pr_auc",
                ]
            ),
            len(best_strategy_by_model) == 3,
            np.isfinite(
                imbalance_comparison[
                    [
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy()
            ).all(),
        ],
    }
)


# ---------------------------------------------------------
# 7. Display consolidated comparison
# ---------------------------------------------------------

display(
    imbalance_comparison[
        [
            "model",
            "strategy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "precision_gain_vs_no_treatment",
            "recall_gain_vs_no_treatment",
            "f1_gain_vs_no_treatment",
        ]
    ]
    .sort_values(
        ["model", "strategy"]
    )
    .round(4)
)

display(
    best_strategy_by_model.round(4)
)

display(
    imbalance_comparison_validation
)

print(
    "Imbalance comparison valid:",
    imbalance_comparison_validation[
        "valid"
    ].all(),
)

,model,strategy,precision,recall,f1,roc_auc,pr_auc,precision_gain_vs_no_treatment,recall_gain_vs_no_treatment,f1_gain_vs_no_treatment
5,Decision Tree,Class weight,0.2928,0.6108,0.3959,0.6331,0.3145,-0.0627,0.6099,0.3941
6,Decision Tree,Controlled undersampling,0.4204,0.1314,0.2003,0.6327,0.3143,0.0649,0.1305,0.1985
1,Decision Tree,No treatment,0.3555,0.0009,0.0018,0.6319,0.3116,0.0000,0.0000,0.0000
3,Logistic Regression,Class weight,0.3079,0.5944,0.4057,0.6456,0.3307,-0.1724,0.5900,0.3970
4,Logistic Regression,Controlled undersampling,0.4099,0.1474,0.2168,0.6454,0.3317,-0.0704,0.1430,0.2081
0,Logistic Regression,No treatment,0.4803,0.0044,0.0087,0.6453,0.3325,0.0000,0.0000,0.0000
7,Random Forest,Class weight,0.2983,0.6112,0.4009,0.6410,0.3244,0.2983,0.6112,0.4009
8,Random Forest,Controlled undersampling,0.5045,0.0017,0.0035,0.6411,0.3248,0.5045,0.0017,0.0035
2,Random Forest,No treatment,0.0000,0.0000,0.0000,0.6412,0.3250,0.0000,0.0000,0.0000


,model,strategy,precision,recall,f1,roc_auc,pr_auc,precision_gain_vs_no_treatment,recall_gain_vs_no_treatment,f1_gain_vs_no_treatment,roc_auc_gain_vs_no_treatment,pr_auc_gain_vs_no_treatment,training_minutes
0,Decision Tree,Class weight,0.2928,0.6108,0.3959,0.6331,0.3145,-0.0627,0.6099,0.3941,0.0012,0.0029,69.5688
1,Logistic Regression,Class weight,0.3079,0.5944,0.4057,0.6456,0.3307,-0.1724,0.5900,0.3970,0.0003,-0.0018,329.7436
2,Random Forest,Class weight,0.2983,0.6112,0.4009,0.6410,0.3244,0.2983,0.6112,0.4009,-0.0002,-0.0006,23.0682


,check,valid
0,Nine experiments available,True
1,Three models represented,True
2,Three strategies represented,True
3,Reference available for every model,True
4,Metric gains computed,True
5,Best strategy identified for every model,True
6,All comparison values finite,True


Imbalance comparison valid: True


#### Interpretación de la comparación del tratamiento del desbalance

La comparación consolidada confirma que `class_weight="balanced"` constituye la estrategia más efectiva de las evaluadas para mejorar la detección de la clase positiva. Esta estrategia obtiene el mayor F1 dentro de las tres familias de modelos y eleva el recall hasta valores próximos al 60 %, frente a los valores prácticamente nulos observados inicialmente.

Logistic Regression alcanza con ponderación de clases un recall de 0.5944 y el mayor F1 de la comparación, con 0.4057. Random Forest presenta un comportamiento próximo, con el mayor recall observado, 0.6112, y un F1 de 0.4009. Decision Tree alcanza un recall de 0.6108 y un F1 de 0.3959.

El incremento de sensibilidad producido por la ponderación de clases implica una reducción de precision respecto a las configuraciones inicialmente capaces de generar predicciones positivas. Este comportamiento refleja el compromiso esperado entre la recuperación de un mayor número de vuelos retrasados y el incremento de falsos positivos.

Las métricas de discriminación permanecen comparativamente estables entre las estrategias. En particular, las variaciones de ROC-AUC y PR-AUC respecto a las configuraciones sin tratamiento son reducidas. Por tanto, la principal consecuencia de la ponderación de clases se observa en el comportamiento de clasificación asociado al umbral de 0.5 y no en una modificación sustancial de la capacidad de ranking de los modelos.

El submuestreo controlado 2:1 ofrece mejoras más moderadas para Logistic Regression y Decision Tree y no consigue una mejora relevante del recall en Random Forest. Aunque reduce el volumen y el coste del entrenamiento, sus resultados no superan el compromiso F1–recall alcanzado mediante ponderación de clases.

En consecuencia, `class_weight="balanced"` se selecciona como estrategia principal de tratamiento del desbalance para la siguiente fase experimental. Logistic Regression y Random Forest destacan como los candidatos de mayor interés: el primero presenta el mejor F1 y la mayor capacidad discriminativa, mientras que el segundo ofrece resultados próximos con un coste computacional considerablemente inferior.

## 7. Optimización de los modelos finalistas

Los experimentos anteriores permitieron identificar Logistic Regression y Random Forest como los modelos de mayor interés para continuar el proceso de selección. Logistic Regression presentó la mayor capacidad discriminativa y el mayor F1 con ponderación de clases, mientras que Random Forest obtuvo resultados próximos con un coste computacional considerablemente inferior.

La optimización se realizará exclusivamente utilizando el período de desarrollo, manteniendo 2026 completamente aislado. El entrenamiento 2022–2024 continuará siendo la fuente de aprendizaje y 2025 la referencia temporal para la selección de configuraciones.

Debido al volumen del conjunto de entrenamiento y a los tiempos observados en los experimentos anteriores, no resulta computacionalmente razonable evaluar una malla extensa de hiperparámetros directamente sobre los más de 21 millones de observaciones. Por este motivo se utilizará una estrategia de búsqueda dirigida en dos etapas. Primero se realizará un cribado computacional sobre una muestra controlada del entrenamiento para identificar configuraciones prometedoras. Posteriormente, únicamente las configuraciones finalistas se confirmarán utilizando el conjunto completo 2022–2024 y la validación temporal completa de 2025.

La ponderación de clases mediante `class_weight="balanced"` se mantiene fija durante esta fase, dado que fue la estrategia seleccionada en el bloque anterior. De este modo, la optimización se concentra en los hiperparámetros propios de los algoritmos y evita volver a evaluar decisiones metodológicas ya comparadas.

El bloque se organiza en los siguientes subbloques:

1. Definición del espacio de búsqueda.
2. Evaluación inicial de configuraciones sobre una muestra controlada.
3. Confirmación temporal de las configuraciones finalistas.
4. Selección de hiperparámetros y modelo candidato.

El objetivo final es obtener una configuración predictiva competitiva y computacionalmente viable, prestando especial atención a su capacidad para aproximarse a un Recall del 80 % sobre la clase positiva sin deteriorar de forma excesiva Precision y el resto de métricas, antes de seleccionar el modelo definitivo del sistema de alerta temprana.

### 7.1 Definición justificada del espacio de hiperparámetros

El espacio de hiperparámetros se define de forma dirigida para Logistic Regression y Random Forest, seleccionados como modelos finalistas tras la evaluación inicial y el análisis del desbalance.

En Logistic Regression se estudiará principalmente el parámetro de regularización `C`, que controla la intensidad de la regularización aplicada al modelo. Se mantendrán la penalización L2 y la ponderación balanceada de clases. Dado que las configuraciones anteriores alcanzaron el límite de iteraciones sin satisfacer el criterio de convergencia, también se evaluará una tolerancia menos restrictiva con el objetivo de estudiar un compromiso entre convergencia numérica y coste computacional.

En Random Forest se evaluarán variaciones controladas del número de árboles, la profundidad máxima y el número mínimo de observaciones por hoja. Estos parámetros permiten estudiar el compromiso entre capacidad del modelo, regularización y coste computacional. La ponderación balanceada de clases se mantendrá fija para garantizar comparabilidad con los resultados del bloque anterior.

El espacio se mantiene deliberadamente reducido. El objetivo no es realizar una búsqueda exhaustiva sobre todas las combinaciones posibles, sino evaluar un conjunto limitado de configuraciones justificadas que permita identificar regiones prometedoras antes de la confirmación sobre el conjunto completo.

In [22]:
# ---------------------------------------------------------
# 1. Definición del espacio de búsqueda de Logistic Regression
# ---------------------------------------------------------

logistic_regression_search_space = {
    "C": [
        0.1,
        1.0,
        10.0,
    ],
    "tol": [
        1e-3,
        1e-2,
    ],
}

logistic_fixed_parameters = {
    "solver": "saga",
    "penalty": "l2",
    "class_weight": "balanced",
    "max_iter": 500,
    "random_state": random_state,
}


# ---------------------------------------------------------
# 2. Definición del espacio de búsqueda de Random Forest
# ---------------------------------------------------------

random_forest_search_space = {
    "n_estimators": [
        100,
        200,
    ],
    "max_depth": [
        10,
        20,
    ],
    "min_samples_leaf": [
        50,
        100,
    ],
}

random_forest_fixed_parameters = {
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": random_state,
}


# ---------------------------------------------------------
# 3. Cálculo del número de configuraciones candidatas
# ---------------------------------------------------------

logistic_configuration_count = (
    len(logistic_regression_search_space["C"])
    * len(logistic_regression_search_space["tol"])
)

random_forest_configuration_count = (
    len(random_forest_search_space["n_estimators"])
    * len(random_forest_search_space["max_depth"])
    * len(random_forest_search_space["min_samples_leaf"])
)

total_configuration_count = (
    logistic_configuration_count
    + random_forest_configuration_count
)


# ---------------------------------------------------------
# 4. Construcción del resumen del espacio de búsqueda
# ---------------------------------------------------------

hyperparameter_search_summary = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "optimized_parameters": "C, tol",
            "fixed_imbalance_strategy": (
                'class_weight="balanced"'
            ),
            "candidate_configurations": (
                logistic_configuration_count
            ),
        },
        {
            "model": "Random Forest",
            "optimized_parameters": (
                "n_estimators, max_depth, "
                "min_samples_leaf"
            ),
            "fixed_imbalance_strategy": (
                'class_weight="balanced"'
            ),
            "candidate_configurations": (
                random_forest_configuration_count
            ),
        },
    ]
)


# ---------------------------------------------------------
# 5. Estimación del coste computacional sobre datos completos
# ---------------------------------------------------------

logistic_reference_minutes = float(
    imbalance_results.loc[
        (
            imbalance_results["model"]
            == "Logistic Regression"
        )
        & (
            imbalance_results["strategy"]
            == "Class weight"
        ),
        "training_minutes",
    ].iloc[0]
)

random_forest_reference_minutes = float(
    imbalance_results.loc[
        (
            imbalance_results["model"]
            == "Random Forest"
        )
        & (
            imbalance_results["strategy"]
            == "Class weight"
        ),
        "training_minutes",
    ].iloc[0]
)

naive_full_search_hours = (
    (
        logistic_configuration_count
        * logistic_reference_minutes
    )
    + (
        random_forest_configuration_count
        * random_forest_reference_minutes
    )
) / 60


# ---------------------------------------------------------
# 6. Validación consolidada
# ---------------------------------------------------------

hyperparameter_space_validation = pd.DataFrame(
    {
        "check": [
            "Two finalist models included",
            "Logistic search space is non-empty",
            "Random Forest search space is non-empty",
            "Class weighting remains fixed",
            "Logistic configurations limited",
            "Random Forest configurations limited",
            "Total configurations limited",
            "External test remains after validation",
        ],
        "valid": [
            len(hyperparameter_search_summary) == 2,
            logistic_configuration_count > 0,
            random_forest_configuration_count > 0,
            (
                logistic_fixed_parameters[
                    "class_weight"
                ]
                == "balanced"
            ),
            logistic_configuration_count <= 6,
            random_forest_configuration_count <= 8,
            total_configuration_count <= 14,
            validation_end_date < external_test_start_date,
        ],
    }
)


# ---------------------------------------------------------
# 7. Presentación de la definición del espacio de búsqueda
# ---------------------------------------------------------

display(
    hyperparameter_search_summary
)

display(
    hyperparameter_space_validation
)

print(
    "Logistic Regression configurations:",
    logistic_configuration_count,
)

print(
    "Random Forest configurations:",
    random_forest_configuration_count,
)

print(
    "Total candidate configurations:",
    total_configuration_count,
)

print(
    "Estimated naive full-data search hours:",
    round(
        naive_full_search_hours,
        2,
    ),
)

print(
    "Hyperparameter search space valid:",
    hyperparameter_space_validation[
        "valid"
    ].all(),
)

,model,optimized_parameters,fixed_imbalance_strategy,candidate_configurations
0,Logistic Regression,"C, tol","class_weight=""balanced""",6
1,Random Forest,"n_estimators, max_depth, min_samples_leaf","class_weight=""balanced""",8


,check,valid
0,Two finalist models included,True
1,Logistic search space is non-empty,True
2,Random Forest search space is non-empty,True
3,Class weighting remains fixed,True
4,Logistic configurations limited,True
5,Random Forest configurations limited,True
6,Total configurations limited,True
7,External test not involved,True


Logistic Regression configurations: 6
Random Forest configurations: 8
Total candidate configurations: 14
Estimated naive full-data search hours: 36.05
Hyperparameter search space valid: True


#### Interpretación del espacio de hiperparámetros

El espacio de optimización queda compuesto por 14 configuraciones: seis correspondientes a Logistic Regression y ocho a Random Forest. En ambos modelos se mantiene fija la ponderación de clases mediante `class_weight="balanced"`, seleccionada previamente como estrategia principal para mejorar la detección de la clase positiva.

Para Logistic Regression se evaluarán tres niveles de regularización mediante `C` y dos valores de tolerancia, dando lugar a seis configuraciones. Esta búsqueda permitirá estudiar simultáneamente el efecto de la regularización y un posible compromiso entre convergencia numérica y coste computacional, aspecto especialmente relevante debido a los límites de iteraciones alcanzados en los experimentos anteriores.

Para Random Forest se evaluarán combinaciones controladas de número de árboles, profundidad máxima y número mínimo de observaciones por hoja, generando ocho configuraciones. El espacio permite modificar la capacidad y regularización del modelo manteniendo un número reducido de experimentos.

Utilizando como referencia los tiempos reales registrados anteriormente, la ejecución directa de las 14 configuraciones sobre el conjunto completo de entrenamiento supondría aproximadamente 36.05 horas de cómputo. Este coste justifica la utilización de una etapa previa de cribado computacional sobre una muestra controlada del período 2022–2024.

Por tanto, el espacio definido se considera suficientemente amplio para estudiar configuraciones alternativas de los dos modelos finalistas, pero deliberadamente limitado para mantener la viabilidad computacional del proceso. El test externo de 2026 permanece completamente excluido de esta fase.

### 7.2 Búsqueda de configuraciones

La búsqueda inicial de hiperparámetros se realiza mediante un cribado computacional sobre una muestra reproducible del período de entrenamiento, evitando evaluar las 14 configuraciones directamente sobre los más de 21 millones de observaciones disponibles entre 2022 y 2024.

Para preservar la estructura temporal del problema, el cribado utiliza 2022–2023 como período de ajuste y 2024 como validación temporal interna. De esta manera, la validación principal de 2025 permanece fuera de la búsqueda inicial y podrá utilizarse posteriormente para confirmar las configuraciones finalistas con menor riesgo de sobreajuste asociado a decisiones experimentales repetidas.

Se selecciona aleatoriamente y de forma reproducible el 5 % de las observaciones de cada período. El muestreo no modifica artificialmente la distribución de clases y se utiliza exclusivamente como aproximación computacional para identificar regiones prometedoras del espacio de hiperparámetros.

El preprocesamiento utilizado durante este cribado se aprende exclusivamente con información de 2022–2023. Las categorías del One-Hot Encoding y los parámetros de estandarización no utilizan información de 2024.

La búsqueda se realiza mediante `GridSearchCV` con una única partición temporal predefinida. Se calculan precision, recall, F1, ROC-AUC y Average Precision, utilizando esta última como criterio de `refit` debido a su especial utilidad en problemas con clases desbalanceadas. La detección final de la clase positiva y la selección del umbral se analizarán posteriormente de forma independiente.

Los resultados de cada modelo se persistirán inmediatamente después de finalizar su búsqueda para reducir el riesgo de pérdida de experimentos costosos.

In [23]:
# ---------------------------------------------------------
# 1. Importación de componentes necesarios
# ---------------------------------------------------------

import gc
import time

from scipy.sparse import hstack, vstack
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Definición de la configuración del cribado temporal
# ---------------------------------------------------------

screening_train_start_date = pd.Timestamp("2022-01-01")
screening_train_end_date = pd.Timestamp("2023-12-31")

screening_validation_start_date = pd.Timestamp("2024-01-01")
screening_validation_end_date = pd.Timestamp("2024-12-31")

screening_sample_fraction = 0.05
screening_random_state = 42

logistic_screening_results_path = (
    modeling_results_path
    / "grid_search_logistic_screening.csv"
)

random_forest_screening_results_path = (
    modeling_results_path
    / "grid_search_random_forest_screening.csv"
)


# ---------------------------------------------------------
# 3. Aprendizaje del vocabulario categórico y escalado
#    numérico exclusivamente a partir de 2022-2023
# ---------------------------------------------------------

screening_categories_sets = {
    feature: set()
    for feature in categorical_modeling_features
}

screening_numerical_scaler = StandardScaler()

screening_scaler_fitted = False
screening_encoder_fit_sample = None
screening_training_rows_full = 0

for parquet_file in tqdm(
    development_parquet_files,
    desc="Learning 2022-2023 preprocessing",
    unit="file",
):

    data_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            *categorical_modeling_features,
            *numerical_modeling_features,
        ],
    )

    data_chunk[temporal_variable] = pd.to_datetime(
        data_chunk[temporal_variable]
    )

    train_mask = (
        (
            data_chunk[temporal_variable]
            >= screening_train_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= screening_train_end_date
        )
    )

    training_chunk = data_chunk.loc[
        train_mask
    ]

    if training_chunk.empty:
        continue

    screening_training_rows_full += len(
        training_chunk
    )

    for feature in categorical_modeling_features:

        screening_categories_sets[
            feature
        ].update(
            training_chunk[
                feature
            ].dropna().unique().tolist()
        )

    numerical_chunk = training_chunk[
        numerical_modeling_features
    ]

    screening_numerical_scaler.partial_fit(
        numerical_chunk
    )

    screening_scaler_fitted = True

    if screening_encoder_fit_sample is None:

        screening_encoder_fit_sample = (
            training_chunk[
                categorical_modeling_features
            ]
            .iloc[[0]]
            .copy()
        )


# ---------------------------------------------------------
# 4. Construcción del codificador categórico con el
#    vocabulario de 2022-2023
# ---------------------------------------------------------

screening_categories = {
    feature: sorted(
        screening_categories_sets[feature]
    )
    for feature in categorical_modeling_features
}

screening_categorical_encoder = OneHotEncoder(
    categories=[
        screening_categories[feature]
        for feature
        in categorical_modeling_features
    ],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)

screening_categorical_encoder.fit(
    screening_encoder_fit_sample
)

screening_categorical_feature_count = sum(
    len(categories)
    for categories
    in screening_categories.values()
)

screening_total_feature_count = (
    screening_categorical_feature_count
    + len(numerical_modeling_features)
)


# ---------------------------------------------------------
# 5. Construcción reproducible de la muestra temporal del 5 %
# ---------------------------------------------------------

screening_rng = np.random.default_rng(
    screening_random_state
)

X_screening_train_chunks = []
y_screening_train_chunks = []

X_screening_validation_chunks = []
y_screening_validation_chunks = []

sample_start_time = time.time()

for parquet_file in tqdm(
    development_parquet_files,
    desc="Building 5% screening sample",
    unit="file",
):

    data_chunk = pd.read_parquet(
        parquet_file,
        columns=modeling_columns,
    )

    data_chunk[temporal_variable] = pd.to_datetime(
        data_chunk[temporal_variable]
    )

    train_mask = (
        (
            data_chunk[temporal_variable]
            >= screening_train_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= screening_train_end_date
        )
    )

    validation_mask = (
        (
            data_chunk[temporal_variable]
            >= screening_validation_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= screening_validation_end_date
        )
    )

    for (
        period_mask,
        X_chunks,
        y_chunks,
    ) in [
        (
            train_mask,
            X_screening_train_chunks,
            y_screening_train_chunks,
        ),
        (
            validation_mask,
            X_screening_validation_chunks,
            y_screening_validation_chunks,
        ),
    ]:

        period_chunk = data_chunk.loc[
            period_mask
        ]

        if period_chunk.empty:
            continue

        sample_mask = (
            screening_rng.random(
                len(period_chunk)
            )
            < screening_sample_fraction
        )

        sampled_chunk = period_chunk.loc[
            sample_mask
        ]

        if sampled_chunk.empty:
            continue

        categorical_matrix = (
            screening_categorical_encoder.transform(
                sampled_chunk[
                    categorical_modeling_features
                ]
            )
        )

        numerical_matrix = (
            screening_numerical_scaler
            .transform(
                sampled_chunk[
                    numerical_modeling_features
                ]
            )
            .astype(np.float32)
        )

        transformed_matrix = hstack(
            [
                categorical_matrix,
                numerical_matrix,
            ],
            format="csr",
            dtype=np.float32,
        )

        X_chunks.append(
            transformed_matrix
        )

        y_chunks.append(
            sampled_chunk[
                target_variable
            ]
            .to_numpy(
                dtype=np.int8
            )
        )


X_screening_train = vstack(
    X_screening_train_chunks,
    format="csr",
)

y_screening_train = np.concatenate(
    y_screening_train_chunks
)

X_screening_validation = vstack(
    X_screening_validation_chunks,
    format="csr",
)

y_screening_validation = np.concatenate(
    y_screening_validation_chunks
)

screening_preparation_minutes = (
    time.time()
    - sample_start_time
) / 60


# ---------------------------------------------------------
# 6. Creación de la partición temporal predefinida
# ---------------------------------------------------------

X_screening = vstack(
    [
        X_screening_train,
        X_screening_validation,
    ],
    format="csr",
)

y_screening = np.concatenate(
    [
        y_screening_train,
        y_screening_validation,
    ]
)

test_fold = np.concatenate(
    [
        np.full(
            len(y_screening_train),
            -1,
            dtype=np.int8,
        ),
        np.zeros(
            len(y_screening_validation),
            dtype=np.int8,
        ),
    ]
)

temporal_screening_split = PredefinedSplit(
    test_fold=test_fold
)


# ---------------------------------------------------------
# 7. Definición de las métricas de evaluación
# ---------------------------------------------------------

screening_scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}


# ---------------------------------------------------------
# 8. Ejecución de GridSearchCV para Logistic Regression
# ---------------------------------------------------------

if logistic_screening_results_path.exists():

    logistic_grid_results = pd.read_csv(
        logistic_screening_results_path
    )

    print(
        "Recovered Logistic Regression "
        "screening results."
    )

else:

    logistic_screening_estimator = (
        LogisticRegression(
            **logistic_fixed_parameters
        )
    )

    logistic_grid_search = GridSearchCV(
        estimator=logistic_screening_estimator,
        param_grid=logistic_regression_search_space,
        scoring=screening_scoring,
        refit="average_precision",
        cv=temporal_screening_split,
        n_jobs=1,
        verbose=2,
        return_train_score=False,
    )

    logistic_search_start = time.time()

    logistic_grid_search.fit(
        X_screening,
        y_screening,
    )

    logistic_search_minutes = (
        time.time()
        - logistic_search_start
    ) / 60

    logistic_grid_results = pd.DataFrame(
        logistic_grid_search.cv_results_
    )

    logistic_grid_results[
        "search_minutes"
    ] = logistic_search_minutes

    logistic_grid_results.to_csv(
        logistic_screening_results_path,
        index=False,
    )

    print(
        "Persisted Logistic Regression "
        "screening results."
    )


# ---------------------------------------------------------
# 9. Ejecución de GridSearchCV para Random Forest
# ---------------------------------------------------------

if random_forest_screening_results_path.exists():

    random_forest_grid_results = pd.read_csv(
        random_forest_screening_results_path
    )

    print(
        "Recovered Random Forest "
        "screening results."
    )

else:

    random_forest_screening_estimator = (
        RandomForestClassifier(
            **random_forest_fixed_parameters
        )
    )

    random_forest_grid_search = GridSearchCV(
        estimator=random_forest_screening_estimator,
        param_grid=random_forest_search_space,
        scoring=screening_scoring,
        refit="average_precision",
        cv=temporal_screening_split,
        n_jobs=1,
        verbose=2,
        return_train_score=False,
    )

    random_forest_search_start = time.time()

    random_forest_grid_search.fit(
        X_screening,
        y_screening,
    )

    random_forest_search_minutes = (
        time.time()
        - random_forest_search_start
    ) / 60

    random_forest_grid_results = pd.DataFrame(
        random_forest_grid_search.cv_results_
    )

    random_forest_grid_results.to_csv(
        random_forest_screening_results_path,
        index=False,
    )

    random_forest_grid_results[
        "search_minutes"
    ] = (
        random_forest_search_minutes
    )

    random_forest_grid_results.to_csv(
        random_forest_screening_results_path,
        index=False,
    )

    print(
        "Persisted Random Forest "
        "screening results."
    )


# ---------------------------------------------------------
# 10. Construcción del ranking consolidado del cribado
# ---------------------------------------------------------

logistic_screening_ranking = pd.DataFrame(
    {
        "model": "Logistic Regression",
        "C": logistic_grid_results[
            "param_C"
        ],
        "tol": logistic_grid_results[
            "param_tol"
        ],
        "n_estimators": np.nan,
        "max_depth": np.nan,
        "min_samples_leaf": np.nan,
        "precision": logistic_grid_results[
            "mean_test_precision"
        ],
        "recall": logistic_grid_results[
            "mean_test_recall"
        ],
        "f1": logistic_grid_results[
            "mean_test_f1"
        ],
        "roc_auc": logistic_grid_results[
            "mean_test_roc_auc"
        ],
        "pr_auc": logistic_grid_results[
            "mean_test_average_precision"
        ],
    }
)

random_forest_screening_ranking = pd.DataFrame(
    {
        "model": "Random Forest",
        "C": np.nan,
        "tol": np.nan,
        "n_estimators": (
            random_forest_grid_results[
                "param_n_estimators"
            ]
        ),
        "max_depth": (
            random_forest_grid_results[
                "param_max_depth"
            ]
        ),
        "min_samples_leaf": (
            random_forest_grid_results[
                "param_min_samples_leaf"
            ]
        ),
        "precision": (
            random_forest_grid_results[
                "mean_test_precision"
            ]
        ),
        "recall": (
            random_forest_grid_results[
                "mean_test_recall"
            ]
        ),
        "f1": (
            random_forest_grid_results[
                "mean_test_f1"
            ]
        ),
        "roc_auc": (
            random_forest_grid_results[
                "mean_test_roc_auc"
            ]
        ),
        "pr_auc": (
            random_forest_grid_results[
                "mean_test_average_precision"
            ]
        ),
    }
)

screening_ranking = pd.concat(
    [
        logistic_screening_ranking,
        random_forest_screening_ranking,
    ],
    ignore_index=True,
)

screening_ranking[
    "rank_within_model"
] = (
    screening_ranking
    .groupby("model")[
        "pr_auc"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


# ---------------------------------------------------------
# 11. Validación consolidada
# ---------------------------------------------------------

screening_validation = pd.DataFrame(
    {
        "check": [
            "Screening train sample available",
            "Screening validation sample available",
            "Both classes in screening train",
            "Both classes in screening validation",
            "Expected number of transformed features",
            "Six Logistic configurations evaluated",
            "Eight Random Forest configurations evaluated",
            "All predictive metrics finite",
            "2025 excluded from screening",
            "2026 excluded from screening",
        ],
        "valid": [
            len(y_screening_train) > 0,
            len(y_screening_validation) > 0,
            np.unique(
                y_screening_train
            ).size == 2,
            np.unique(
                y_screening_validation
            ).size == 2,
            (
                X_screening.shape[1]
                == screening_total_feature_count
            ),
            len(
                logistic_grid_results
            ) == logistic_configuration_count,
            len(
                random_forest_grid_results
            ) == random_forest_configuration_count,
            np.isfinite(
                screening_ranking[
                    [
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy()
            ).all(),
            screening_validation_end_date.year == 2024,
            screening_validation_end_date.year < 2026,
        ],
    }
)


# ---------------------------------------------------------
# 12. Presentación de los resultados del cribado
# ---------------------------------------------------------

print(
    "Screening train sample:",
    X_screening_train.shape,
)

print(
    "Screening validation sample:",
    X_screening_validation.shape,
)

print(
    "Screening train positive rate:",
    round(
        y_screening_train.mean(),
        4,
    ),
)

print(
    "Screening validation positive rate:",
    round(
        y_screening_validation.mean(),
        4,
    ),
)

print(
    "Screening preparation minutes:",
    round(
        screening_preparation_minutes,
        2,
    ),
)

display(
    screening_ranking
    .sort_values(
        [
            "model",
            "rank_within_model",
        ]
    )
    .round(4)
)

display(
    screening_validation
)

print(
    "Screening valid:",
    screening_validation[
        "valid"
    ].all(),
)

print(
    "Logistic results:",
    logistic_screening_results_path,
)

print(
    "Random Forest results:",
    random_forest_screening_results_path,
)


# ---------------------------------------------------------
# 13. Liberación de objetos combinados del cribado
#     que ya no son necesarios
# ---------------------------------------------------------

gc.collect()

Learning 2022-2023 preprocessing:   0%|          | 0/320 [00:00<?, ?file/s]

Building 5% screening sample:   0%|          | 0/320 [00:00<?, ?file/s]

Fitting 1 folds for each of 6 candidates, totalling 6 fits
[CV] END ...................................C=0.1, tol=0.001; total time=  17.9s
[CV] END ....................................C=0.1, tol=0.01; total time=   8.4s
[CV] END ...................................C=1.0, tol=0.001; total time= 3.5min
[CV] END ....................................C=1.0, tol=0.01; total time=  26.6s
[CV] END ..................................C=10.0, tol=0.001; total time= 3.7min
[CV] END ...................................C=10.0, tol=0.01; total time=  27.5s
Persisted Logistic Regression screening results.
Fitting 1 folds for each of 8 candidates, totalling 8 fits
[CV] END max_depth=10, min_samples_leaf=50, n_estimators=100; total time=   9.2s
[CV] END max_depth=10, min_samples_leaf=50, n_estimators=200; total time=  16.8s
[CV] END max_depth=10, min_samples_leaf=100, n_estimators=100; total time=   8.3s
[CV] END max_depth=10, min_samples_leaf=100, n_estimators=200; total time=  19.5s
[CV] END max_depth=20

,model,C,tol,n_estimators,max_depth,min_samples_leaf,precision,recall,f1,roc_auc,pr_auc,rank_within_model
0,Logistic Regression,0.1,0.001,NaN,NaN,NaN,0.2903,0.6149,0.3943,0.6505,0.3154,1
1,Logistic Regression,0.1,0.010,NaN,NaN,NaN,0.2902,0.6149,0.3943,0.6505,0.3154,2
3,Logistic Regression,1.0,0.010,NaN,NaN,NaN,0.2904,0.6142,0.3944,0.6502,0.3151,3
5,Logistic Regression,10.0,0.010,NaN,NaN,NaN,0.2904,0.6142,0.3944,0.6502,0.3151,4
4,Logistic Regression,10.0,0.001,NaN,NaN,NaN,0.2904,0.6141,0.3943,0.6502,0.3151,5
2,Logistic Regression,1.0,0.001,NaN,NaN,NaN,0.2904,0.6141,0.3943,0.6502,0.3151,6
11,Random Forest,NaN,NaN,200.0,20.0,50.0,0.2846,0.6114,0.3884,0.6463,0.3079,1
10,Random Forest,NaN,NaN,100.0,20.0,50.0,0.2851,0.6100,0.3886,0.6460,0.3076,2
12,Random Forest,NaN,NaN,100.0,20.0,100.0,0.2831,0.6136,0.3875,0.6445,0.3064,3
13,Random Forest,NaN,NaN,200.0,20.0,100.0,0.2826,0.6149,0.3873,0.6445,0.3062,4


,check,valid
0,Screening train sample available,True
1,Screening validation sample available,True
2,Both classes in screening train,True
3,Both classes in screening validation,True
4,Expected number of transformed features,True
5,Six Logistic configurations evaluated,True
6,Eight Random Forest configurations evaluated,True
7,All predictive metrics finite,True
8,2025 excluded from screening,True
9,2026 excluded from screening,True


Screening valid: True
Logistic results: G:\My Drive\MASTER Big Data\TFM\results\modeling\grid_search_logistic_screening.csv
Random Forest results: G:\My Drive\MASTER Big Data\TFM\results\modeling\grid_search_random_forest_screening.csv


201

#### Interpretación

El cribado computacional se realizó correctamente utilizando una muestra reproducible del 5 % y preservando la estructura temporal del experimento. La muestra de ajuste correspondiente a 2022–2023 quedó formada por 698,187 observaciones, mientras que la validación temporal interna de 2024 incluyó 370,887 observaciones. Las tasas de la clase positiva fueron 20.69 % y 20.62 %, respectivamente, manteniéndose próximas a la distribución observada en los conjuntos completos.

El preprocesamiento aprendido exclusivamente sobre 2022–2023 generó 841 variables transformadas. La dimensionalidad ligeramente inferior a la obtenida posteriormente con 2022–2024 es coherente con la restricción temporal aplicada al aprendizaje del vocabulario categórico y evita incorporar información procedente del período utilizado como validación interna.

En Logistic Regression se observaron diferencias reducidas entre las seis configuraciones evaluadas. La mayor Average Precision se obtuvo con `C=0.1` y `tol=0.001`, alcanzando un ROC-AUC de 0.6505 y un PR-AUC de 0.3154, con recall de 0.6149 y F1 de 0.3943. La configuración equivalente con `tol=0.01` presentó métricas prácticamente idénticas y un menor tiempo de ajuste, lo que evidencia la conveniencia de considerar conjuntamente rendimiento predictivo y eficiencia computacional durante la confirmación posterior.

En Random Forest, la configuración con 200 estimadores, profundidad máxima de 20 y un mínimo de 50 observaciones por hoja obtuvo la mayor Average Precision, con ROC-AUC de 0.6463 y PR-AUC de 0.3079. Las configuraciones con profundidad 20 superaron a las de profundidad 10 en capacidad discriminativa, mientras que el incremento de 100 a 200 árboles produjo diferencias comparativamente reducidas.

Logistic Regression presentó durante el cribado los mayores valores de ROC-AUC y PR-AUC, aunque las diferencias respecto a Random Forest permanecen moderadas. Por tanto, estos resultados se utilizan exclusivamente para identificar configuraciones prometedoras y no para seleccionar todavía el modelo definitivo.

La totalidad de las comprobaciones del cribado resultó satisfactoria. El período 2025 permaneció excluido de esta búsqueda y 2026 continúa completamente aislado, preservando la estructura temporal establecida para las fases posteriores de selección y evaluación externa.

### 7.3 Evaluación temporal de las configuraciones finalistas

Los resultados del cribado permiten reducir el espacio inicial de 14 combinaciones a un conjunto limitado de configuraciones finalistas. Esta etapa tiene como objetivo comprobar si las configuraciones identificadas sobre la muestra controlada mantienen su comportamiento cuando se entrenan utilizando la totalidad del período 2022–2024 y se evalúan sobre la validación temporal completa de 2025.

Para Logistic Regression se seleccionan dos configuraciones con `C=0.1`, diferenciadas por la tolerancia utilizada. Ambas presentaron prácticamente el mismo rendimiento durante el cribado, pero `tol=0.01` mostró un coste computacional inferior. Su evaluación sobre el conjunto completo permitirá determinar si esta reducción del coste se mantiene sin una pérdida relevante de capacidad predictiva.

Para Random Forest se seleccionan las dos mejores configuraciones con profundidad máxima de 20 y un mínimo de 50 observaciones por hoja, utilizando 100 y 200 estimadores. Esta comparación permitirá determinar si duplicar el número de árboles aporta una mejora suficientemente relevante para justificar el incremento del coste computacional.

En todos los casos se mantiene `class_weight="balanced"`, seleccionado previamente como estrategia de tratamiento del desbalance. El preprocesamiento aprendido exclusivamente sobre 2022–2024 y la validación temporal de 2025 permanecen constantes entre las configuraciones.

La comparación considera precision, recall, F1, ROC-AUC y Average Precision, además del tiempo de entrenamiento. El objetivo no es maximizar una única métrica de forma aislada, sino identificar configuraciones que ofrezcan un equilibrio adecuado entre detección de la clase positiva, capacidad discriminativa y coste computacional.

El test externo correspondiente a enero–mayo de 2026 permanece completamente excluido de esta fase.

In [24]:
# ---------------------------------------------------------
# 1. Importación de componentes necesarios
# ---------------------------------------------------------

import gc
import time

from pathlib import Path

from scipy.sparse import hstack, vstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Definición de las configuraciones finalistas
# ---------------------------------------------------------

finalist_configurations = [
    {
        "configuration": "LR_C0.1_tol0.001",
        "model": "Logistic Regression",
        "C": 0.1,
        "tol": 1e-3,
        "n_estimators": np.nan,
        "max_depth": np.nan,
        "min_samples_leaf": np.nan,
    },
    {
        "configuration": "LR_C0.1_tol0.01",
        "model": "Logistic Regression",
        "C": 0.1,
        "tol": 1e-2,
        "n_estimators": np.nan,
        "max_depth": np.nan,
        "min_samples_leaf": np.nan,
    },
    {
        "configuration": "RF_100_depth20_leaf50",
        "model": "Random Forest",
        "C": np.nan,
        "tol": np.nan,
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_leaf": 50,
    },
    {
        "configuration": "RF_200_depth20_leaf50",
        "model": "Random Forest",
        "C": np.nan,
        "tol": np.nan,
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_leaf": 50,
    },
]

finalist_results_path = (
    modeling_results_path
    / "finalist_temporal_validation_results.csv"
)


# ---------------------------------------------------------
# 3. Recuperación de resultados finalistas previos si existen
# ---------------------------------------------------------

finalist_result_columns = [
    "configuration",
    "model",
    "C",
    "tol",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "training_rows",
    "validation_rows",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "training_minutes",
    "evaluation_minutes",
]

if finalist_results_path.exists():

    finalist_results = pd.read_csv(
        finalist_results_path
    )

    print(
        "Recovered persisted finalist results:",
        len(finalist_results),
    )

else:

    finalist_results = pd.DataFrame(
        columns=finalist_result_columns
    )

    print(
        "No previous finalist results found."
    )


# ---------------------------------------------------------
# 4. Construcción de las matrices completas 2022-2024 y 2025
# ---------------------------------------------------------

X_train_scaled_chunks = []
X_validation_scaled_chunks = []

X_train_tree_chunks = []
X_validation_tree_chunks = []

y_train_chunks = []
y_validation_chunks = []

preparation_start = time.time()

for parquet_file in tqdm(
    development_parquet_files,
    desc="Preparing full finalist matrices",
    unit="file",
):

    data_chunk = pd.read_parquet(
        parquet_file,
        columns=modeling_columns,
    )

    data_chunk[temporal_variable] = pd.to_datetime(
        data_chunk[temporal_variable]
    )

    train_mask = (
        (
            data_chunk[temporal_variable]
            >= training_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= training_end_date
        )
    )

    validation_mask = (
        (
            data_chunk[temporal_variable]
            >= validation_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= validation_end_date
        )
    )

    for (
        period_mask,
        scaled_chunks,
        tree_chunks,
        target_chunks,
    ) in [
        (
            train_mask,
            X_train_scaled_chunks,
            X_train_tree_chunks,
            y_train_chunks,
        ),
        (
            validation_mask,
            X_validation_scaled_chunks,
            X_validation_tree_chunks,
            y_validation_chunks,
        ),
    ]:

        period_chunk = data_chunk.loc[
            period_mask
        ]

        if period_chunk.empty:
            continue

        categorical_matrix = (
            categorical_encoder.transform(
                period_chunk[
                    categorical_modeling_features
                ]
            )
        )

        numerical_scaled = (
            numerical_scaler
            .transform(
                period_chunk[
                    numerical_modeling_features
                ]
            )
            .astype(np.float32)
        )

        numerical_tree = (
            period_chunk[
                numerical_modeling_features
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        scaled_matrix = hstack(
            [
                categorical_matrix,
                numerical_scaled,
            ],
            format="csr",
            dtype=np.float32,
        )

        tree_matrix = hstack(
            [
                categorical_matrix,
                numerical_tree,
            ],
            format="csr",
            dtype=np.float32,
        )

        scaled_chunks.append(
            scaled_matrix
        )

        tree_chunks.append(
            tree_matrix
        )

        target_chunks.append(
            period_chunk[
                target_variable
            ].to_numpy(
                dtype=np.int8
            )
        )


X_train_scaled = vstack(
    X_train_scaled_chunks,
    format="csr",
)

X_validation_scaled = vstack(
    X_validation_scaled_chunks,
    format="csr",
)

X_train_tree = vstack(
    X_train_tree_chunks,
    format="csr",
)

X_validation_tree = vstack(
    X_validation_tree_chunks,
    format="csr",
)

y_train = np.concatenate(
    y_train_chunks
)

y_validation = np.concatenate(
    y_validation_chunks
)

preparation_minutes = (
    time.time()
    - preparation_start
) / 60


# ---------------------------------------------------------
# 5. Evaluación de las configuraciones finalistas
# ---------------------------------------------------------

for configuration in tqdm(
    finalist_configurations,
    desc="Evaluating finalists",
    unit="configuration",
):

    configuration_name = (
        configuration["configuration"]
    )

    already_completed = (
        configuration_name
        in finalist_results[
            "configuration"
        ].astype(str).values
    )

    if already_completed:

        print(
            "Skipping completed:",
            configuration_name,
        )

        continue

    if (
        configuration["model"]
        == "Logistic Regression"
    ):

        estimator = LogisticRegression(
            C=configuration["C"],
            tol=configuration["tol"],
            solver="saga",
            penalty="l2",
            class_weight="balanced",
            max_iter=500,
            random_state=random_state,
        )

        X_train_current = X_train_scaled
        X_validation_current = (
            X_validation_scaled
        )

    else:

        estimator = RandomForestClassifier(
            n_estimators=int(
                configuration[
                    "n_estimators"
                ]
            ),
            max_depth=int(
                configuration[
                    "max_depth"
                ]
            ),
            min_samples_leaf=int(
                configuration[
                    "min_samples_leaf"
                ]
            ),
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
        )

        X_train_current = X_train_tree
        X_validation_current = (
            X_validation_tree
        )

    training_start = time.time()

    estimator.fit(
        X_train_current,
        y_train,
    )

    training_minutes = (
        time.time()
        - training_start
    ) / 60

    evaluation_start = time.time()

    validation_probability = (
        estimator.predict_proba(
            X_validation_current
        )[:, 1]
    )

    validation_prediction = (
        validation_probability
        >= 0.5
    ).astype(np.int8)

    evaluation_minutes = (
        time.time()
        - evaluation_start
    ) / 60

    result = {
        "configuration": (
            configuration_name
        ),
        "model": configuration["model"],
        "C": configuration["C"],
        "tol": configuration["tol"],
        "n_estimators": (
            configuration["n_estimators"]
        ),
        "max_depth": (
            configuration["max_depth"]
        ),
        "min_samples_leaf": (
            configuration[
                "min_samples_leaf"
            ]
        ),
        "training_rows": len(y_train),
        "validation_rows": len(
            y_validation
        ),
        "accuracy": accuracy_score(
            y_validation,
            validation_prediction,
        ),
        "precision": precision_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "f1": f1_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probability,
        ),
        "pr_auc": (
            average_precision_score(
                y_validation,
                validation_probability,
            )
        ),
        "training_minutes": (
            training_minutes
        ),
        "evaluation_minutes": (
            evaluation_minutes
        ),
    }

    finalist_results = pd.concat(
        [
            finalist_results,
            pd.DataFrame([result]),
        ],
        ignore_index=True,
    )

    finalist_results.to_csv(
        finalist_results_path,
        index=False,
    )

    print(
        "Persisted:",
        configuration_name,
    )


# ---------------------------------------------------------
# 6. Ordenación de las configuraciones confirmadas
# ---------------------------------------------------------

finalist_ranking = (
    finalist_results
    .sort_values(
        by=[
            "pr_auc",
            "roc_auc",
            "f1",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

finalist_ranking[
    "overall_rank"
] = (
    np.arange(
        1,
        len(finalist_ranking) + 1,
    )
)


# ---------------------------------------------------------
# 7. Validación consolidada
# ---------------------------------------------------------

finalist_validation = pd.DataFrame(
    {
        "check": [
            "Four finalist configurations available",
            "Two Logistic configurations available",
            "Two Random Forest configurations available",
            "Training size matches protocol",
            "Validation size matches protocol",
            "All predictive metrics finite",
            "All configurations use class weighting",
            "2026 excluded from confirmation",
        ],
        "valid": [
            len(finalist_results) == 4,
            (
                finalist_results[
                    "model"
                ]
                == "Logistic Regression"
            ).sum() == 2,
            (
                finalist_results[
                    "model"
                ]
                == "Random Forest"
            ).sum() == 2,
            (
                finalist_results[
                    "training_rows"
                ].astype(int)
                == 21_399_093
            ).all(),
            (
                finalist_results[
                    "validation_rows"
                ].astype(int)
                == 7_597_494
            ).all(),
            np.isfinite(
                finalist_results[
                    [
                        "accuracy",
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy(
                    dtype=float
                )
            ).all(),
            True,
            validation_end_date.year < 2026,
        ],
    }
)


# ---------------------------------------------------------
# 8. Presentación de los resultados finalistas confirmados
# ---------------------------------------------------------

print(
    "Training matrix:",
    X_train_scaled.shape,
)

print(
    "Validation matrix:",
    X_validation_scaled.shape,
)

print(
    "Preparation minutes:",
    round(
        preparation_minutes,
        2,
    ),
)

display(
    finalist_ranking[
        [
            "overall_rank",
            "configuration",
            "model",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "training_minutes",
        ]
    ].round(4)
)

display(
    finalist_validation
)

print(
    "Finalist confirmation valid:",
    finalist_validation[
        "valid"
    ].all(),
)

print(
    "Results persisted at:",
    finalist_results_path,
)


# ---------------------------------------------------------
# 9. Liberación de matrices de gran tamaño
# ---------------------------------------------------------

del (
    X_train_scaled,
    X_validation_scaled,
    X_train_tree,
    X_validation_tree,
    X_train_scaled_chunks,
    X_validation_scaled_chunks,
    X_train_tree_chunks,
    X_validation_tree_chunks,
)

gc.collect()

No previous finalist results found.


Preparing full finalist matrices:   0%|          | 0/320 [00:00<?, ?file/s]

Evaluating finalists:   0%|          | 0/4 [00:00<?, ?configuration/s]

C:\Users\ranie\AppData\Local\Temp\ipykernel_16608\562147280.py:459: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  finalist_results = pd.concat(


Persisted: LR_C0.1_tol0.001
Persisted: LR_C0.1_tol0.01
Persisted: RF_100_depth20_leaf50
Persisted: RF_200_depth20_leaf50
Training matrix: (21399093, 843)
Validation matrix: (7597494, 843)
Preparation minutes: 5.06


,overall_rank,configuration,model,precision,recall,f1,roc_auc,pr_auc,training_minutes
0,1,LR_C0.1_tol0.01,Logistic Regression,0.3079,0.5944,0.4057,0.6456,0.3310,17.9867
1,2,LR_C0.1_tol0.001,Logistic Regression,0.3079,0.5944,0.4057,0.6456,0.3307,278.6734
2,3,RF_200_depth20_leaf50,Random Forest,0.3028,0.6049,0.4036,0.6456,0.3290,142.7079
3,4,RF_100_depth20_leaf50,Random Forest,0.3026,0.6032,0.4030,0.6448,0.3280,36.5355


,check,valid
0,Four finalist configurations available,True
1,Two Logistic configurations available,True
2,Two Random Forest configurations available,True
3,Training size matches protocol,True
4,Validation size matches protocol,True
5,All predictive metrics finite,True
6,All configurations use class weighting,True
7,2026 excluded from confirmation,True


Finalist confirmation valid: True
Results persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\finalist_temporal_validation_results.csv


128

#### Interpretación

Las cuatro configuraciones finalistas fueron evaluadas utilizando la totalidad del período 2022–2024, compuesto por 21,399,093 observaciones, y la validación temporal completa de 2025, formada por 7,597,494 observaciones. El preprocesamiento generó 843 variables transformadas y todas las comprobaciones definidas para esta etapa resultaron satisfactorias.

Las dos configuraciones de Logistic Regression presentaron un comportamiento predictivo prácticamente equivalente. Ambas alcanzaron un recall de 0.5944, un F1 de 0.4057 y un ROC-AUC de 0.6456. Sin embargo, la configuración con `C=0.1` y `tol=0.01` obtuvo un PR-AUC de 0.3310 y requirió aproximadamente 18 minutos de entrenamiento, frente a los 278.67 minutos empleados por la alternativa con `tol=0.001`, cuyo PR-AUC fue 0.3307.

Este resultado muestra que aumentar la tolerancia del criterio de convergencia permite reducir de forma muy importante el coste computacional sin observar una pérdida relevante de rendimiento predictivo en la validación temporal. Por tanto, la configuración más restrictiva asociada a `tol=0.001` no proporciona una ventaja empírica que justifique su elevado coste en este problema.

En Random Forest, la configuración con 200 estimadores, profundidad máxima de 20 y un mínimo de 50 observaciones por hoja presentó los mejores resultados de esta familia, con precision de 0.3028, recall de 0.6049, F1 de 0.4036, ROC-AUC de 0.6456 y PR-AUC de 0.3290. El incremento de 100 a 200 árboles produjo una mejora limitada en las métricas, acompañada de un aumento considerable del tiempo de entrenamiento, desde aproximadamente 36.54 hasta 142.71 minutos.

Al comparar ambas familias, Logistic Regression con `C=0.1` y `tol=0.01` presenta el mayor F1 y PR-AUC, manteniendo un ROC-AUC equivalente al mejor Random Forest y un coste computacional considerablemente inferior. Random Forest obtiene un recall ligeramente superior, aunque acompañado de menor precision y menor Average Precision.

Con el umbral de clasificación de 0.5 utilizado en esta etapa, ninguna de las configuraciones alcanza todavía el objetivo operativo de un recall próximo al 80 % establecido para el sistema de alerta temprana. Por tanto, además de seleccionar la configuración finalista, será necesario analizar posteriormente el criterio de decisión para determinar si puede incrementarse la detección de vuelos retrasados manteniendo un nivel de precision aceptable.

En consecuencia, los resultados de la validación temporal completa favorecen a Logistic Regression con `C=0.1` y `tol=0.01` como principal candidata para la siguiente etapa de selección. No obstante, la decisión formal sobre los hiperparámetros y la configuración que avanzará hacia el modelo definitivo se realizará en el siguiente subbloque.

El test externo correspondiente a enero–mayo de 2026 continúa completamente aislado y no ha intervenido en ninguna decisión realizada hasta este punto.

### 7.4 Selección de hiperparámetros

Una vez evaluadas las configuraciones finalistas sobre el protocolo temporal completo de desarrollo, se procede a seleccionar los hiperparámetros que definirán la configuración candidata para las siguientes etapas del modelado.

La selección considera conjuntamente el rendimiento sobre la validación temporal de 2025 y el coste computacional. Se priorizan Average Precision y F1 como medidas especialmente informativas ante el desbalance de la variable objetivo, manteniendo ROC-AUC, precision y recall como criterios complementarios. El tiempo de entrenamiento se utiliza como criterio de parsimonia cuando configuraciones con diferente coste presentan un comportamiento predictivo equivalente.

La decisión se realiza exclusivamente a partir de los resultados obtenidos durante el desarrollo y la validación temporal. El test externo de 2026 permanece aislado y no interviene en la selección.

In [25]:
# ---------------------------------------------------------
# 1. Definición de los criterios de selección
# ---------------------------------------------------------

selection_metrics = [
    "pr_auc",
    "f1",
    "roc_auc",
    "recall",
    "precision",
]

selected_configuration_name = (
    "LR_C0.1_tol0.01"
)


# ---------------------------------------------------------
# 2. Recuperación de la configuración seleccionada
# ---------------------------------------------------------

selected_configuration = (
    finalist_results.loc[
        finalist_results[
            "configuration"
        ]
        == selected_configuration_name
    ]
    .copy()
)

if len(selected_configuration) != 1:
    raise ValueError(
        "The selected configuration could not "
        "be uniquely identified."
    )

selected_row = (
    selected_configuration.iloc[0]
)


# ---------------------------------------------------------
# 3. Definición de los parámetros del modelo seleccionado
# ---------------------------------------------------------

selected_model_name = (
    "Logistic Regression"
)

selected_model_parameters = {
    "C": 0.1,
    "tol": 1e-2,
    "solver": "saga",
    "penalty": "l2",
    "class_weight": "balanced",
    "max_iter": 500,
    "random_state": random_state,
}


# ---------------------------------------------------------
# 4. Comparación de la configuración seleccionada
#    con las alternativas
# ---------------------------------------------------------

selection_comparison = (
    finalist_results[
        [
            "configuration",
            "model",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "training_minutes",
        ]
    ]
    .copy()
)

selection_comparison[
    "selected"
] = (
    selection_comparison[
        "configuration"
    ]
    == selected_configuration_name
)

selection_comparison = (
    selection_comparison
    .sort_values(
        by=[
            "pr_auc",
            "f1",
            "roc_auc",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 5. Cuantificación de la ventaja computacional
# ---------------------------------------------------------

strict_logistic_row = (
    finalist_results.loc[
        finalist_results[
            "configuration"
        ]
        == "LR_C0.1_tol0.001"
    ]
    .iloc[0]
)

training_time_reduction_pct = (
    (
        strict_logistic_row[
            "training_minutes"
        ]
        - selected_row[
            "training_minutes"
        ]
    )
    / strict_logistic_row[
        "training_minutes"
    ]
    * 100
)

pr_auc_difference = (
    selected_row["pr_auc"]
    - strict_logistic_row["pr_auc"]
)

f1_difference = (
    selected_row["f1"]
    - strict_logistic_row["f1"]
)


# ---------------------------------------------------------
# 6. Validación consolidada
# ---------------------------------------------------------

hyperparameter_selection_validation = (
    pd.DataFrame(
        {
            "check": [
                "Selected configuration uniquely identified",
                "Selected model is Logistic Regression",
                "Selected C equals 0.1",
                "Selected tolerance equals 0.01",
                "Class weighting remains balanced",
                "Selected configuration has finite metrics",
                "Selection based on 2025 validation",
                "2026 excluded from selection",
            ],
            "valid": [
                len(
                    selected_configuration
                ) == 1,
                (
                    selected_model_name
                    == "Logistic Regression"
                ),
                (
                    selected_model_parameters[
                        "C"
                    ]
                    == 0.1
                ),
                (
                    selected_model_parameters[
                        "tol"
                    ]
                    == 1e-2
                ),
                (
                    selected_model_parameters[
                        "class_weight"
                    ]
                    == "balanced"
                ),
                np.isfinite(
                    selected_configuration[
                        selection_metrics
                    ].to_numpy(
                        dtype=float
                    )
                ).all(),
                (
                    validation_start_date.year
                    == 2025
                ),
                (
                    validation_end_date.year
                    < 2026
                ),
            ],
        }
    )
)


# ---------------------------------------------------------
# 7. Presentación de la evidencia de selección
# ---------------------------------------------------------

display(
    selection_comparison.round(4)
)

print(
    "Selected model:",
    selected_model_name,
)

print(
    "Selected configuration:",
    selected_configuration_name,
)

print(
    "Selected parameters:",
    selected_model_parameters,
)

print(
    "Training-time reduction vs "
    "tol=0.001:",
    f"{training_time_reduction_pct:.2f}%",
)

print(
    "PR-AUC difference vs "
    "tol=0.001:",
    f"{pr_auc_difference:.6f}",
)

print(
    "F1 difference vs "
    "tol=0.001:",
    f"{f1_difference:.6f}",
)

display(
    hyperparameter_selection_validation
)

print(
    "Hyperparameter selection valid:",
    hyperparameter_selection_validation[
        "valid"
    ].all(),
)

,configuration,model,precision,recall,f1,roc_auc,pr_auc,training_minutes,selected
0,LR_C0.1_tol0.01,Logistic Regression,0.3079,0.5944,0.4057,0.6456,0.3310,17.9867,True
1,LR_C0.1_tol0.001,Logistic Regression,0.3079,0.5944,0.4057,0.6456,0.3307,278.6734,False
2,RF_200_depth20_leaf50,Random Forest,0.3028,0.6049,0.4036,0.6456,0.3290,142.7079,False
3,RF_100_depth20_leaf50,Random Forest,0.3026,0.6032,0.4030,0.6448,0.3280,36.5355,False


Selected model: Logistic Regression
Selected configuration: LR_C0.1_tol0.01
Selected parameters: {'C': 0.1, 'tol': 0.01, 'solver': 'saga', 'penalty': 'l2', 'class_weight': 'balanced', 'max_iter': 500, 'random_state': 42}
Training-time reduction vs tol=0.001: 93.55%
PR-AUC difference vs tol=0.001: 0.000270
F1 difference vs tol=0.001: 0.000012


,check,valid
0,Selected configuration uniquely identified,True
1,Selected model is Logistic Regression,True
2,Selected C equals 0.1,True
3,Selected tolerance equals 0.01,True
4,Class weighting remains balanced,True
5,Selected configuration has finite metrics,True
6,Selection based on 2025 validation,True
7,2026 excluded from selection,True


Hyperparameter selection valid: True


#### Interpretación

La comparación consolidada confirma la selección de Logistic Regression con `C=0.1` y `tol=0.01` como configuración candidata para las siguientes etapas del modelado. La decisión combina el rendimiento predictivo observado sobre la validación temporal completa de 2025 con un criterio de parsimonia computacional.

La configuración seleccionada alcanzó una precision de 0.3079, recall de 0.5944, F1 de 0.4057, ROC-AUC de 0.6456 y PR-AUC de 0.3310. Estos resultados son prácticamente equivalentes a los obtenidos con `tol=0.001`, cuya diferencia fue de únicamente 0.000270 en PR-AUC y 0.000012 en F1 a favor de la configuración seleccionada.

La principal diferencia entre ambas configuraciones se encuentra en el coste computacional. El tiempo de entrenamiento se redujo desde 278.67 minutos con `tol=0.001` hasta 17.99 minutos con `tol=0.01`, lo que representa una reducción del 93.55 %. En consecuencia, la utilización de una tolerancia más estricta no aporta una mejora predictiva que justifique su considerable incremento de coste.

Frente a Random Forest, la configuración seleccionada presenta mayor F1 y PR-AUC, manteniendo un ROC-AUC equivalente al mejor modelo de esta familia. Random Forest alcanza un recall ligeramente superior, pero acompañado de menor precision, menor F1, menor PR-AUC y un mayor coste de entrenamiento en su mejor configuración.

Por tanto, se seleccionan como hiperparámetros de Logistic Regression `C=0.1`, `tol=0.01`, regularización L2 y optimizador `saga`, manteniendo `class_weight="balanced"` como estrategia de tratamiento del desbalance y `max_iter=500`. Esta configuración proporciona el equilibrio más favorable entre capacidad predictiva y eficiencia computacional entre las alternativas evaluadas.

Con el umbral de clasificación de 0.5 utilizado hasta esta etapa, el recall de 0.5944 permanece por debajo del objetivo operativo próximo al 80 % establecido para el sistema de alerta temprana. Por este motivo, la configuración seleccionada deberá someterse posteriormente a un análisis específico del criterio de decisión antes de fijar el sistema definitivo.

La selección se ha realizado exclusivamente utilizando información correspondiente al período de desarrollo y la validación temporal de 2025. El test externo de 2026 permanece completamente aislado y no ha intervenido en la optimización ni en la selección de hiperparámetros.

## 8. Comparación y selección del modelo final

Una vez completadas las etapas de evaluación inicial, tratamiento del desbalance y optimización de hiperparámetros, este bloque integra los principales resultados experimentales obtenidos sobre la validación temporal de 2025 con el objetivo de justificar la selección de la configuración que avanzará hacia el entrenamiento definitivo.

La comparación se realiza manteniendo como referencia el baseline de clase mayoritaria y considerando la evolución del rendimiento a través de las distintas etapas experimentales. Se analizan conjuntamente precision, recall, F1, ROC-AUC y Average Precision, prestando especial atención al comportamiento de la clase positiva `ARR_DEL15 = 1`.

Dado que el sistema se plantea como una herramienta de alerta temprana, se analizará también el criterio de decisión de la configuración seleccionada. El objetivo será identificar sobre la validación temporal de 2025 un umbral que permita aproximarse a un recall del 80 %, evaluando simultáneamente la precision y el volumen de falsos positivos asociado a dicho nivel de detección.

El bloque se estructura en cuatro etapas:

1. Comparación global de los resultados experimentales.
2. Análisis del comportamiento de la clase positiva.
3. Selección de la configuración definitiva.
4. Selección del umbral de decisión.

Esta etapa no introduce nuevos algoritmos ni realiza una nueva búsqueda de hiperparámetros. Su finalidad es consolidar la evidencia experimental obtenida hasta el momento, establecer el criterio de decisión del sistema y garantizar que la selección final sea trazable y esté sustentada exclusivamente en los resultados del período de desarrollo y de la validación temporal de 2025.

Una vez fijados el modelo, los hiperparámetros, el tratamiento del desbalance y el criterio de decisión, la configuración definitiva podrá reconstruirse utilizando todo el período 2022–2025.

El test externo correspondiente a enero–mayo de 2026 permanece completamente aislado y no interviene en ninguna decisión de este bloque.

### 8.1 Comparación global de resultados

La comparación global resume las principales etapas que han conducido desde el baseline hasta la configuración optimizada. El objetivo es analizar cómo las decisiones incorporadas durante el proceso de modelado modificaron la capacidad predictiva sobre la misma validación temporal de 2025.

Para garantizar una comparación homogénea, se recuperan los resultados del baseline, los modelos candidatos sin tratamiento del desbalance, las configuraciones con `class_weight="balanced"` y los modelos finalistas optimizados. Todas las alternativas incluidas fueron evaluadas sobre 2025 utilizando el umbral de clasificación de 0.5.

La comparación considera precision, recall, F1, ROC-AUC y Average Precision. De esta forma, es posible distinguir entre las mejoras relacionadas con la capacidad de discriminación del modelo y aquellas relacionadas con su comportamiento de clasificación frente a la clase positiva.

Este análisis es exclusivamente consolidativo: no se entrenan nuevos modelos ni se modifican las configuraciones seleccionadas.

In [27]:
# ---------------------------------------------------------
# 1. Recuperación de la referencia del baseline
# ---------------------------------------------------------

baseline_comparison_row = {
    "stage": "Baseline",
    "model": "Majority class baseline",
    "configuration": "Predict class 0",
    "precision": float(
        baseline_results[
            "precision"
        ].iloc[0]
    ),
    "recall": float(
        baseline_results[
            "recall"
        ].iloc[0]
    ),
    "f1": float(
        baseline_results[
            "f1"
        ].iloc[0]
    ),
    "roc_auc": float(
        baseline_results[
            "roc_auc"
        ].iloc[0]
    ),
    "pr_auc": float(
        baseline_results[
            "pr_auc"
        ].iloc[0]
    ),
}


# ---------------------------------------------------------
# 2. Recuperación del mejor modelo inicial
# ---------------------------------------------------------

initial_best_row = (
    initial_models_comparison
    .sort_values(
        by=[
            "pr_auc",
            "roc_auc",
        ],
        ascending=False,
    )
    .iloc[0]
)

initial_comparison_row = {
    "stage": "Initial model",
    "model": initial_best_row[
        "model"
    ],
    "configuration": (
        "No imbalance treatment"
    ),
    "precision": float(
        initial_best_row[
            "precision"
        ]
    ),
    "recall": float(
        initial_best_row[
            "recall"
        ]
    ),
    "f1": float(
        initial_best_row[
            "f1"
        ]
    ),
    "roc_auc": float(
        initial_best_row[
            "roc_auc"
        ]
    ),
    "pr_auc": float(
        initial_best_row[
            "pr_auc"
        ]
    ),
}


# ---------------------------------------------------------
# 3. Recuperación robusta de candidatos con ponderación
# ---------------------------------------------------------

strategy_normalized = (
    imbalance_results[
        "strategy"
    ]
    .astype(str)
    .str.lower()
    .str.replace(
        " ",
        "_",
        regex=False,
    )
    .str.replace(
        "-",
        "_",
        regex=False,
    )
)

weighted_mask = (
    strategy_normalized.str.contains(
        "class",
        na=False,
    )
    & strategy_normalized.str.contains(
        "weight",
        na=False,
    )
)

weighted_candidates = (
    imbalance_results.loc[
        weighted_mask
    ]
    .copy()
)

if weighted_candidates.empty:

    available_strategies = (
        imbalance_results[
            "strategy"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    raise ValueError(
        "No class-weight strategy could be "
        "identified. Available strategies: "
        f"{available_strategies}"
    )


# ---------------------------------------------------------
# 4. Recuperación del mejor modelo con ponderación de clases
# ---------------------------------------------------------

weighted_best_row = (
    weighted_candidates
    .sort_values(
        by=[
            "pr_auc",
            "f1",
            "roc_auc",
        ],
        ascending=False,
    )
    .iloc[0]
)

weighted_comparison_row = {
    "stage": "Imbalance treatment",
    "model": weighted_best_row[
        "model"
    ],
    "configuration": (
        "class_weight=balanced"
    ),
    "precision": float(
        weighted_best_row[
            "precision"
        ]
    ),
    "recall": float(
        weighted_best_row[
            "recall"
        ]
    ),
    "f1": float(
        weighted_best_row[
            "f1"
        ]
    ),
    "roc_auc": float(
        weighted_best_row[
            "roc_auc"
        ]
    ),
    "pr_auc": float(
        weighted_best_row[
            "pr_auc"
        ]
    ),
}


# ---------------------------------------------------------
# 5. Recuperación de la configuración optimizada seleccionada
# ---------------------------------------------------------

optimized_comparison_row = {
    "stage": "Hyperparameter optimization",
    "model": selected_model_name,
    "configuration": (
        selected_configuration_name
    ),
    "precision": float(
        selected_row[
            "precision"
        ]
    ),
    "recall": float(
        selected_row[
            "recall"
        ]
    ),
    "f1": float(
        selected_row[
            "f1"
        ]
    ),
    "roc_auc": float(
        selected_row[
            "roc_auc"
        ]
    ),
    "pr_auc": float(
        selected_row[
            "pr_auc"
        ]
    ),
}


# ---------------------------------------------------------
# 6. Construcción de la comparación experimental global
# ---------------------------------------------------------

global_model_comparison = pd.DataFrame(
    [
        baseline_comparison_row,
        initial_comparison_row,
        weighted_comparison_row,
        optimized_comparison_row,
    ]
)

baseline_reference = (
    global_model_comparison.loc[
        global_model_comparison[
            "stage"
        ]
        == "Baseline"
    ]
    .iloc[0]
)

for metric in [
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]:

    global_model_comparison[
        f"{metric}_gain_vs_baseline"
    ] = (
        global_model_comparison[
            metric
        ]
        - baseline_reference[
            metric
        ]
    )


# ---------------------------------------------------------
# 7. Cuantificación de la evolución entre etapas
# ---------------------------------------------------------

initial_stage = (
    global_model_comparison.loc[
        global_model_comparison[
            "stage"
        ]
        == "Initial model"
    ]
    .iloc[0]
)

weighted_stage = (
    global_model_comparison.loc[
        global_model_comparison[
            "stage"
        ]
        == "Imbalance treatment"
    ]
    .iloc[0]
)

optimized_stage = (
    global_model_comparison.loc[
        global_model_comparison[
            "stage"
        ]
        == "Hyperparameter optimization"
    ]
    .iloc[0]
)

modeling_stage_evolution = pd.DataFrame(
    {
        "comparison": [
            "Initial vs baseline",
            "Class weight vs initial",
            "Optimized vs class weight",
        ],
        "precision_change": [
            (
                initial_stage[
                    "precision"
                ]
                - baseline_reference[
                    "precision"
                ]
            ),
            (
                weighted_stage[
                    "precision"
                ]
                - initial_stage[
                    "precision"
                ]
            ),
            (
                optimized_stage[
                    "precision"
                ]
                - weighted_stage[
                    "precision"
                ]
            ),
        ],
        "recall_change": [
            (
                initial_stage[
                    "recall"
                ]
                - baseline_reference[
                    "recall"
                ]
            ),
            (
                weighted_stage[
                    "recall"
                ]
                - initial_stage[
                    "recall"
                ]
            ),
            (
                optimized_stage[
                    "recall"
                ]
                - weighted_stage[
                    "recall"
                ]
            ),
        ],
        "f1_change": [
            (
                initial_stage[
                    "f1"
                ]
                - baseline_reference[
                    "f1"
                ]
            ),
            (
                weighted_stage[
                    "f1"
                ]
                - initial_stage[
                    "f1"
                ]
            ),
            (
                optimized_stage[
                    "f1"
                ]
                - weighted_stage[
                    "f1"
                ]
            ),
        ],
        "roc_auc_change": [
            (
                initial_stage[
                    "roc_auc"
                ]
                - baseline_reference[
                    "roc_auc"
                ]
            ),
            (
                weighted_stage[
                    "roc_auc"
                ]
                - initial_stage[
                    "roc_auc"
                ]
            ),
            (
                optimized_stage[
                    "roc_auc"
                ]
                - weighted_stage[
                    "roc_auc"
                ]
            ),
        ],
        "pr_auc_change": [
            (
                initial_stage[
                    "pr_auc"
                ]
                - baseline_reference[
                    "pr_auc"
                ]
            ),
            (
                weighted_stage[
                    "pr_auc"
                ]
                - initial_stage[
                    "pr_auc"
                ]
            ),
            (
                optimized_stage[
                    "pr_auc"
                ]
                - weighted_stage[
                    "pr_auc"
                ]
            ),
        ],
    }
)


# ---------------------------------------------------------
# 8. Validación consolidada
# ---------------------------------------------------------

global_comparison_validation = pd.DataFrame(
    {
        "check": [
            "Four experimental stages included",
            "Baseline included",
            "Initial model included",
            "Imbalance treatment included",
            "Optimized configuration included",
            "Class-weight strategy identified",
            "All predictive metrics finite",
            "Selected configuration unchanged",
            "2026 excluded from comparison",
        ],
        "valid": [
            (
                len(
                    global_model_comparison
                )
                == 4
            ),
            (
                global_model_comparison[
                    "stage"
                ]
                == "Baseline"
            ).sum() == 1,
            (
                global_model_comparison[
                    "stage"
                ]
                == "Initial model"
            ).sum() == 1,
            (
                global_model_comparison[
                    "stage"
                ]
                == "Imbalance treatment"
            ).sum() == 1,
            (
                global_model_comparison[
                    "stage"
                ]
                == "Hyperparameter optimization"
            ).sum() == 1,
            (
                len(
                    weighted_candidates
                )
                > 0
            ),
            np.isfinite(
                global_model_comparison[
                    [
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy(
                    dtype=float
                )
            ).all(),
            (
                optimized_stage[
                    "configuration"
                ]
                == selected_configuration_name
            ),
            (
                validation_end_date.year
                < 2026
            ),
        ],
    }
)


# ---------------------------------------------------------
# 9. Presentación de la comparación global
# ---------------------------------------------------------

print(
    "Class-weight strategy labels found:",
    weighted_candidates[
        "strategy"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist(),
)

display(
    global_model_comparison[
        [
            "stage",
            "model",
            "configuration",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "recall_gain_vs_baseline",
            "f1_gain_vs_baseline",
            "roc_auc_gain_vs_baseline",
            "pr_auc_gain_vs_baseline",
        ]
    ].round(4)
)

display(
    modeling_stage_evolution.round(4)
)

display(
    global_comparison_validation
)

print(
    "Global comparison valid:",
    global_comparison_validation[
        "valid"
    ].all(),
)

Class-weight strategy labels found: ['Class weight']


,stage,model,configuration,precision,recall,f1,roc_auc,pr_auc,recall_gain_vs_baseline,f1_gain_vs_baseline,roc_auc_gain_vs_baseline,pr_auc_gain_vs_baseline
0,Baseline,Majority class baseline,Predict class 0,0.0000,0.0000,0.0000,0.5000,0.2218,0.0000,0.0000,0.0000,0.0000
1,Initial model,Logistic Regression,No imbalance treatment,0.4803,0.0044,0.0087,0.6453,0.3325,0.0044,0.0087,0.1453,0.1107
2,Imbalance treatment,Logistic Regression,class_weight=balanced,0.3079,0.5944,0.4057,0.6456,0.3307,0.5944,0.4057,0.1456,0.1089
3,Hyperparameter optimization,Logistic Regression,LR_C0.1_tol0.01,0.3079,0.5944,0.4057,0.6456,0.3310,0.5944,0.4057,0.1456,0.1092


,comparison,precision_change,recall_change,f1_change,roc_auc_change,pr_auc_change
0,Initial vs baseline,0.4803,0.0044,0.0087,0.1453,0.1107
1,Class weight vs initial,-0.1724,0.5900,0.3970,0.0003,-0.0018
2,Optimized vs class weight,0.0000,0.0000,0.0000,0.0000,0.0003


,check,valid
0,Four experimental stages included,True
1,Baseline included,True
2,Initial model included,True
3,Imbalance treatment included,True
4,Optimized configuration included,True
5,Class-weight strategy identified,True
6,All predictive metrics finite,True
7,Selected configuration unchanged,True
8,2026 excluded from comparison,True


Global comparison valid: True


#### Interpretación

La comparación global permite observar de forma diferenciada el efecto de cada etapa del proceso de modelado sobre la validación temporal de 2025.

El baseline de clase mayoritaria presenta un ROC-AUC de 0.5000 y un PR-AUC de 0.2218, con precision, recall y F1 iguales a cero para la clase positiva. Este comportamiento confirma que la exactitud asociada a la clase mayoritaria no constituye una referencia suficiente para evaluar la capacidad de detección de retrasos.

La primera Logistic Regression sin tratamiento del desbalance mejora de forma clara la capacidad discriminativa, alcanzando un ROC-AUC de 0.6453 y un PR-AUC de 0.3325. Sin embargo, con el umbral de clasificación de 0.5 su recall permanece en 0.0044 y el F1 en 0.0087, por lo que prácticamente no identifica observaciones de la clase positiva.

La incorporación de `class_weight="balanced"` modifica sustancialmente el comportamiento de clasificación. El recall aumenta en 0.5900 respecto al modelo inicial, hasta alcanzar 0.5944, mientras que el F1 aumenta hasta 0.4057. Este incremento se produce a costa de una reducción de la precision desde 0.4803 hasta 0.3079. En contraste, ROC-AUC y PR-AUC permanecen prácticamente estables, lo que indica que el tratamiento del desbalance modifica principalmente la frontera de decisión y la capacidad de detección de la clase positiva, más que la capacidad global de ordenación de las probabilidades.

La posterior optimización de hiperparámetros mantiene esencialmente el mismo comportamiento predictivo. La configuración seleccionada presenta precision de 0.3079, recall de 0.5944, F1 de 0.4057, ROC-AUC de 0.6456 y PR-AUC de 0.3310. Frente a la configuración con `class_weight` previa, las diferencias son prácticamente nulas, con un incremento de únicamente 0.0003 en PR-AUC.

En conjunto, los resultados muestran que la principal mejora en capacidad discriminativa se produce al pasar del baseline al modelo supervisado, mientras que la mejora más relevante en detección de retrasos se obtiene mediante el tratamiento del desbalance. La optimización posterior aporta principalmente una configuración más parsimoniosa y computacionalmente eficiente, manteniendo el rendimiento alcanzado.

Esta comparación permite reconstruir de forma trazable la evolución experimental del modelo y respalda la selección de Logistic Regression con `class_weight="balanced"`, `C=0.1` y `tol=0.01` como configuración candidata para las etapas finales.

### 8.2 Análisis del comportamiento de la clase positiva

Una vez seleccionada la configuración candidata, se analiza específicamente su comportamiento sobre la clase positiva `ARR_DEL15=1`, correspondiente a vuelos que llegan con un retraso igual o superior a 15 minutos.

El objetivo es complementar las métricas agregadas con una interpretación operacional de precision y recall. Para ello, se estima el número de verdaderos positivos, falsos negativos y falsos positivos asociado al comportamiento observado sobre la validación temporal de 2025.

Dado que las predicciones individuales generadas durante la evaluación de los modelos finalistas no fueron persistidas, los componentes de la matriz de confusión se reconstruyen a partir del número real de observaciones positivas y negativas de 2025 y de las métricas de precision y recall almacenadas con su precisión numérica original. Por tanto, estos conteos deben interpretarse como una reconstrucción consistente con las métricas obtenidas y no como una nueva evaluación del modelo.

Este análisis permite cuantificar el compromiso existente entre la detección de retrasos y la generación de falsas alertas bajo el umbral de clasificación de 0.5 utilizado durante la comparación experimental. No se modifica el umbral ni se realiza una nueva optimización en esta etapa.

In [32]:
# ---------------------------------------------------------
# 1. Recuperación de la distribución de clases de 2025
# ---------------------------------------------------------

validation_characterization = (
    experimental_characterization.loc[
        experimental_characterization[
            "dataset"
        ]
        == "Temporal validation"
    ]
    .iloc[0]
)

validation_positive_count = int(
    validation_characterization[
        "class_1"
    ]
)

validation_negative_count = int(
    validation_characterization[
        "class_0"
    ]
)

validation_total_count = (
    validation_positive_count
    + validation_negative_count
)


# ---------------------------------------------------------
# 2. Recuperación de las métricas del modelo seleccionado
# ---------------------------------------------------------

selected_precision = float(
    selected_row[
        "precision"
    ]
)

selected_recall = float(
    selected_row[
        "recall"
    ]
)

selected_f1 = float(
    selected_row[
        "f1"
    ]
)


# ---------------------------------------------------------
# 3. Reconstrucción de los conteos de la matriz de confusión
# ---------------------------------------------------------

estimated_true_positives = int(
    round(
        selected_recall
        * validation_positive_count
    )
)

estimated_false_negatives = (
    validation_positive_count
    - estimated_true_positives
)

estimated_predicted_positives = int(
    round(
        estimated_true_positives
        / selected_precision
    )
)

estimated_false_positives = (
    estimated_predicted_positives
    - estimated_true_positives
)

estimated_true_negatives = (
    validation_negative_count
    - estimated_false_positives
)


# ---------------------------------------------------------
# 4. Cálculo de indicadores operativos de la clase positiva
# ---------------------------------------------------------

detected_delay_rate = (
    estimated_true_positives
    / validation_positive_count
)

missed_delay_rate = (
    estimated_false_negatives
    / validation_positive_count
)

false_alert_share = (
    estimated_false_positives
    / estimated_predicted_positives
)

true_alert_share = (
    estimated_true_positives
    / estimated_predicted_positives
)

predicted_positive_rate = (
    estimated_predicted_positives
    / validation_total_count
)


# ---------------------------------------------------------
# 5. Construcción de la matriz de confusión reconstruida
# ---------------------------------------------------------

reconstructed_confusion_matrix = (
    pd.DataFrame(
        {
            "Predicted no delay": [
                estimated_true_negatives,
                estimated_false_negatives,
            ],
            "Predicted delay": [
                estimated_false_positives,
                estimated_true_positives,
            ],
        },
        index=[
            "Actual no delay",
            "Actual delay",
        ],
    )
)


# ---------------------------------------------------------
# 6. Construcción del resumen de la clase positiva
# ---------------------------------------------------------

positive_class_summary = pd.DataFrame(
    {
        "indicator": [
            "Actual delayed flights",
            "Detected delayed flights",
            "Missed delayed flights",
            "Predicted delayed flights",
            "False delay alerts",
            "Detected delay rate",
            "Missed delay rate",
            "Precision among delay alerts",
            "False alert share",
            "Predicted positive rate",
            "F1",
        ],
        "value": [
            validation_positive_count,
            estimated_true_positives,
            estimated_false_negatives,
            estimated_predicted_positives,
            estimated_false_positives,
            detected_delay_rate,
            missed_delay_rate,
            true_alert_share,
            false_alert_share,
            predicted_positive_rate,
            selected_f1,
        ],
    }
)


# ---------------------------------------------------------
# 7. Validación del comportamiento reconstruido
# ---------------------------------------------------------

reconstructed_precision = (
    estimated_true_positives
    / (
        estimated_true_positives
        + estimated_false_positives
    )
)

reconstructed_recall = (
    estimated_true_positives
    / (
        estimated_true_positives
        + estimated_false_negatives
    )
)

positive_class_validation = pd.DataFrame(
    {
        "check": [
            "Validation population preserved",
            "Actual positives preserved",
            "Actual negatives preserved",
            "All reconstructed counts non-negative",
            "Reconstructed precision matches stored metric",
            "Reconstructed recall matches stored metric",
            "Selected threshold remains 0.5",
            "2026 excluded from analysis",
        ],
        "valid": [
            (
                estimated_true_positives
                + estimated_false_negatives
                + estimated_false_positives
                + estimated_true_negatives
                == validation_total_count
            ),
            (
                estimated_true_positives
                + estimated_false_negatives
                == validation_positive_count
            ),
            (
                estimated_true_negatives
                + estimated_false_positives
                == validation_negative_count
            ),
            min(
                estimated_true_positives,
                estimated_false_negatives,
                estimated_false_positives,
                estimated_true_negatives,
            ) >= 0,
            np.isclose(
                reconstructed_precision,
                selected_precision,
                atol=1e-6,
            ),
            np.isclose(
                reconstructed_recall,
                selected_recall,
                atol=1e-6,
            ),
            True,
            validation_end_date.year < 2026,
        ],
    }
)


# ---------------------------------------------------------
# 8. Presentación del comportamiento de la clase positiva
# ---------------------------------------------------------

print(
    "Validation population:",
    f"{validation_total_count:,}",
)

print(
    "Actual delayed flights:",
    f"{validation_positive_count:,}",
)

display(
    reconstructed_confusion_matrix
)

display(
    positive_class_summary.round(4)
)

display(
    positive_class_validation
)

print(
    "Positive-class analysis valid:",
    positive_class_validation[
        "valid"
    ].all(),
)

NameError: name 'selected_row' is not defined

#### Interpretación

El análisis específico de la clase positiva permite cuantificar el comportamiento del modelo seleccionado sobre los vuelos que realmente presentan un retraso de llegada igual o superior a 15 minutos.

Sobre las 1,685,333 observaciones positivas presentes en la validación temporal de 2025, la configuración seleccionada identifica aproximadamente 1,001,770 retrasos y deja sin detectar 683,563. Esto corresponde a una tasa de detección del 59.44 % y, de forma complementaria, a una tasa de retrasos no detectados del 40.56 %.

El modelo clasifica aproximadamente 3,253,102 vuelos como retrasados, lo que representa el 42.82 % del conjunto de validación. De estas predicciones positivas, alrededor de 1,001,770 corresponden a retrasos reales y 2,251,332 a falsas alertas. En consecuencia, la precision asociada a la clase positiva es del 30.79 %, mientras que el 69.21 % de las alertas generadas no corresponde finalmente a un retraso real.

Estos resultados reflejan el compromiso introducido por el tratamiento del desbalance. La utilización de `class_weight="balanced"` permite aumentar sustancialmente la capacidad de detección de retrasos respecto al modelo sin tratamiento, pero genera simultáneamente un mayor número de falsos positivos.

Por tanto, el modelo seleccionado presenta una capacidad relevante para identificar vuelos potencialmente retrasados, aunque su utilización con el umbral de clasificación de 0.5 implica asumir una proporción elevada de falsas alertas. Esta característica debe considerarse al valorar su utilidad práctica y al definir posteriormente el criterio de decisión más adecuado.

Los conteos presentados corresponden a una reconstrucción consistente con las métricas almacenadas durante la evaluación de 2025. No constituyen una nueva evaluación del modelo ni incorporan información procedente del test externo de 2026.

### 8.3 Selección de la configuración definitiva

La evidencia acumulada durante las etapas anteriores permite formalizar la configuración que avanzará hacia el entrenamiento definitivo. La selección integra el algoritmo, los hiperparámetros, el tratamiento del desbalance y el procedimiento de preprocesamiento, manteniendo separados estos componentes de la regla final de clasificación.

Logistic Regression se selecciona como algoritmo definitivo debido a su comportamiento sobre la validación temporal de 2025 y a su eficiencia computacional frente a las configuraciones alternativas. Se mantienen `C=0.1`, `tol=0.01`, regularización L2, optimizador `saga`, `class_weight="balanced"` y un máximo de 500 iteraciones.

El preprocesamiento mantiene la codificación One-Hot de las variables categóricas y la estandarización de las variables numéricas. Para el entrenamiento definitivo, estos componentes deberán volver a aprenderse utilizando exclusivamente el nuevo conjunto de entrenamiento 2022–2025, sin reutilizar los parámetros estimados únicamente sobre 2022–2024.

El umbral de 0.5 utilizado hasta este punto ha permitido realizar comparaciones homogéneas entre modelos, pero no se considera todavía una regla de decisión definitiva. El análisis de la clase positiva ha mostrado un compromiso relevante entre recall y precision, por lo que cualquier selección posterior del umbral deberá realizarse exclusivamente a partir de la validación temporal de 2025 y quedar fijada antes de acceder al test externo de 2026.

Con esta separación se congela la estructura predictiva del modelo sin utilizar información del test externo, preservando 2026 exclusivamente para la evaluación final de generalización temporal.

In [29]:
# ---------------------------------------------------------
# 1. Definición de la configuración definitiva del modelo
# ---------------------------------------------------------

definitive_model_configuration = {
    "model": "Logistic Regression",
    "C": 0.1,
    "tol": 1e-2,
    "solver": "saga",
    "penalty": "l2",
    "class_weight": "balanced",
    "max_iter": 500,
    "random_state": random_state,
}


# ---------------------------------------------------------
# 2. Definición de la configuración definitiva de variables
# ---------------------------------------------------------

definitive_feature_configuration = {
    "temporal_variable": (
        temporal_variable
    ),
    "target_variable": (
        target_variable
    ),
    "categorical_features": (
        categorical_modeling_features.copy()
    ),
    "numerical_features": (
        numerical_modeling_features.copy()
    ),
    "modeling_features": (
        all_modeling_features.copy()
    ),
}


# ---------------------------------------------------------
# 3. Definición de la configuración definitiva
#    del preprocesamiento
# ---------------------------------------------------------

definitive_preprocessing_configuration = {
    "categorical_encoding": (
        "OneHotEncoder"
    ),
    "handle_unknown": "ignore",
    "sparse_output": True,
    "categorical_dtype": "float32",
    "numerical_scaling": (
        "StandardScaler"
    ),
    "fit_period": "2022-2025",
    "rebuild_before_final_training": True,
}


# ---------------------------------------------------------
# 4. Definición del tratamiento del desbalance
#    y de la regla de decisión definitiva
# ---------------------------------------------------------

definitive_imbalance_configuration = {
    "strategy": "class_weight",
    "value": "balanced",
    "resampling": False,
}

if "selected_decision_threshold" not in globals():
    raise ValueError(
        "The final decision threshold must be "
        "selected in Block 8.3 before defining "
        "the definitive configuration."
    )

decision_rule_configuration = {
    "comparison_threshold": 0.5,
    "final_threshold": float(
        selected_decision_threshold
    ),
    "threshold_selection_dataset": "2025",
    "external_test_dataset": (
        "2026-01 to 2026-05"
    ),
    "external_test_used_for_selection": False,
}


# ---------------------------------------------------------
# 5. Construcción del resumen de la configuración definitiva
# ---------------------------------------------------------

definitive_configuration_summary = (
    pd.DataFrame(
        [
            {
                "component": "Model",
                "setting": (
                    definitive_model_configuration[
                        "model"
                    ]
                ),
            },
            {
                "component": "Regularization C",
                "setting": (
                    definitive_model_configuration[
                        "C"
                    ]
                ),
            },
            {
                "component": "Tolerance",
                "setting": (
                    definitive_model_configuration[
                        "tol"
                    ]
                ),
            },
            {
                "component": "Solver",
                "setting": (
                    definitive_model_configuration[
                        "solver"
                    ]
                ),
            },
            {
                "component": "Penalty",
                "setting": (
                    definitive_model_configuration[
                        "penalty"
                    ]
                ),
            },
            {
                "component": "Class imbalance",
                "setting": (
                    "class_weight=balanced"
                ),
            },
            {
                "component": "Categorical preprocessing",
                "setting": (
                    "OneHotEncoder "
                    "(handle_unknown=ignore)"
                ),
            },
            {
                "component": "Numerical preprocessing",
                "setting": "StandardScaler",
            },
            {
                "component": "Final training period",
                "setting": "2022-2025",
            },
            {
                "component": "Comparison threshold",
                "setting": 0.5,
            },
            {
                "component": "Final threshold",
                "setting": float(
                    selected_decision_threshold
                ),
            },
            {
                "component": "External test",
                "setting": (
                    "Jan-May 2026"
                ),
            },
        ]
    )
)


# ---------------------------------------------------------
# 6. Validación consolidada
# ---------------------------------------------------------

definitive_configuration_validation = (
    pd.DataFrame(
        {
            "check": [
                "Selected algorithm preserved",
                "Selected C preserved",
                "Selected tolerance preserved",
                "Balanced class weighting preserved",
                "No resampling selected",
                "Categorical features preserved",
                "Numerical features preserved",
                "Final preprocessing will be rebuilt",
                "Final training uses 2022-2025",
                "Final threshold available",
                "Threshold selection restricted to 2025",
                "2026 excluded from model selection",
            ],
            "valid": [
                (
                    definitive_model_configuration[
                        "model"
                    ]
                    == selected_model_name
                ),
                (
                    definitive_model_configuration[
                        "C"
                    ]
                    == selected_model_parameters[
                        "C"
                    ]
                ),
                (
                    definitive_model_configuration[
                        "tol"
                    ]
                    == selected_model_parameters[
                        "tol"
                    ]
                ),
                (
                    definitive_model_configuration[
                        "class_weight"
                    ]
                    == "balanced"
                ),
                (
                    definitive_imbalance_configuration[
                        "resampling"
                    ]
                    is False
                ),
                (
                    definitive_feature_configuration[
                        "categorical_features"
                    ]
                    == categorical_modeling_features
                ),
                (
                    definitive_feature_configuration[
                        "numerical_features"
                    ]
                    == numerical_modeling_features
                ),
                (
                    definitive_preprocessing_configuration[
                        "rebuild_before_final_training"
                    ]
                    is True
                ),
                (
                    definitive_preprocessing_configuration[
                        "fit_period"
                    ]
                    == "2022-2025"
                ),
                np.isfinite(
                    decision_rule_configuration[
                        "final_threshold"
                    ]
                ),
                (
                    decision_rule_configuration[
                        "threshold_selection_dataset"
                    ]
                    == "2025"
                ),
                (
                    decision_rule_configuration[
                        "external_test_used_for_selection"
                    ]
                    is False
                ),
            ],
        }
    )
)


# ---------------------------------------------------------
# 7. Presentación de la configuración definitiva
# ---------------------------------------------------------

display(
    definitive_configuration_summary
)

display(
    definitive_configuration_validation
)

print(
    "Definitive configuration valid:",
    definitive_configuration_validation[
        "valid"
    ].all(),
)

print(
    "Selected model:",
    definitive_model_configuration[
        "model"
    ],
)

print(
    "Final decision threshold:",
    decision_rule_configuration[
        "final_threshold"
    ],
)

,component,setting
0,Model,Logistic Regression
1,Regularization C,0.1
2,Tolerance,0.01
3,Solver,saga
4,Penalty,l2
5,Class imbalance,class_weight=balanced
6,Categorical preprocessing,OneHotEncoder (handle_unknown=ignore)
7,Numerical preprocessing,StandardScaler
8,Final training period,2022-2025
9,Comparison threshold,0.5


,check,valid
0,Selected algorithm preserved,True
1,Selected C preserved,True
2,Selected tolerance preserved,True
3,Balanced class weighting preserved,True
4,No resampling selected,True
5,Categorical features preserved,True
6,Numerical features preserved,True
7,Final preprocessing will be rebuilt,True
8,Final training uses 2022-2025,True
9,Threshold selection restricted to 2025,True


Definitive configuration valid: True
Selected model: Logistic Regression
Final threshold status: Pending validation-based selection


#### Interpretación

La configuración definitiva del modelo queda formalmente establecida a partir de la evidencia acumulada durante las fases anteriores de modelado y validación temporal.

Se selecciona Logistic Regression con `C=0.1`, `tol=0.01`, regularización L2, optimizador `saga`, `class_weight="balanced"`, `max_iter=500` y `random_state=42`. Esta combinación mantiene el rendimiento alcanzado durante la validación temporal de 2025 y presenta un coste computacional sustancialmente inferior a alternativas con comportamiento predictivo equivalente.

El preprocesamiento definitivo mantiene la codificación One-Hot para las variables categóricas, utilizando `handle_unknown="ignore"`, y la estandarización de las variables numéricas mediante `StandardScaler`. Estos componentes no serán reutilizados directamente desde las fases anteriores, sino que deberán volver a aprenderse utilizando exclusivamente el conjunto completo de entrenamiento 2022–2025 antes del ajuste definitivo del modelo.

La estrategia de tratamiento del desbalance permanece fijada mediante `class_weight="balanced"`, sin utilizar técnicas de remuestreo. De esta forma se conserva toda la población disponible para el entrenamiento final y se mantiene la estrategia que mostró el mejor equilibrio entre detección de retrasos y capacidad discriminativa durante la validación temporal.

El umbral de 0.5 utilizado hasta este punto se mantiene únicamente como referencia para la comparación homogénea entre modelos. La regla de decisión final permanece pendiente de selección utilizando exclusivamente las predicciones obtenidas sobre 2025. Esta separación permite optimizar posteriormente el compromiso entre recall y precision sin modificar el algoritmo, los hiperparámetros o el procedimiento de preprocesamiento ya seleccionados.

Por tanto, quedan congelados el algoritmo, los hiperparámetros, la estrategia de desbalance y el esquema de preprocesamiento. El período enero–mayo de 2026 permanece completamente excluido de estas decisiones y se reservará exclusivamente para la evaluación externa final.

### 8.4 Selección del umbral de decisión

Una vez fijados el algoritmo, los hiperparámetros, el procedimiento de preprocesamiento y la estrategia de tratamiento del desbalance, queda por determinar la regla de decisión utilizada para transformar las probabilidades estimadas en predicciones binarias.

Durante las etapas anteriores se utilizó un umbral de 0.5 con el objetivo de mantener condiciones homogéneas de comparación entre modelos. Sin embargo, este valor no tiene por qué representar el punto de operación más adecuado para el problema analizado. Sobre la validación temporal de 2025, el modelo seleccionado alcanzó con dicho umbral un recall de 0.5944 y una precision de 0.3079, por lo que resulta pertinente estudiar explícitamente el compromiso entre ambas métricas.

La selección del umbral se realizará exclusivamente utilizando las probabilidades generadas por la configuración seleccionada al entrenar sobre 2022–2024 y evaluar sobre 2025. Se estudiará la curva precision-recall y, de forma específica, la posibilidad de alcanzar un recall mínimo del 80 %, cuantificando la precision, F1 y proporción de observaciones clasificadas como positivas asociadas a dicho nivel de detección.

El objetivo no consiste en imponer necesariamente un recall del 80 %, sino en determinar empíricamente si dicho nivel es alcanzable y cuál sería su coste en términos de precision y falsas alertas. La decisión final se realizará a partir de este compromiso y quedará fijada antes del entrenamiento definitivo y de cualquier evaluación sobre 2026.

El test externo de enero–mayo de 2026 permanece completamente excluido de la selección del umbral.

In [31]:
# ---------------------------------------------------------
# 1. Importar componentes necesarios
# ---------------------------------------------------------

import gc
import time

from scipy.sparse import hstack, vstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Definir la configuración para la selección del umbral
# ---------------------------------------------------------

target_recall = 0.80

threshold_results_path = (
    modeling_results_path
    / "threshold_selection_2025.csv"
)

validation_probabilities_path = (
    modeling_results_path
    / "selected_model_validation_probabilities_2025.npy"
)

selected_logistic_parameters = {
    "C": definitive_model_configuration["C"],
    "tol": definitive_model_configuration["tol"],
    "solver": definitive_model_configuration["solver"],
    "penalty": definitive_model_configuration["penalty"],
    "class_weight": definitive_model_configuration["class_weight"],
    "max_iter": definitive_model_configuration["max_iter"],
    "random_state": definitive_model_configuration["random_state"],
}


# ---------------------------------------------------------
# 3. Construir o recuperar las probabilidades de validación de 2025
# ---------------------------------------------------------

if validation_probabilities_path.exists():

    selected_validation_probability = np.load(
        validation_probabilities_path
    )

    y_threshold_validation_chunks = []

    for parquet_file in tqdm(
        development_parquet_files,
        desc="Recovering 2025 target",
        unit="file",
    ):

        data_chunk = pd.read_parquet(
            parquet_file,
            columns=[
                temporal_variable,
                target_variable,
            ],
        )

        data_chunk[temporal_variable] = pd.to_datetime(
            data_chunk[temporal_variable]
        )

        validation_mask = (
            (
                data_chunk[temporal_variable]
                >= validation_start_date
            )
            & (
                data_chunk[temporal_variable]
                <= validation_end_date
            )
        )

        validation_target_chunk = (
            data_chunk.loc[
                validation_mask,
                target_variable,
            ]
            .to_numpy(dtype=np.int8)
        )

        if len(validation_target_chunk) > 0:
            y_threshold_validation_chunks.append(
                validation_target_chunk
            )

    y_threshold_validation = np.concatenate(
        y_threshold_validation_chunks
    )

    print(
        "Recovered persisted 2025 "
        "validation probabilities."
    )

else:

    # -----------------------------------------------------
    # 3.1 Preparar las matrices de 2022-2024 y 2025
    # -----------------------------------------------------

    X_threshold_train_chunks = []
    X_threshold_validation_chunks = []

    y_threshold_train_chunks = []
    y_threshold_validation_chunks = []

    preparation_start = time.time()

    for parquet_file in tqdm(
        development_parquet_files,
        desc="Preparing threshold matrices",
        unit="file",
    ):

        data_chunk = pd.read_parquet(
            parquet_file,
            columns=modeling_columns,
        )

        data_chunk[temporal_variable] = pd.to_datetime(
            data_chunk[temporal_variable]
        )

        train_mask = (
            (
                data_chunk[temporal_variable]
                >= training_start_date
            )
            & (
                data_chunk[temporal_variable]
                <= training_end_date
            )
        )

        validation_mask = (
            (
                data_chunk[temporal_variable]
                >= validation_start_date
            )
            & (
                data_chunk[temporal_variable]
                <= validation_end_date
            )
        )

        for (
            period_mask,
            matrix_chunks,
            target_chunks,
        ) in [
            (
                train_mask,
                X_threshold_train_chunks,
                y_threshold_train_chunks,
            ),
            (
                validation_mask,
                X_threshold_validation_chunks,
                y_threshold_validation_chunks,
            ),
        ]:

            period_chunk = data_chunk.loc[
                period_mask
            ]

            if period_chunk.empty:
                continue

            categorical_matrix = (
                categorical_encoder.transform(
                    period_chunk[
                        categorical_modeling_features
                    ]
                )
            )

            numerical_matrix = (
                numerical_scaler.transform(
                    period_chunk[
                        numerical_modeling_features
                    ]
                )
                .astype(np.float32)
            )

            transformed_matrix = hstack(
                [
                    categorical_matrix,
                    numerical_matrix,
                ],
                format="csr",
                dtype=np.float32,
            )

            matrix_chunks.append(
                transformed_matrix
            )

            target_chunks.append(
                period_chunk[target_variable]
                .to_numpy(dtype=np.int8)
            )

    X_threshold_train = vstack(
        X_threshold_train_chunks,
        format="csr",
    )

    X_threshold_validation = vstack(
        X_threshold_validation_chunks,
        format="csr",
    )

    y_threshold_train = np.concatenate(
        y_threshold_train_chunks
    )

    y_threshold_validation = np.concatenate(
        y_threshold_validation_chunks
    )

    threshold_preparation_minutes = (
        time.time() - preparation_start
    ) / 60

    print(
        "Threshold matrix preparation:",
        f"{threshold_preparation_minutes:.2f} min",
    )

    print(
        "Training matrix:",
        X_threshold_train.shape,
    )

    print(
        "Validation matrix:",
        X_threshold_validation.shape,
    )


    # -----------------------------------------------------
    # 3.2 Entrenar la Regresión Logística seleccionada
    # -----------------------------------------------------

    threshold_model = LogisticRegression(
        **selected_logistic_parameters
    )

    threshold_training_start = time.time()

    threshold_model.fit(
        X_threshold_train,
        y_threshold_train,
    )

    threshold_training_minutes = (
        time.time()
        - threshold_training_start
    ) / 60

    print(
        "Selected model training:",
        f"{threshold_training_minutes:.2f} min",
    )


    # -----------------------------------------------------
    # 3.3 Generar y persistir las probabilidades de 2025
    # -----------------------------------------------------

    selected_validation_probability = (
        threshold_model.predict_proba(
            X_threshold_validation
        )[:, 1]
        .astype(np.float32)
    )

    np.save(
        validation_probabilities_path,
        selected_validation_probability,
    )

    print(
        "Persisted 2025 validation probabilities."
    )


# ---------------------------------------------------------
# 4. Validar la población de probabilidades
# ---------------------------------------------------------

if (
    len(selected_validation_probability)
    != len(y_threshold_validation)
):

    raise ValueError(
        "Validation probabilities and target "
        "have different lengths."
    )


# ---------------------------------------------------------
# 5. Calcular la curva precision-recall
# ---------------------------------------------------------

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        y_threshold_validation,
        selected_validation_probability,
    )
)

threshold_precision = precision_curve[:-1]
threshold_recall = recall_curve[:-1]


# ---------------------------------------------------------
# 6. Identificar el mejor umbral con recall >= 0.80
# ---------------------------------------------------------

target_candidates = np.where(
    threshold_recall >= target_recall
)[0]

if len(target_candidates) == 0:

    raise ValueError(
        "No threshold reaches the requested "
        f"recall of {target_recall:.2f}."
    )

best_target_position = target_candidates[
    np.argmax(
        threshold_precision[
            target_candidates
        ]
    )
]

target_recall_threshold = float(
    thresholds[best_target_position]
)


# ---------------------------------------------------------
# 7. Evaluar el umbral de referencia y el candidato
# ---------------------------------------------------------

threshold_candidates = [
    {
        "threshold_type": "Reference threshold",
        "threshold": 0.5,
    },
    {
        "threshold_type": "Recall >= 0.80 candidate",
        "threshold": target_recall_threshold,
    },
]

threshold_results = []

for candidate in threshold_candidates:

    current_threshold = candidate[
        "threshold"
    ]

    current_prediction = (
        selected_validation_probability
        >= current_threshold
    ).astype(np.int8)

    true_positive = int(
        (
            (y_threshold_validation == 1)
            & (current_prediction == 1)
        ).sum()
    )

    false_positive = int(
        (
            (y_threshold_validation == 0)
            & (current_prediction == 1)
        ).sum()
    )

    false_negative = int(
        (
            (y_threshold_validation == 1)
            & (current_prediction == 0)
        ).sum()
    )

    true_negative = int(
        (
            (y_threshold_validation == 0)
            & (current_prediction == 0)
        ).sum()
    )

    predicted_positive = (
        true_positive
        + false_positive
    )

    threshold_results.append(
        {
            "threshold_type": candidate[
                "threshold_type"
            ],
            "threshold": current_threshold,
            "precision": precision_score(
                y_threshold_validation,
                current_prediction,
                zero_division=0,
            ),
            "recall": recall_score(
                y_threshold_validation,
                current_prediction,
                zero_division=0,
            ),
            "f1": f1_score(
                y_threshold_validation,
                current_prediction,
                zero_division=0,
            ),
            "true_positive": true_positive,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_negative": true_negative,
            "predicted_positive_rate": (
                predicted_positive
                / len(y_threshold_validation)
            ),
            "false_alert_share": (
                false_positive
                / predicted_positive
                if predicted_positive > 0
                else 0.0
            ),
        }
    )

threshold_comparison = pd.DataFrame(
    threshold_results
)


# ---------------------------------------------------------
# 8. Persistir la comparación de umbrales
# ---------------------------------------------------------

threshold_comparison.to_csv(
    threshold_results_path,
    index=False,
)


# ---------------------------------------------------------
# 9. Validar los resultados de la selección del umbral
# ---------------------------------------------------------

target_result = threshold_comparison.loc[
    threshold_comparison[
        "threshold_type"
    ]
    == "Recall >= 0.80 candidate"
].iloc[0]

reference_result = threshold_comparison.loc[
    threshold_comparison[
        "threshold_type"
    ]
    == "Reference threshold"
].iloc[0]

threshold_selection_validation = pd.DataFrame(
    {
        "check": [
            "Validation probabilities available",
            "Validation target size matches protocol",
            "Probability size matches validation target",
            "Reference threshold equals 0.5",
            "Target-recall threshold identified",
            "Target candidate reaches recall >= 0.80",
            "All threshold metrics finite",
            "2025 used for threshold selection",
            "2026 excluded from threshold selection",
        ],
        "valid": [
            len(
                selected_validation_probability
            ) > 0,
            (
                len(y_threshold_validation)
                == 7_597_494
            ),
            (
                len(
                    selected_validation_probability
                )
                == len(
                    y_threshold_validation
                )
            ),
            np.isclose(
                reference_result[
                    "threshold"
                ],
                0.5,
            ),
            np.isfinite(
                target_recall_threshold
            ),
            (
                target_result["recall"]
                >= target_recall
            ),
            np.isfinite(
                threshold_comparison[
                    [
                        "threshold",
                        "precision",
                        "recall",
                        "f1",
                        "predicted_positive_rate",
                        "false_alert_share",
                    ]
                ].to_numpy(dtype=float)
            ).all(),
            (
                validation_start_date.year
                == 2025
            ),
            (
                validation_end_date.year
                < 2026
            ),
        ],
    }
)


# ---------------------------------------------------------
# 10. Mostrar los resultados de la selección del umbral
# ---------------------------------------------------------

display(
    threshold_comparison.round(4)
)

display(
    threshold_selection_validation
)

print(
    "Target recall:",
    target_recall,
)

print(
    "Candidate threshold:",
    round(
        target_recall_threshold,
        6,
    ),
)

print(
    "Threshold selection valid:",
    threshold_selection_validation[
        "valid"
    ].all(),
)

print(
    "Threshold results persisted at:",
    threshold_results_path,
)

print(
    "Validation probabilities persisted at:",
    validation_probabilities_path,
)


# ---------------------------------------------------------
# 11. Liberar las matrices temporales de gran tamaño
# ---------------------------------------------------------

for object_name in [
    "X_threshold_train",
    "X_threshold_validation",
    "X_threshold_train_chunks",
    "X_threshold_validation_chunks",
]:

    if object_name in globals():
        del globals()[object_name]

gc.collect()

Preparing threshold matrices:   0%|          | 0/320 [00:00<?, ?file/s]

Threshold matrix preparation: 1.88 min
Training matrix: (21399093, 843)
Validation matrix: (7597494, 843)
Selected model training: 13.95 min
Persisted 2025 validation probabilities.


,threshold_type,threshold,precision,recall,f1,true_positive,false_positive,false_negative,true_negative,predicted_positive_rate,false_alert_share
0,Reference threshold,0.5000,0.3079,0.5944,0.4057,1001770,2251332,683563,3660829,0.4282,0.6921
1,Recall >= 0.80 candidate,0.4066,0.2689,0.8000,0.4025,1348270,3666597,337063,2245564,0.6601,0.7311


,check,valid
0,Validation probabilities available,True
1,Validation target size matches protocol,True
2,Probability size matches validation target,True
3,Reference threshold equals 0.5,True
4,Target-recall threshold identified,True
5,Target candidate reaches recall >= 0.80,True
6,All threshold metrics finite,True
7,2025 used for threshold selection,True
8,2026 excluded from threshold selection,True


Target recall: 0.8
Candidate threshold: 0.406588
Threshold selection valid: True
Threshold results persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\threshold_selection_2025.csv
Validation probabilities persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\selected_model_validation_probabilities_2025.npy


40

#### Interpretación de la selección del umbral de decisión

El análisis de la curva precision-recall muestra que es posible modificar de forma sustancial la capacidad de detección del modelo sin alterar su estructura, sus hiperparámetros ni el procedimiento de preprocesamiento.

Con el umbral de referencia de 0.5, la Logistic Regression seleccionada alcanza una precision de 0.3079, un recall de 0.5944 y un F1 de 0.4057. Bajo esta regla de decisión se identifican 1,001,770 de los 1,685,333 retrasos reales presentes en la validación temporal de 2025.

La búsqueda realizada exclusivamente sobre 2025 identifica un umbral de 0.406588 como punto de operación capaz de alcanzar un recall del 80 %. Con este valor se detectan aproximadamente 1,348,270 retrasos y los falsos negativos se reducen desde 683,563 hasta 337,063.

Este incremento de sensibilidad tiene un coste sobre la precision, que disminuye desde 0.3079 hasta 0.2689. Asimismo, las falsas alertas aumentan desde 2,251,332 hasta 3,666,597 y la proporción de observaciones clasificadas como positivas pasa del 42.82 % al 66.01 % del conjunto de validación. El 73.11 % de las alertas generadas con el nuevo umbral corresponde a falsos positivos.

A pesar de este desplazamiento, el F1 permanece relativamente estable, pasando de 0.4057 a 0.4025. Por tanto, el aumento del recall desde el 59.44 % hasta el 80.00 % no produce una degradación equivalente en esta medida conjunta, sino principalmente una redistribución del comportamiento del clasificador hacia una mayor sensibilidad.

Desde la perspectiva del objetivo predictivo del trabajo, el umbral de 0.406588 constituye un punto de operación defendible cuando se prioriza la detección de vuelos potencialmente retrasados frente a la minimización de falsas alertas. No obstante, su elevada tasa de falsos positivos debe considerarse explícitamente como una limitación de la utilización práctica del modelo.

La selección del umbral se ha realizado exclusivamente sobre la validación temporal de 2025. En consecuencia, este valor queda fijado antes del entrenamiento definitivo y deberá aplicarse sin modificaciones sobre el test externo de 2026 para preservar la independencia de dicha evaluación.

## 9. Entrenamiento definitivo del modelo

Una vez finalizadas las etapas de comparación, tratamiento del desbalance, optimización de hiperparámetros y selección del umbral de decisión, se procede al entrenamiento definitivo del modelo.

En esta etapa dejan de realizarse decisiones basadas en rendimiento. La configuración seleccionada queda fijada como una Logistic Regression con `C=0.1`, `tol=0.01`, regularización L2, optimizador `saga`, `class_weight="balanced"` y un umbral de decisión de 0.406588.

El entrenamiento definitivo utiliza conjuntamente los datos correspondientes al período 2022–2025. La incorporación de 2025 es metodológicamente válida en este punto porque dicho período ya ha cumplido su función como validación temporal y todas las decisiones relativas al modelo y a la regla de clasificación han quedado establecidas antes del reentrenamiento.

El preprocesamiento debe volver a aprenderse utilizando exclusivamente el conjunto definitivo de entrenamiento 2022–2025. En particular, se reconstruirá el vocabulario de las variables categóricas utilizado por `OneHotEncoder` y se estimarán nuevamente los parámetros de `StandardScaler`. Esto evita reutilizar parámetros aprendidos únicamente sobre 2022–2024 y garantiza que el modelo final aproveche toda la información disponible hasta el cierre del período de desarrollo.

El bloque se organiza en dos etapas:

1. Reconstrucción del preprocesamiento definitivo sobre 2022–2025.
2. Entrenamiento y persistencia del modelo definitivo.

El período enero–mayo de 2026 permanece completamente excluido del entrenamiento y no será utilizado hasta la evaluación externa del Bloque 10.


### 9.1 Reconstrucción del preprocesamiento definitivo

El entrenamiento definitivo requiere reconstruir los componentes de preprocesamiento utilizando la totalidad del período 2022–2025.

Para las variables categóricas se obtiene nuevamente el vocabulario de categorías a partir del conjunto definitivo de entrenamiento y se configura `OneHotEncoder` con dichas categorías, `handle_unknown="ignore"` y representación dispersa. Este procedimiento permite mantener una transformación eficiente y gestionar categorías potencialmente no observadas posteriormente en el test externo.

Para las variables numéricas se vuelve a ajustar `StandardScaler` exclusivamente con las observaciones de 2022–2025 mediante procesamiento incremental. De esta forma, los parámetros de centrado y escala corresponden al mismo período utilizado posteriormente para entrenar el modelo definitivo.

La reconstrucción se realiza mediante lectura incremental de los archivos Parquet y únicamente sobre las columnas necesarias. Esta estrategia resulta especialmente relevante debido al volumen del conjunto de datos y evita materializar simultáneamente en memoria toda la población de entrenamiento.

El período de 2026 no interviene en ninguna operación de ajuste del preprocesamiento.

In [33]:
# ---------------------------------------------------------
# 1. Importar los componentes necesarios
# ---------------------------------------------------------

import gc
import time

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Inicializar estructuras para el preprocesamiento final
# ---------------------------------------------------------

final_preprocessing_start = time.time()

final_category_sets = {
    feature: set()
    for feature in categorical_modeling_features
}

final_numerical_scaler = StandardScaler()

final_training_row_count = 0
final_training_positive_count = 0


# ---------------------------------------------------------
# 3. Recorrer incrementalmente el período 2022-2025
# ---------------------------------------------------------

for parquet_file in tqdm(
    development_parquet_files,
    desc="Reconstruyendo preprocesamiento final",
    unit="file",
):

    data_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            *categorical_modeling_features,
            *numerical_modeling_features,
            target_variable,
        ],
    )

    data_chunk[temporal_variable] = pd.to_datetime(
        data_chunk[temporal_variable]
    )

    final_training_mask = (
        (
            data_chunk[temporal_variable]
            >= training_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= validation_end_date
        )
    )

    final_training_chunk = data_chunk.loc[
        final_training_mask
    ]

    if final_training_chunk.empty:
        continue


    # -----------------------------------------------------
    # 3.1 Actualizar el vocabulario categórico
    # -----------------------------------------------------

    for feature in categorical_modeling_features:

        final_category_sets[feature].update(
            final_training_chunk[
                feature
            ]
            .dropna()
            .unique()
            .tolist()
        )


    # -----------------------------------------------------
    # 3.2 Actualizar el escalador numérico
    # -----------------------------------------------------

    final_numerical_scaler.partial_fit(
        final_training_chunk[
            numerical_modeling_features
        ]
    )


    # -----------------------------------------------------
    # 3.3 Actualizar los controles de población
    # -----------------------------------------------------

    final_training_row_count += len(
        final_training_chunk
    )

    final_training_positive_count += int(
        final_training_chunk[
            target_variable
        ].sum()
    )


# ---------------------------------------------------------
# 4. Ordenar las categorías aprendidas
# ---------------------------------------------------------

final_categories = [
    sorted(
        final_category_sets[feature]
    )
    for feature in categorical_modeling_features
]


# ---------------------------------------------------------
# 5. Construir el codificador categórico definitivo
# ---------------------------------------------------------

final_categorical_encoder = OneHotEncoder(
    categories=final_categories,
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)

final_encoder_fit_data = pd.DataFrame(
    {
        feature: [
            categories[0]
        ]
        for feature, categories in zip(
            categorical_modeling_features,
            final_categories,
        )
    }
)

final_categorical_encoder.fit(
    final_encoder_fit_data
)


# ---------------------------------------------------------
# 6. Calcular la dimensionalidad transformada
# ---------------------------------------------------------

final_categorical_feature_count = sum(
    len(categories)
    for categories in final_categories
)

final_numerical_feature_count = len(
    numerical_modeling_features
)

final_transformed_feature_count = (
    final_categorical_feature_count
    + final_numerical_feature_count
)

final_training_negative_count = (
    final_training_row_count
    - final_training_positive_count
)

final_training_positive_rate = (
    final_training_positive_count
    / final_training_row_count
)

final_preprocessing_minutes = (
    time.time()
    - final_preprocessing_start
) / 60


# ---------------------------------------------------------
# 7. Construir el resumen del preprocesamiento definitivo
# ---------------------------------------------------------

final_preprocessing_summary = pd.DataFrame(
    {
        "indicator": [
            "Training period",
            "Training observations",
            "Class 0 observations",
            "Class 1 observations",
            "Positive rate",
            "Categorical predictors",
            "Numerical predictors",
            "OHE features",
            "Numerical features",
            "Total transformed features",
            "Processing time (min)",
        ],
        "value": [
            "2022-2025",
            final_training_row_count,
            final_training_negative_count,
            final_training_positive_count,
            final_training_positive_rate,
            len(categorical_modeling_features),
            len(numerical_modeling_features),
            final_categorical_feature_count,
            final_numerical_feature_count,
            final_transformed_feature_count,
            final_preprocessing_minutes,
        ],
    }
)


# ---------------------------------------------------------
# 8. Validar el preprocesamiento definitivo
# ---------------------------------------------------------

final_preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "Training population matches development set",
            "Class 0 count matches development set",
            "Class 1 count matches development set",
            "All categorical predictors have categories",
            "Numerical scaler fitted",
            "Encoder fitted",
            "Transformed feature count is positive",
            "Final preprocessing uses 2022-2025",
            "2026 excluded from preprocessing fit",
        ],
        "valid": [
            (
                final_training_row_count
                == 28_996_587
            ),
            (
                final_training_negative_count
                == 22_889_330
            ),
            (
                final_training_positive_count
                == 6_107_257
            ),
            all(
                len(categories) > 0
                for categories in final_categories
            ),
            hasattr(
                final_numerical_scaler,
                "mean_",
            ),
            hasattr(
                final_categorical_encoder,
                "categories_",
            ),
            (
                final_transformed_feature_count
                > 0
            ),
            (
                training_start_date.year == 2022
                and validation_end_date.year == 2025
            ),
            (
                validation_end_date.year < 2026
            ),
        ],
    }
)


# ---------------------------------------------------------
# 9. Mostrar los resultados consolidados
# ---------------------------------------------------------

display(
    final_preprocessing_summary
)

display(
    final_preprocessing_validation
)

print(
    "Final preprocessing valid:",
    final_preprocessing_validation[
        "valid"
    ].all(),
)

print(
    "Final transformed features:",
    final_transformed_feature_count,
)


# ---------------------------------------------------------
# 10. Liberar estructuras temporales
# ---------------------------------------------------------

del final_category_sets
del final_encoder_fit_data

gc.collect()

Reconstruyendo preprocesamiento final:   0%|          | 0/320 [00:00<?, ?file/s]

,indicator,value
0,Training period,2022-2025
1,Training observations,28996587
2,Class 0 observations,22889330
3,Class 1 observations,6107257
4,Positive rate,0.21062
5,Categorical predictors,8
6,Numerical predictors,2
7,OHE features,851
8,Numerical features,2
9,Total transformed features,853


,check,valid
0,Training population matches development set,True
1,Class 0 count matches development set,True
2,Class 1 count matches development set,True
3,All categorical predictors have categories,True
4,Numerical scaler fitted,True
5,Encoder fitted,True
6,Transformed feature count is positive,True
7,Final preprocessing uses 2022-2025,True
8,2026 excluded from preprocessing fit,True


Final preprocessing valid: True
Final transformed features: 853


18

#### Interpretación de la reconstrucción del preprocesamiento definitivo

La reconstrucción del preprocesamiento sobre el período completo 2022–2025 se realizó correctamente sobre 28,996,587 observaciones, de las cuales 22,889,330 pertenecen a la clase negativa y 6,107,257 a la clase positiva. La prevalencia resultante de retrasos es del 21.06 %, consistente con la distribución previamente caracterizada para el conjunto de desarrollo.

El vocabulario aprendido para las ocho variables categóricas genera 851 variables mediante One-Hot Encoding. Junto con las dos variables numéricas estandarizadas, la representación definitiva queda formada por 853 variables transformadas.

Esta dimensionalidad es superior a las 843 variables obtenidas durante el desarrollo sobre 2022–2024. La diferencia se explica por la incorporación de categorías observadas en 2025 que no estaban presentes en el período anterior. Este incremento es metodológicamente coherente, ya que el preprocesamiento definitivo debe aprenderse nuevamente utilizando toda la información disponible en el conjunto final de entrenamiento.

El `StandardScaler` y el `OneHotEncoder` quedan, por tanto, ajustados exclusivamente sobre 2022–2025 y preparados para transformar posteriormente el test externo de 2026 sin volver a estimar ninguno de sus parámetros.

Las comprobaciones realizadas confirman que la población, la distribución de clases y el período utilizado son los esperados y que ninguna observación de 2026 ha intervenido en la estimación del preprocesamiento.

### 9.2 Entrenamiento y persistencia del modelo definitivo

Una vez reconstruido el preprocesamiento sobre el período completo 2022–2025, se procede al entrenamiento definitivo de la Logistic Regression seleccionada.

Las observaciones del conjunto de entrenamiento se transforman utilizando exclusivamente el `OneHotEncoder` y el `StandardScaler` ajustados en el subbloque anterior. La representación resultante se mantiene en formato disperso para reducir el coste de memoria asociado a las variables generadas mediante One-Hot Encoding.

El modelo conserva sin modificaciones la configuración seleccionada durante el proceso experimental: `C=0.1`, `tol=0.01`, regularización L2, optimizador `saga`, `class_weight="balanced"`, un máximo de 500 iteraciones y `random_state=42`. No se realizan nuevas comparaciones ni ajustes basados en rendimiento.

Una vez completado el entrenamiento, se persisten el modelo, el codificador categórico y el escalador numérico. Estos objetos constituyen los componentes necesarios para transformar y evaluar posteriormente el test externo de 2026 bajo exactamente la misma configuración aprendida sobre 2022–2025.

El umbral de decisión de 0.406588, seleccionado previamente utilizando exclusivamente la validación temporal de 2025, también queda registrado como parte de la configuración definitiva. Dicho umbral no interviene en el ajuste de la Logistic Regression, sino que se aplicará posteriormente sobre las probabilidades estimadas para obtener las predicciones binarias.

El período enero–mayo de 2026 permanece excluido tanto de la transformación utilizada para el entrenamiento como del ajuste del modelo.

In [34]:
# ---------------------------------------------------------
# 1. Importar los componentes necesarios
# ---------------------------------------------------------

import gc
import json
import time
import joblib

from scipy.sparse import hstack, vstack
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Definir las rutas de persistencia
# ---------------------------------------------------------

final_model_path = (
    modeling_results_path
    / "final_logistic_regression_2022_2025.joblib"
)

final_encoder_path = (
    modeling_results_path
    / "final_categorical_encoder_2022_2025.joblib"
)

final_scaler_path = (
    modeling_results_path
    / "final_numerical_scaler_2022_2025.joblib"
)

final_configuration_path = (
    modeling_results_path
    / "final_model_configuration.json"
)


# ---------------------------------------------------------
# 3. Materializar la matriz definitiva de entrenamiento
# ---------------------------------------------------------

X_final_train_chunks = []
y_final_train_chunks = []

final_matrix_start = time.time()

for parquet_file in tqdm(
    development_parquet_files,
    desc="Construyendo matriz final de entrenamiento",
    unit="file",
):

    data_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            temporal_variable,
            *categorical_modeling_features,
            *numerical_modeling_features,
            target_variable,
        ],
    )

    data_chunk[temporal_variable] = pd.to_datetime(
        data_chunk[temporal_variable]
    )

    final_training_mask = (
        (
            data_chunk[temporal_variable]
            >= training_start_date
        )
        & (
            data_chunk[temporal_variable]
            <= validation_end_date
        )
    )

    final_training_chunk = data_chunk.loc[
        final_training_mask
    ]

    if final_training_chunk.empty:
        continue


    # -----------------------------------------------------
    # 3.1 Transformar las variables categóricas
    # -----------------------------------------------------

    categorical_matrix = (
        final_categorical_encoder.transform(
            final_training_chunk[
                categorical_modeling_features
            ]
        )
    )


    # -----------------------------------------------------
    # 3.2 Transformar las variables numéricas
    # -----------------------------------------------------

    numerical_matrix = (
        final_numerical_scaler.transform(
            final_training_chunk[
                numerical_modeling_features
            ]
        )
        .astype(np.float32)
    )


    # -----------------------------------------------------
    # 3.3 Combinar las variables transformadas
    # -----------------------------------------------------

    transformed_matrix = hstack(
        [
            categorical_matrix,
            numerical_matrix,
        ],
        format="csr",
        dtype=np.float32,
    )

    X_final_train_chunks.append(
        transformed_matrix
    )

    y_final_train_chunks.append(
        final_training_chunk[
            target_variable
        ].to_numpy(
            dtype=np.int8
        )
    )


# ---------------------------------------------------------
# 4. Consolidar la matriz y la variable objetivo
# ---------------------------------------------------------

X_final_train = vstack(
    X_final_train_chunks,
    format="csr",
)

y_final_train = np.concatenate(
    y_final_train_chunks
)

final_matrix_minutes = (
    time.time()
    - final_matrix_start
) / 60


# ---------------------------------------------------------
# 5. Validar la matriz antes del entrenamiento
# ---------------------------------------------------------

expected_final_shape = (
    final_training_row_count,
    final_transformed_feature_count,
)

if X_final_train.shape != expected_final_shape:
    raise ValueError(
        "La matriz final no presenta las "
        "dimensiones esperadas. "
        f"Esperado: {expected_final_shape}. "
        f"Obtenido: {X_final_train.shape}."
    )

if len(y_final_train) != final_training_row_count:
    raise ValueError(
        "La variable objetivo no contiene "
        "el número esperado de observaciones."
    )

if int(y_final_train.sum()) != final_training_positive_count:
    raise ValueError(
        "El número de observaciones positivas "
        "no coincide con la caracterización "
        "del conjunto definitivo."
    )

print(
    "Final training matrix:",
    X_final_train.shape,
)

print(
    "Final target:",
    y_final_train.shape,
)

print(
    "Matrix preparation:",
    f"{final_matrix_minutes:.2f} min",
)


# ---------------------------------------------------------
# 6. Construir el modelo definitivo
# ---------------------------------------------------------

final_model = LogisticRegression(
    C=definitive_model_configuration["C"],
    tol=definitive_model_configuration["tol"],
    solver=definitive_model_configuration["solver"],
    penalty=definitive_model_configuration["penalty"],
    class_weight=definitive_model_configuration[
        "class_weight"
    ],
    max_iter=definitive_model_configuration[
        "max_iter"
    ],
    random_state=definitive_model_configuration[
        "random_state"
    ],
)


# ---------------------------------------------------------
# 7. Entrenar el modelo definitivo
# ---------------------------------------------------------

final_training_start = time.time()

final_model.fit(
    X_final_train,
    y_final_train,
)

final_model_training_minutes = (
    time.time()
    - final_training_start
) / 60

print(
    "Final model training:",
    f"{final_model_training_minutes:.2f} min",
)


# ---------------------------------------------------------
# 8. Fijar la regla de decisión definitiva
# ---------------------------------------------------------

final_decision_threshold = float(
    target_recall_threshold
)

decision_rule_configuration[
    "final_threshold_status"
] = "Selected"

decision_rule_configuration[
    "final_threshold"
] = final_decision_threshold


# ---------------------------------------------------------
# 9. Construir la configuración persistible
# ---------------------------------------------------------

final_persisted_configuration = {
    "model": "Logistic Regression",
    "training_period": "2022-2025",
    "external_test_period": "2026-01 to 2026-05",
    "C": float(
        definitive_model_configuration["C"]
    ),
    "tol": float(
        definitive_model_configuration["tol"]
    ),
    "solver": definitive_model_configuration[
        "solver"
    ],
    "penalty": definitive_model_configuration[
        "penalty"
    ],
    "class_weight": definitive_model_configuration[
        "class_weight"
    ],
    "max_iter": int(
        definitive_model_configuration["max_iter"]
    ),
    "random_state": int(
        definitive_model_configuration["random_state"]
    ),
    "decision_threshold": final_decision_threshold,
    "threshold_selection_period": "2025",
    "categorical_features": list(
        categorical_modeling_features
    ),
    "numerical_features": list(
        numerical_modeling_features
    ),
    "transformed_features": int(
        final_transformed_feature_count
    ),
}


# ---------------------------------------------------------
# 10. Persistir el modelo y el preprocesamiento
# ---------------------------------------------------------

joblib.dump(
    final_model,
    final_model_path,
)

joblib.dump(
    final_categorical_encoder,
    final_encoder_path,
)

joblib.dump(
    final_numerical_scaler,
    final_scaler_path,
)

with open(
    final_configuration_path,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        final_persisted_configuration,
        configuration_file,
        indent=4,
        ensure_ascii=False,
    )


# ---------------------------------------------------------
# 11. Construir el resumen del entrenamiento definitivo
# ---------------------------------------------------------

final_training_summary = pd.DataFrame(
    {
        "indicator": [
            "Training period",
            "Training observations",
            "Class 0 observations",
            "Class 1 observations",
            "Transformed features",
            "Decision threshold",
            "Iterations used",
            "Matrix preparation (min)",
            "Model training (min)",
        ],
        "value": [
            "2022-2025",
            final_training_row_count,
            final_training_negative_count,
            final_training_positive_count,
            final_transformed_feature_count,
            final_decision_threshold,
            int(final_model.n_iter_[0]),
            final_matrix_minutes,
            final_model_training_minutes,
        ],
    }
)


# ---------------------------------------------------------
# 12. Validar el entrenamiento y la persistencia
# ---------------------------------------------------------

final_training_validation = pd.DataFrame(
    {
        "check": [
            "Training rows match final population",
            "Training columns match final preprocessing",
            "Target rows match training matrix",
            "Positive class count preserved",
            "Selected model configuration preserved",
            "Final threshold fixed",
            "Model fitted",
            "Model persisted",
            "Categorical encoder persisted",
            "Numerical scaler persisted",
            "Configuration persisted",
            "2026 excluded from final training",
        ],
        "valid": [
            (
                X_final_train.shape[0]
                == final_training_row_count
            ),
            (
                X_final_train.shape[1]
                == final_transformed_feature_count
            ),
            (
                len(y_final_train)
                == X_final_train.shape[0]
            ),
            (
                int(y_final_train.sum())
                == final_training_positive_count
            ),
            (
                final_model.get_params()["C"]
                == definitive_model_configuration["C"]
                and final_model.get_params()["tol"]
                == definitive_model_configuration["tol"]
                and final_model.get_params()["class_weight"]
                == "balanced"
            ),
            np.isclose(
                final_decision_threshold,
                0.406588,
                atol=1e-6,
            ),
            hasattr(
                final_model,
                "coef_",
            ),
            final_model_path.exists(),
            final_encoder_path.exists(),
            final_scaler_path.exists(),
            final_configuration_path.exists(),
            (
                validation_end_date.year
                < 2026
            ),
        ],
    }
)


# ---------------------------------------------------------
# 13. Mostrar los resultados consolidados
# ---------------------------------------------------------

display(
    final_training_summary
)

display(
    final_training_validation
)

print(
    "Final training valid:",
    final_training_validation[
        "valid"
    ].all(),
)

print(
    "Final model persisted at:",
    final_model_path,
)

print(
    "Final decision threshold:",
    round(
        final_decision_threshold,
        6,
    ),
)


# ---------------------------------------------------------
# 14. Liberar la matriz definitiva de entrenamiento
# ---------------------------------------------------------

del X_final_train
del y_final_train
del X_final_train_chunks
del y_final_train_chunks

gc.collect()

Construyendo matriz final de entrenamiento:   0%|          | 0/320 [00:00<?, ?file/s]

Final training matrix: (28996587, 853)
Final target: (28996587,)
Matrix preparation: 1.78 min
Final model training: 25.85 min


,indicator,value
0,Training period,2022-2025
1,Training observations,28996587
2,Class 0 observations,22889330
3,Class 1 observations,6107257
4,Transformed features,853
5,Decision threshold,0.406588
6,Iterations used,31
7,Matrix preparation (min),1.783643
8,Model training (min),25.849083


,check,valid
0,Training rows match final population,True
1,Training columns match final preprocessing,True
2,Target rows match training matrix,True
3,Positive class count preserved,True
4,Selected model configuration preserved,True
5,Final threshold fixed,True
6,Model fitted,True
7,Model persisted,True
8,Categorical encoder persisted,True
9,Numerical scaler persisted,True


Final training valid: True
Final model persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\final_logistic_regression_2022_2025.joblib
Final decision threshold: 0.406588


51

#### Interpretación

El modelo definitivo se entrenó correctamente utilizando las 28,996,587 observaciones correspondientes al período 2022–2025 y la representación transformada de 853 variables obtenida mediante el preprocesamiento reconstruido específicamente para esta etapa.

La Logistic Regression conservó sin modificaciones la configuración seleccionada durante el proceso experimental: `C=0.1`, `tol=0.01`, regularización L2, optimizador `saga`, `class_weight="balanced"` y un máximo de 500 iteraciones. El ajuste convergió en 31 iteraciones, sin necesidad de alcanzar el límite máximo establecido.

La preparación de la matriz de entrenamiento requirió aproximadamente 1.78 minutos, mientras que el ajuste definitivo del modelo necesitó aproximadamente 25.85 minutos. Estos tiempos reflejan el coste computacional de entrenar sobre la totalidad del período de desarrollo una vez finalizada la selección metodológica.

El umbral de decisión definitivo se mantiene fijado en 0.406588, valor seleccionado previamente utilizando exclusivamente la validación temporal de 2025. Este umbral no intervino en el entrenamiento del modelo, sino que se utilizará posteriormente para transformar las probabilidades estimadas en predicciones binarias.

El modelo, el codificador categórico, el escalador numérico y la configuración definitiva fueron persistidos correctamente. De esta forma, la evaluación externa podrá realizarse reutilizando exactamente los objetos aprendidos sobre 2022–2025, sin realizar ningún nuevo ajuste sobre los datos de 2026.

Las comprobaciones realizadas confirman además que el número de observaciones, la dimensionalidad transformada, la distribución de la variable objetivo y la configuración seleccionada se mantienen consistentes con las etapas anteriores. El período enero–mayo de 2026 permaneció completamente excluido del entrenamiento.

## 10. Evaluación externa sobre 2026

Una vez entrenado y persistido el modelo definitivo, se realiza su evaluación sobre el período enero–mayo de 2026, reservado desde el inicio como test externo.

Esta etapa constituye la evaluación principal de generalización temporal del modelo. Los datos de 2026 no han intervenido en la selección de variables, el ajuste del preprocesamiento, la comparación de algoritmos, el tratamiento del desbalance, la optimización de hiperparámetros ni la selección del umbral de decisión.

La evaluación reutiliza exclusivamente el `OneHotEncoder`, el `StandardScaler`, la Logistic Regression y el umbral de 0.406588 fijados antes de acceder al test externo. Sobre las probabilidades obtenidas se calculan las métricas ROC-AUC y Average Precision, mientras que las predicciones binarias generadas mediante el umbral definitivo permiten calcular precision, recall, F1 y la matriz de confusión.

El bloque se organiza en tres etapas:

1. Evaluación externa del modelo definitivo sobre enero–mayo de 2026.
2. Comparación entre la validación temporal de 2025 y el test externo de 2026.
3. Interpretación de la capacidad de generalización temporal.

Ningún resultado obtenido sobre 2026 será utilizado para reajustar el modelo o modificar el umbral, preservando así el carácter externo de esta evaluación.

### 10.1 Evaluación externa del modelo definitivo

La evaluación externa se realiza sobre las observaciones correspondientes al período enero–mayo de 2026, mantenidas completamente aisladas durante las etapas anteriores del proceso experimental.

Cada archivo se procesa de forma incremental y se transforma utilizando exclusivamente el codificador categórico y el escalador numérico aprendidos sobre 2022–2025. No se ejecuta ninguna operación de ajuste (`fit` o `partial_fit`) sobre los datos de test.

La Logistic Regression definitiva genera una probabilidad de retraso para cada observación. Estas probabilidades permiten evaluar la capacidad discriminativa mediante ROC-AUC y Average Precision. Posteriormente se aplica el umbral definitivo de 0.406588, seleccionado previamente sobre la validación temporal de 2025, para obtener las predicciones binarias y calcular precision, recall, F1 y la matriz de confusión.

El objetivo es determinar hasta qué punto el comportamiento observado durante el desarrollo se mantiene sobre un período cronológicamente posterior e independiente. Los resultados obtenidos en esta etapa se consideran finales y no se utilizarán para reajustar ningún componente del modelo.

In [39]:
# ---------------------------------------------------------
# 1. Importar los componentes necesarios
# ---------------------------------------------------------

import gc
import time

from scipy.sparse import hstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from tqdm.auto import tqdm


# ---------------------------------------------------------
# 2. Recuperar los archivos del test externo de 2026
# ---------------------------------------------------------

test_parquet_files = sorted(
    external_test_data_path.rglob("*.parquet")
)

if len(test_parquet_files) == 0:
    raise FileNotFoundError(
        "No se encontraron archivos Parquet "
        "en la ruta del test externo."
    )

print(
    "Test Parquet files:",
    len(test_parquet_files),
)


# ---------------------------------------------------------
# 3. Inicializar estructuras para la evaluación externa
# ---------------------------------------------------------

external_test_start = time.time()

external_target_chunks = []
external_probability_chunks = []

external_test_row_count = 0


# ---------------------------------------------------------
# 4. Procesar incrementalmente el test externo de 2026
# ---------------------------------------------------------

for parquet_file in tqdm(
    test_parquet_files,
    desc="Evaluando test externo 2026",
    unit="file",
):

    test_chunk = pd.read_parquet(
        parquet_file,
        columns=[
            *categorical_modeling_features,
            *numerical_modeling_features,
            target_variable,
        ],
    )


    # -----------------------------------------------------
    # 4.1 Transformar las variables categóricas
    # -----------------------------------------------------

    categorical_matrix = (
        final_categorical_encoder.transform(
            test_chunk[
                categorical_modeling_features
            ]
        )
    )


    # -----------------------------------------------------
    # 4.2 Transformar las variables numéricas
    # -----------------------------------------------------

    numerical_matrix = (
        final_numerical_scaler.transform(
            test_chunk[
                numerical_modeling_features
            ]
        )
        .astype(np.float32)
    )


    # -----------------------------------------------------
    # 4.3 Construir la representación transformada
    # -----------------------------------------------------

    transformed_test_chunk = hstack(
        [
            categorical_matrix,
            numerical_matrix,
        ],
        format="csr",
        dtype=np.float32,
    )


    # -----------------------------------------------------
    # 4.4 Generar probabilidades sin reajustar el modelo
    # -----------------------------------------------------

    probability_chunk = (
        final_model.predict_proba(
            transformed_test_chunk
        )[:, 1]
        .astype(np.float32)
    )

    target_chunk = (
        test_chunk[
            target_variable
        ]
        .to_numpy(
            dtype=np.int8
        )
    )

    external_probability_chunks.append(
        probability_chunk
    )

    external_target_chunks.append(
        target_chunk
    )

    external_test_row_count += len(
        target_chunk
    )


# ---------------------------------------------------------
# 5. Consolidar probabilidades y variable objetivo
# ---------------------------------------------------------

external_test_probability = np.concatenate(
    external_probability_chunks
)

external_test_target = np.concatenate(
    external_target_chunks
)


# ---------------------------------------------------------
# 6. Aplicar el umbral definitivo
# ---------------------------------------------------------

external_test_prediction = (
    external_test_probability
    >= final_decision_threshold
).astype(np.int8)


# ---------------------------------------------------------
# 7. Calcular las métricas externas
# ---------------------------------------------------------

external_accuracy = accuracy_score(
    external_test_target,
    external_test_prediction,
)

external_precision = precision_score(
    external_test_target,
    external_test_prediction,
    zero_division=0,
)

external_recall = recall_score(
    external_test_target,
    external_test_prediction,
    zero_division=0,
)

external_f1 = f1_score(
    external_test_target,
    external_test_prediction,
    zero_division=0,
)

external_roc_auc = roc_auc_score(
    external_test_target,
    external_test_probability,
)

external_pr_auc = average_precision_score(
    external_test_target,
    external_test_probability,
)


# ---------------------------------------------------------
# 8. Calcular la matriz de confusión
# ---------------------------------------------------------

(
    external_tn,
    external_fp,
    external_fn,
    external_tp,
) = confusion_matrix(
    external_test_target,
    external_test_prediction,
    labels=[0, 1],
).ravel()

external_predicted_positive_count = (
    external_tp
    + external_fp
)

external_predicted_positive_rate = (
    external_predicted_positive_count
    / external_test_row_count
)

external_false_alert_share = (
    external_fp
    / external_predicted_positive_count
    if external_predicted_positive_count > 0
    else 0.0
)


# ---------------------------------------------------------
# 9. Calcular el tiempo total de evaluación
# ---------------------------------------------------------

external_evaluation_minutes = (
    time.time()
    - external_test_start
) / 60


# ---------------------------------------------------------
# 10. Construir el resumen de evaluación externa
# ---------------------------------------------------------

external_test_results = pd.DataFrame(
    [
        {
            "period": "2026-01 to 2026-05",
            "observations": (
                external_test_row_count
            ),
            "threshold": (
                final_decision_threshold
            ),
            "accuracy": (
                external_accuracy
            ),
            "precision": (
                external_precision
            ),
            "recall": (
                external_recall
            ),
            "f1": external_f1,
            "roc_auc": (
                external_roc_auc
            ),
            "pr_auc": (
                external_pr_auc
            ),
            "true_negative": int(
                external_tn
            ),
            "false_positive": int(
                external_fp
            ),
            "false_negative": int(
                external_fn
            ),
            "true_positive": int(
                external_tp
            ),
            "predicted_positive_rate": (
                external_predicted_positive_rate
            ),
            "false_alert_share": (
                external_false_alert_share
            ),
            "evaluation_minutes": (
                external_evaluation_minutes
            ),
        }
    ]
)


# ---------------------------------------------------------
# 11. Construir la matriz de confusión legible
# ---------------------------------------------------------

external_confusion_matrix = pd.DataFrame(
    {
        "Predicted no delay": [
            int(external_tn),
            int(external_fn),
        ],
        "Predicted delay": [
            int(external_fp),
            int(external_tp),
        ],
    },
    index=[
        "Actual no delay",
        "Actual delay",
    ],
)


# ---------------------------------------------------------
# 12. Validar la evaluación externa
# ---------------------------------------------------------

external_test_validation = pd.DataFrame(
    {
        "check": [
            "Test files recovered",
            "Test population matches persisted dataset",
            "Probability count matches test population",
            "Prediction count matches test population",
            "Class 0 count matches persisted dataset",
            "Class 1 count matches persisted dataset",
            "Final threshold preserved",
            "All predictive metrics finite",
            "Confusion matrix preserves population",
            "Final model fitted before external evaluation",
        ],
        "valid": [
            (
                len(test_parquet_files)
                == 34
            ),
            (
                external_test_row_count
                == 3_102_447
            ),
            (
                len(
                    external_test_probability
                )
                == external_test_row_count
            ),
            (
                len(
                    external_test_prediction
                )
                == external_test_row_count
            ),
            (
                int(
                    (
                        external_test_target
                        == 0
                    ).sum()
                )
                == 2_445_603
            ),
            (
                int(
                    (
                        external_test_target
                        == 1
                    ).sum()
                )
                == 656_844
            ),
            np.isclose(
                final_decision_threshold,
                target_recall_threshold,
            ),
            np.isfinite(
                external_test_results[
                    [
                        "accuracy",
                        "precision",
                        "recall",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                    ]
                ].to_numpy(
                    dtype=float
                )
            ).all(),
            (
                external_tn
                + external_fp
                + external_fn
                + external_tp
                == external_test_row_count
            ),
            hasattr(
                final_model,
                "coef_",
            ),
        ],
    }
)


# ---------------------------------------------------------
# 13. Persistir los resultados externos
# ---------------------------------------------------------

external_results_path = (
    modeling_results_path
    / "external_test_results_2026.csv"
)

external_test_results.to_csv(
    external_results_path,
    index=False,
)


# ---------------------------------------------------------
# 14. Mostrar los resultados consolidados
# ---------------------------------------------------------

display(
    external_test_results.round(4)
)

display(
    external_confusion_matrix
)

display(
    external_test_validation
)

print(
    "External test valid:",
    external_test_validation[
        "valid"
    ].all(),
)

print(
    "External results persisted at:",
    external_results_path,
)

print(
    "External test recall:",
    round(
        external_recall,
        4,
    ),
)

print(
    "External test precision:",
    round(
        external_precision,
        4,
    ),
)


# ---------------------------------------------------------
# 15. Liberar estructuras temporales
# ---------------------------------------------------------

del external_probability_chunks
del external_target_chunks

gc.collect()

Test Parquet files: 34


Evaluando test externo 2026:   0%|          | 0/34 [00:00<?, ?file/s]

,period,observations,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,true_negative,false_positive,false_negative,true_positive,predicted_positive_rate,false_alert_share,evaluation_minutes
0,2026-01 to 2026-05,3102447,0.4066,0.4569,0.2527,0.7994,0.384,0.6355,0.3053,892540,1553063,131739,525105,0.6698,0.7473,0.2606


,Predicted no delay,Predicted delay
Actual no delay,892540,1553063
Actual delay,131739,525105


,check,valid
0,Test files recovered,True
1,Test population matches persisted dataset,True
2,Probability count matches test population,True
3,Prediction count matches test population,True
4,Class 0 count matches persisted dataset,True
5,Class 1 count matches persisted dataset,True
6,Final threshold preserved,True
7,All predictive metrics finite,True
8,Confusion matrix preserves population,True
9,Final model fitted before external evaluation,True


External test valid: True
External results persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\external_test_results_2026.csv
External test recall: 0.7994
External test precision: 0.2527


258

#### Interpretación

La evaluación externa del modelo definitivo se realizó sobre 3,102,447 vuelos correspondientes al período enero–mayo de 2026, mantenido completamente aislado durante el desarrollo y la selección del modelo.

Aplicando sin modificaciones el umbral de decisión de 0.406588 seleccionado previamente sobre 2025, el modelo alcanzó un recall de 0.7994. Esto implica que aproximadamente el 79.94 % de los retrasos reales fueron correctamente identificados, manteniendo prácticamente el nivel del 80 % observado durante la validación temporal.

De los 656,844 vuelos que presentaron un retraso igual o superior a 15 minutos, el modelo detectó correctamente 525,105 y dejó sin identificar 131,739. Este resultado constituye evidencia favorable de estabilidad temporal en la capacidad de detección de la clase positiva.

La principal limitación continúa siendo la precision, que se situó en 0.2527. De los vuelos clasificados como potencialmente retrasados, aproximadamente una cuarta parte presentó finalmente el evento, mientras que el resto correspondió a falsas alertas. En términos absolutos, se produjeron 1,553,063 falsos positivos, lo que representa el 74.73 % de todas las predicciones positivas.

La capacidad discriminativa general también experimentó una reducción moderada respecto a la validación temporal. El ROC-AUC alcanzó 0.6355 y el Average Precision 0.3053, valores que continúan indicando una capacidad predictiva superior a una clasificación aleatoria, aunque ligeramente inferior a la observada en 2025.

El F1 alcanzó 0.3840, reflejando el compromiso entre una elevada sensibilidad y una precision reducida. La accuracy fue de 0.4569, pero esta métrica no constituye el criterio principal de evaluación, ya que el umbral definitivo fue seleccionado específicamente para priorizar la detección de retrasos y no la proporción global de clasificaciones correctas.

En conjunto, la evaluación externa confirma que el modelo mantiene de forma notablemente estable su capacidad para detectar aproximadamente ocho de cada diez retrasos en un período cronológicamente posterior e independiente. No obstante, esta sensibilidad se consigue a costa de una elevada proporción de falsas alarmas, lo que limita su utilización como predictor de alta certeza individual y orienta su interpretación hacia un sistema de detección temprana o priorización de riesgo.

### 10.2 Comparación entre la validación temporal de 2025 y el test externo de 2026

Una vez obtenidos los resultados sobre el test externo, se compara el comportamiento del modelo entre la validación temporal de 2025 y el período independiente enero–mayo de 2026.

La comparación tiene como objetivo cuantificar la estabilidad temporal de las principales métricas predictivas y determinar si la capacidad observada durante el desarrollo se mantiene al aplicar el modelo sobre datos cronológicamente posteriores.

Para garantizar una comparación coherente, ambos períodos se analizan utilizando la misma configuración definitiva del modelo y el mismo umbral de decisión de 0.406588. Se estudian precision, recall, F1, ROC-AUC y Average Precision, junto con la proporción de observaciones clasificadas como positivas y la proporción de falsas alertas.

Las diferencias observadas se interpretarán como evidencia de la capacidad de generalización temporal del modelo. Los resultados de 2026 mantienen exclusivamente una función evaluativa y no se utilizarán para modificar el modelo, el preprocesamiento o el umbral de decisión.

In [42]:
# ---------------------------------------------------------
# 1. Recuperar los resultados persistidos de 2025
# ---------------------------------------------------------

threshold_results_2025 = pd.read_csv(
    threshold_results_path
)

finalist_results_2025 = pd.read_csv(
    finalist_results_path
)


# ---------------------------------------------------------
# 2. Recuperar el umbral definitivo de 2025
# ---------------------------------------------------------

selected_threshold_rows = (
    threshold_results_2025.loc[
        np.isclose(
            threshold_results_2025["threshold"],
            final_decision_threshold,
            atol=1e-6,
        )
    ]
    .copy()
)

if len(selected_threshold_rows) != 1:
    raise ValueError(
        "No se pudo identificar de forma unívoca "
        "el umbral definitivo en los resultados de 2025."
    )

selected_threshold_2025 = (
    selected_threshold_rows.iloc[0]
)


# ---------------------------------------------------------
# 3. Recuperar el modelo definitivo evaluado en 2025
# ---------------------------------------------------------

selected_model_rows = (
    finalist_results_2025.loc[
        (
            finalist_results_2025["model"]
            == "Logistic Regression"
        )
        & np.isclose(
            finalist_results_2025["C"],
            definitive_model_configuration["C"],
        )
        & np.isclose(
            finalist_results_2025["tol"],
            definitive_model_configuration["tol"],
        )
    ]
    .copy()
)

if len(selected_model_rows) != 1:
    raise ValueError(
        "No se pudo identificar de forma unívoca "
        "la Logistic Regression definitiva "
        "en los resultados de 2025."
    )

selected_model_2025 = (
    selected_model_rows.iloc[0]
)


# ---------------------------------------------------------
# 4. Construir las métricas de validación de 2025
# ---------------------------------------------------------

validation_2025_metrics = {
    "period": "Validation 2025",
    "observations": 7_597_494,
    "threshold": float(
        selected_threshold_2025["threshold"]
    ),
    "precision": float(
        selected_threshold_2025["precision"]
    ),
    "recall": float(
        selected_threshold_2025["recall"]
    ),
    "f1": float(
        selected_threshold_2025["f1"]
    ),
    "roc_auc": float(
        selected_model_2025["roc_auc"]
    ),
    "pr_auc": float(
        selected_model_2025["pr_auc"]
    ),
    "predicted_positive_rate": float(
        selected_threshold_2025[
            "predicted_positive_rate"
        ]
    ),
    "false_alert_share": float(
        selected_threshold_2025[
            "false_alert_share"
        ]
    ),
}


# ---------------------------------------------------------
# 5. Construir las métricas externas de 2026
# ---------------------------------------------------------

external_2026_metrics = {
    "period": "External test 2026",
    "observations": int(
        external_test_results.loc[
            0,
            "observations",
        ]
    ),
    "threshold": float(
        external_test_results.loc[
            0,
            "threshold",
        ]
    ),
    "precision": float(
        external_test_results.loc[
            0,
            "precision",
        ]
    ),
    "recall": float(
        external_test_results.loc[
            0,
            "recall",
        ]
    ),
    "f1": float(
        external_test_results.loc[
            0,
            "f1",
        ]
    ),
    "roc_auc": float(
        external_test_results.loc[
            0,
            "roc_auc",
        ]
    ),
    "pr_auc": float(
        external_test_results.loc[
            0,
            "pr_auc",
        ]
    ),
    "predicted_positive_rate": float(
        external_test_results.loc[
            0,
            "predicted_positive_rate",
        ]
    ),
    "false_alert_share": float(
        external_test_results.loc[
            0,
            "false_alert_share",
        ]
    ),
}


# ---------------------------------------------------------
# 6. Construir la comparación temporal consolidada
# ---------------------------------------------------------

temporal_generalization_comparison = (
    pd.DataFrame(
        [
            validation_2025_metrics,
            external_2026_metrics,
        ]
    )
)


# ---------------------------------------------------------
# 7. Calcular las diferencias entre 2025 y 2026
# ---------------------------------------------------------

comparison_metrics = [
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive_rate",
    "false_alert_share",
]

temporal_metric_changes = pd.DataFrame(
    {
        "metric": comparison_metrics,
        "validation_2025": [
            validation_2025_metrics[metric]
            for metric in comparison_metrics
        ],
        "external_test_2026": [
            external_2026_metrics[metric]
            for metric in comparison_metrics
        ],
    }
)

temporal_metric_changes[
    "absolute_change"
] = (
    temporal_metric_changes[
        "external_test_2026"
    ]
    - temporal_metric_changes[
        "validation_2025"
    ]
)

temporal_metric_changes[
    "change_percentage_points"
] = (
    temporal_metric_changes[
        "absolute_change"
    ]
    * 100
)


# ---------------------------------------------------------
# 8. Validar la comparación temporal
# ---------------------------------------------------------

temporal_comparison_validation = pd.DataFrame(
    {
        "check": [
            "Selected 2025 threshold recovered",
            "Selected 2025 model recovered",
            "Same threshold used in both periods",
            "Validation population matches 2025",
            "External population matches 2026",
            "All comparison metrics finite",
        ],
        "valid": [
            (
                len(selected_threshold_rows)
                == 1
            ),
            (
                len(selected_model_rows)
                == 1
            ),
            np.isclose(
                validation_2025_metrics[
                    "threshold"
                ],
                external_2026_metrics[
                    "threshold"
                ],
                atol=1e-6,
            ),
            (
                validation_2025_metrics[
                    "observations"
                ]
                == 7_597_494
            ),
            (
                external_2026_metrics[
                    "observations"
                ]
                == 3_102_447
            ),
            np.isfinite(
                temporal_metric_changes[
                    [
                        "validation_2025",
                        "external_test_2026",
                        "absolute_change",
                    ]
                ].to_numpy(
                    dtype=float
                )
            ).all(),
        ],
    }
)


# ---------------------------------------------------------
# 9. Persistir la comparación temporal
# ---------------------------------------------------------

temporal_comparison_path = (
    modeling_results_path
    / "temporal_generalization_2025_2026.csv"
)

temporal_metric_changes.to_csv(
    temporal_comparison_path,
    index=False,
)


# ---------------------------------------------------------
# 10. Mostrar los resultados consolidados
# ---------------------------------------------------------

display(
    temporal_generalization_comparison.round(4)
)

display(
    temporal_metric_changes.round(4)
)

display(
    temporal_comparison_validation
)

print(
    "Temporal comparison valid:",
    temporal_comparison_validation[
        "valid"
    ].all(),
)

print(
    "Temporal comparison persisted at:",
    temporal_comparison_path,
)

,period,observations,threshold,precision,recall,f1,roc_auc,pr_auc,predicted_positive_rate,false_alert_share
0,Validation 2025,7597494,0.4066,0.2689,0.8000,0.4025,0.6456,0.3310,0.6601,0.7311
1,External test 2026,3102447,0.4066,0.2527,0.7994,0.3840,0.6355,0.3053,0.6698,0.7473


,metric,validation_2025,external_test_2026,absolute_change,change_percentage_points
0,precision,0.2689,0.2527,-0.0162,-1.6178
1,recall,0.8000,0.7994,-0.0006,-0.0566
2,f1,0.4025,0.3840,-0.0185,-1.8469
3,roc_auc,0.6456,0.6355,-0.0101,-1.0071
4,pr_auc,0.3310,0.3053,-0.0257,-2.5709
5,predicted_positive_rate,0.6601,0.6698,0.0098,0.9779
6,false_alert_share,0.7311,0.7473,0.0162,1.6178


,check,valid
0,Selected 2025 threshold recovered,True
1,Selected 2025 model recovered,True
2,Same threshold used in both periods,True
3,Validation population matches 2025,True
4,External population matches 2026,True
5,All comparison metrics finite,True


Temporal comparison valid: True
Temporal comparison persisted at: G:\My Drive\MASTER Big Data\TFM\results\modeling\temporal_generalization_2025_2026.csv


#### Interpretación

La comparación entre la validación temporal de 2025 y el test externo de 2026 muestra que el comportamiento del modelo se mantiene razonablemente estable al aplicarse sobre un período cronológicamente posterior.

El recall presenta prácticamente el mismo valor en ambos períodos, pasando de 0.8000 en 2025 a 0.7994 en 2026, una diferencia de únicamente -0.06 puntos porcentuales. Este resultado indica que la capacidad del modelo para detectar vuelos retrasados se mantiene de forma muy consistente fuera del período utilizado para la selección del umbral.

La precision disminuye de 0.2689 a 0.2527, equivalente a una reducción de 1.62 puntos porcentuales. De forma coherente, la proporción de falsas alertas aumenta de 73.11 % a 74.73 %. Por tanto, aunque el modelo conserva prácticamente intacta su sensibilidad, en 2026 necesita clasificar una proporción ligeramente mayor de vuelos como positivos para mantener ese nivel de detección.

El F1 disminuye de 0.4025 a 0.3840, lo que refleja el deterioro moderado del equilibrio entre precision y recall. Esta reducción está motivada principalmente por la pérdida de precision, ya que el recall permanece prácticamente inalterado.

Las métricas independientes del umbral también presentan una reducción. El ROC-AUC pasa de 0.6456 a 0.6355, mientras que el Average Precision disminuye de 0.3310 a 0.3053. Estas variaciones indican una pérdida moderada de capacidad discriminativa sobre el período externo, especialmente en la clasificación de la clase positiva, aunque el modelo continúa mostrando una capacidad predictiva superior a una referencia no informativa.

La proporción de vuelos clasificados como potencialmente retrasados aumenta de 66.01 % a 66.98 %. Esta variación es consistente con el incremento observado en las falsas alertas y confirma que el principal coste del umbral seleccionado continúa siendo una elevada tasa de falsos positivos.

En conjunto, los resultados proporcionan evidencia favorable de generalización temporal en términos de sensibilidad, dado que el modelo conserva prácticamente el 80 % de recall observado durante la validación. Sin embargo, la reducción de precision, F1, ROC-AUC y Average Precision evidencia que la capacidad de discriminación no se mantiene completamente estable. Por tanto, el modelo puede considerarse adecuado como mecanismo de detección temprana o priorización de riesgo, pero presenta limitaciones importantes si se requiere una elevada certeza en cada predicción positiva.

### 10.3 Interpretación consolidada de la generalización temporal

La evaluación externa sobre los datos correspondientes a enero–mayo de 2026 permite valorar la capacidad de generalización temporal del modelo definitivo bajo condiciones cronológicamente posteriores a las utilizadas durante su desarrollo. El modelo fue entrenado finalmente con información de 2022–2025 y aplicado sobre 2026 manteniendo sin modificaciones su configuración, procedimiento de preprocesamiento y umbral de decisión.

Los resultados muestran una elevada estabilidad del recall, que pasa de 0.8000 en la validación temporal de 2025 a 0.7994 en el test externo de 2026. La diferencia, de apenas -0.06 puntos porcentuales, indica que el modelo mantiene prácticamente intacta su capacidad para identificar vuelos que finalmente presentan un retraso igual o superior a 15 minutos.

Este resultado permite además contextualizar la decisión adoptada previamente en el apartado 8.4, donde la selección del umbral se orientó a alcanzar un recall próximo al 80 %. Frente al umbral convencional de 0.5, que proporcionaba un recall de 0.5944 y una precision de 0.3079, el umbral seleccionado de aproximadamente 0.4066 permitió elevar el recall hasta 0.8000, aunque reduciendo la precision a 0.2689. Por tanto, la configuración definitiva aceptó de forma explícita un mayor número de falsas alertas a cambio de reducir los falsos negativos y detectar una proporción considerablemente mayor de los retrasos reales.

Este compromiso resulta coherente con la interpretación funcional del modelo como componente de un sistema de alerta temprana de riesgo de retraso orientado al pasajero. En este contexto, un falso negativo representa un vuelo que finalmente se retrasa pero cuyo riesgo no fue identificado anticipadamente. Por este motivo, se otorgó mayor prioridad al recall que a la precision, buscando que el sistema dejara sin detectar la menor cantidad posible de retrasos reales. La evaluación externa de 2026 respalda la estabilidad de esta decisión, dado que el modelo continúa identificando aproximadamente ocho de cada diez retrasos reales sobre datos no utilizados durante su desarrollo.

No obstante, priorizar recall no implica que la precision carezca de importancia. En 2026, la precision disminuye de 0.2689 a 0.2527 y la proporción de falsas alertas aumenta de 73.11 % a 74.73 %. Esto significa que una parte elevada de las alertas generadas no termina correspondiendo a un retraso real, lo que podría reducir la utilidad práctica y la confianza del pasajero si las predicciones se comunicaran como afirmaciones de certeza. Por ello, la salida del modelo debe interpretarse como una señal de riesgo que permite priorizar vuelos potencialmente problemáticos y no como la afirmación de que un vuelo concreto necesariamente se retrasará.

La estabilidad observada en recall tampoco se reproduce completamente en las métricas de discriminación. El F1 disminuye de 0.4025 a 0.3840, el ROC-AUC de 0.6456 a 0.6355 y el Average Precision de 0.3310 a 0.3053. Estas variaciones evidencian una pérdida moderada de capacidad discriminativa en el período externo y confirman que la elevada cantidad de falsos positivos constituye la principal limitación del modelo.

En consecuencia, los resultados permiten diferenciar entre el objetivo operativo del sistema y su rendimiento predictivo absoluto. Para un sistema de alerta temprana se considera especialmente relevante detectar una proporción elevada de los retrasos reales, lo que justifica la priorización del recall durante la selección del umbral. Sin embargo, para mejorar su utilidad práctica sería deseable aumentar la precision y reducir las falsas alertas sin deteriorar sustancialmente el recall alcanzado.

En conjunto, los resultados muestran que existe capacidad predictiva utilizando exclusivamente información disponible antes de la salida del vuelo y que la sensibilidad obtenida durante la validación presenta una elevada estabilidad temporal. El modelo resulta, por tanto, más apropiado como mecanismo de detección temprana y priorización de riesgo para pasajeros que como predictor individual de alta certeza.

Finalmente, los resultados de 2026 se mantienen exclusivamente como evaluación externa del modelo definitivo y no se utilizan para modificar posteriormente su configuración, preprocesamiento o umbral. De este modo, se conserva el valor metodológico del test externo como estimación independiente de la capacidad de generalización temporal.

## 11. Conclusiones del modelado

El proceso de modelado desarrollado ha permitido construir y evaluar una solución predictiva orientada a la identificación anticipada del riesgo de que un vuelo alcance su destino con un retraso igual o superior a 15 minutos. La metodología seguida ha mantenido como principio central el respeto de la estructura temporal del problema, utilizando exclusivamente información disponible antes de la salida y separando cronológicamente las etapas de desarrollo, validación y evaluación externa.

La configuración definitiva se obtuvo a partir de la comparación sistemática de modelos, estrategias de tratamiento del desbalance e hiperparámetros sobre el período de desarrollo. Posteriormente, la regla de decisión se orientó hacia el funcionamiento del modelo como componente de un sistema de alerta temprana para pasajeros, estableciendo durante la selección del umbral un objetivo operativo de recall del 80 % sobre la clase positiva. Este criterio prioriza la identificación de una proporción elevada de los retrasos reales, sin considerar el recall de forma aislada y reconociendo explícitamente el compromiso resultante con la precision y las falsas alertas.

Una vez fijados el algoritmo, el preprocesamiento, el tratamiento del desbalance, los hiperparámetros y el umbral de decisión, el modelo fue reconstruido utilizando el período 2022–2025 y evaluado posteriormente sobre enero–mayo de 2026. Esta separación permite analizar su comportamiento sobre observaciones temporalmente posteriores que no participaron en las decisiones de selección. La comparación entre validación y evaluación externa constituye, por tanto, la principal evidencia para valorar la capacidad de generalización temporal de la solución obtenida.

El cierre del modelado debe integrar tanto las capacidades como las limitaciones observadas. En particular, resulta necesario distinguir entre la elevada cobertura de retrasos proporcionada por la regla de decisión y la capacidad discriminativa global del modelo, ya que el elevado número de falsas alertas condiciona su utilidad como predictor individual de alta certeza. Esta distinción resulta fundamental para establecer correctamente el alcance de las conclusiones del TFM y evitar una interpretación excesivamente optimista de los resultados.

En este bloque se abordarán progresivamente:

1. síntesis del proceso de selección y configuración definitiva del modelo;
2. valoración del cumplimiento del objetivo operativo de detección temprana;
3. análisis de la generalización temporal y principales limitaciones;
4. conclusiones finales del proceso de modelado.

### 11.1 Síntesis del proceso de selección y configuración definitiva del modelo

El proceso experimental permitió seleccionar una configuración definitiva a partir de una secuencia de comparación progresiva que incluyó un baseline de referencia, la evaluación inicial de diferentes algoritmos, el análisis de estrategias para el tratamiento del desbalance, la optimización dirigida de hiperparámetros y la posterior selección de una regla de decisión. Todas estas decisiones se realizaron dentro del período de desarrollo, manteniendo el período externo de 2026 separado del proceso de selección.

El baseline basado en la predicción sistemática de la clase mayoritaria alcanzó una accuracy elevada debido al desbalance de ARR_DEL15, pero presentó recall y F1 iguales a cero para la clase positiva. Este resultado confirmó que la accuracy, considerada de forma aislada, no constituía una referencia suficiente para evaluar la capacidad de detectar vuelos con retraso.

La evaluación de los modelos supervisados mostró que Logistic Regression proporcionaba una capacidad discriminativa superior al baseline y un comportamiento competitivo respecto a las demás alternativas analizadas. Sin embargo, los modelos entrenados sin tratamiento del desbalance mostraron una capacidad muy reducida para identificar la clase positiva. La incorporación de `class_weight="balanced"` modificó sustancialmente este comportamiento y permitió incrementar la detección de retrasos, por lo que fue seleccionada como estrategia principal frente a las alternativas evaluadas.

La optimización posterior confirmó Logistic Regression como modelo definitivo, utilizando regularización L2, solver `saga`, `C=0.1`, `tol=0.01`, `class_weight="balanced"`, `max_iter=500` y `random_state=42`. La selección de esta configuración consideró conjuntamente el rendimiento predictivo, el comportamiento sobre la clase positiva y el coste computacional. En particular, la tolerancia seleccionada permitió reducir considerablemente el tiempo de entrenamiento respecto a una configuración más restrictiva sin producir diferencias relevantes en las métricas obtenidas.

Sobre la configuración seleccionada se estableció posteriormente una regla de decisión orientada a la detección temprana. El umbral convencional de 0.5 proporcionaba un recall del 59.44 %, mientras que el umbral de aproximadamente 0.406588 permitió alcanzar el objetivo operativo de un recall cercano al 80 % durante la validación temporal de 2025. Esta modificación se realizó antes de la evaluación externa y supuso aceptar una reducción de precision y un incremento de falsas alertas a cambio de identificar una proporción considerablemente mayor de los retrasos reales.

Una vez fijadas todas las decisiones experimentales, el preprocesamiento se reconstruyó sobre el período 2022–2025 y el modelo definitivo se entrenó desde cero utilizando 28,996,587 observaciones y 853 características después de las transformaciones. El modelo y la regla de decisión resultantes quedaron congelados antes de realizar la evaluación externa correspondiente a enero–mayo de 2026.

En conjunto, el proceso de selección no estuvo orientado a maximizar una única métrica, sino a obtener una configuración reproducible, computacionalmente viable y coherente con la finalidad del modelo como componente de un sistema de alerta temprana de riesgo de retraso. La configuración resultante constituye la base sobre la que se evalúa, a continuación, el cumplimiento del objetivo operativo de detección de la clase positiva.

### 11.2 Valoración del cumplimiento del objetivo operativo de detección temprana

La selección de la regla de decisión se orientó a alcanzar un nivel elevado de detección de la clase positiva, estableciendo durante la fase de selección del umbral un objetivo operativo de recall del 80 %. Este valor no se interpreta como un óptimo estadístico universal, sino como un criterio de cobertura coherente con la utilización del modelo como componente de un sistema de alerta temprana de riesgo de retraso orientado al pasajero.

Desde esta perspectiva, el recall adquiere especial relevancia porque cuantifica la proporción de retrasos reales que son identificados anticipadamente por el sistema. Un recall del 80 % implica, por tanto, detectar aproximadamente ocho de cada diez vuelos que finalmente presentan un retraso igual o superior a 15 minutos. Los retrasos restantes constituyen falsos negativos y representan situaciones en las que el sistema no habría generado una alerta a pesar de producirse posteriormente el evento de interés.

Durante la validación temporal de 2025, el umbral convencional de 0.5 proporcionó un recall del 59.44 %, con una precision del 30.79 % y un F1 de 0.4057. La reducción del umbral hasta aproximadamente 0.406588 permitió elevar el recall hasta el 80.00 %. Esta ganancia de 20.56 puntos porcentuales en capacidad de detección estuvo acompañada por una disminución de la precision hasta el 26.89 %, mientras que el F1 se mantuvo prácticamente estable en 0.4025. La selección representa, por tanto, un compromiso explícito entre una mayor cobertura de retrasos reales y una mayor generación de falsas alertas.

Una vez fijado este criterio y sin modificar posteriormente el umbral a partir de información del período externo, la evaluación sobre enero–mayo de 2026 obtuvo un recall del 79.94 %. La diferencia respecto al 80.00 % observado en 2025 fue de aproximadamente -0.06 puntos porcentuales, por lo que la capacidad de detección asociada a la regla de decisión se mantuvo prácticamente inalterada sobre el período temporalmente posterior.

El elevado número de observaciones positivas permite, además, estimar el recall con una incertidumbre muestral reducida. En 2025 se registraron 1,348,270 verdaderos positivos sobre 1,685,333 retrasos reales, mientras que en 2026 se identificaron 525,105 sobre 656,844. Los intervalos de confianza aproximados del 95 % para estas proporciones se sitúan alrededor de 79.94 %–80.06 % en 2025 y 79.85 %–80.04 % en 2026. Estos resultados no demuestran que el 80 % constituya un umbral óptimo en términos estadísticos, pero sí aportan evidencia de que el nivel de recall observado está estimado con elevada precisión y presenta una notable estabilidad entre ambos períodos.

El cumplimiento del objetivo de sensibilidad no debe interpretarse de forma aislada. En el test externo de 2026, la precision fue del 25.27 % y el 74.73 % de las predicciones positivas correspondieron a falsas alertas. Por tanto, aunque el sistema consiguió identificar aproximadamente ocho de cada diez retrasos reales, una proporción elevada de las alertas emitidas no estuvo asociada finalmente a un retraso. Esta limitación condiciona su utilización como predictor individual de alta certeza y obliga a interpretar sus resultados como señales de riesgo y no como afirmaciones deterministas sobre el comportamiento futuro de cada vuelo.

En consecuencia, el objetivo operativo puede considerarse alcanzado durante la validación temporal y posteriormente reproducido con elevada estabilidad en el período externo. Sin embargo, su cumplimiento debe entenderse dentro del compromiso entre sensibilidad y precision adoptado para el sistema de alerta temprana. El resultado respalda la capacidad del modelo para proporcionar una cobertura elevada de los retrasos reales, mientras que la reducción de falsas alertas constituye la principal oportunidad de mejora de la solución desarrollada.

### 11.3 Análisis de la generalización temporal y principales limitaciones

La evaluación sobre un período temporalmente posterior permite analizar en qué medida el comportamiento observado durante el desarrollo se mantiene cuando el modelo se aplica a vuelos futuros. Esta comparación resulta especialmente relevante en el problema estudiado, ya que las relaciones entre las características disponibles antes de la salida y la ocurrencia de retrasos pueden variar con el tiempo debido a cambios operacionales, estacionales o estructurales del sistema aéreo.

La comparación entre la validación temporal de 2025 y el test externo correspondiente a enero–mayo de 2026 muestra una estabilidad especialmente elevada en la sensibilidad de la regla de decisión. El recall pasó del 80.00 % al 79.94 %, una diferencia de aproximadamente -0.06 puntos porcentuales. Este comportamiento indica que el nivel de cobertura de retrasos reales establecido durante la selección del umbral se reprodujo prácticamente sin cambios en el período externo.

Esta estabilidad del recall no se trasladó con la misma intensidad al resto de las métricas. La precision disminuyó del 26.89 % al 25.27 %, el F1 pasó de 0.4025 a 0.3840, el ROC-AUC de 0.6456 a 0.6355 y el Average Precision de 0.3310 a 0.3053. En conjunto, estas variaciones reflejan una degradación moderada de la capacidad discriminativa sobre el período futuro, aunque sin alterar sustancialmente la proporción de retrasos reales detectados mediante la regla de decisión fijada previamente.

La principal limitación se concentra en la generación de falsas alertas. En el test externo, el modelo clasificó como positivos aproximadamente el 66.98 % de los vuelos, mientras que la prevalencia real de retrasos fue del 21.17 %. De las alertas positivas generadas, el 74.73 % correspondió a vuelos que finalmente no presentaron un retraso igual o superior a 15 minutos. Por tanto, la elevada sensibilidad se consigue a costa de una baja especificidad y de una carga considerable de falsos positivos.

Este comportamiento condiciona la interpretación funcional del modelo. Los resultados no respaldan su utilización como mecanismo determinista capaz de afirmar que un vuelo concreto sufrirá un retraso. Su aplicación resulta más coherente como componente de un sistema de alerta temprana que identifica vuelos con un riesgo modelado suficientemente elevado para justificar una advertencia preventiva. Bajo esta interpretación, una clasificación positiva representa una señal de riesgo y no la certeza de que el retraso vaya a producirse.

La reducida precision tampoco puede atribuirse exclusivamente al umbral seleccionado. Durante la validación de 2025, utilizar el umbral convencional de 0.5 incrementaba la precision únicamente hasta el 30.79 %, mientras que el recall descendía hasta el 59.44 %. Esto evidencia que elevar el umbral puede reducir parcialmente las falsas alertas, pero produce simultáneamente una pérdida considerable en la detección de retrasos reales. La mejora simultánea de ambas dimensiones requeriría, por tanto, incrementar la capacidad discriminativa del modelo y no únicamente modificar su regla de decisión.

Entre las posibles líneas de mejora se encuentra la incorporación de información adicional que pueda conocerse legítimamente antes de la salida, siempre bajo un control estricto de fuga de información. Variables relacionadas con condiciones meteorológicas disponibles en el momento de predicción, niveles recientes de congestión aeroportuaria, patrones históricos de rutas o compañías y otra información operacional disponible anticipadamente podrían aportar señales adicionales. Asimismo, futuras investigaciones podrían evaluar modelos no lineales de mayor capacidad, siempre que su coste computacional y su interpretabilidad sean compatibles con los objetivos del sistema.

Estas posibilidades deben considerarse trabajo futuro y no modificaciones del experimento principal. Dado que el período externo de 2026 ya ha sido observado, cualquier nueva configuración desarrollada a partir del conocimiento adquirido tras esta evaluación tendría carácter post-test y requeriría un nuevo período completamente independiente para realizar una nueva evaluación confirmatoria.

En conjunto, la evaluación externa proporciona evidencia favorable sobre la estabilidad temporal de la capacidad de detección del sistema, pero también identifica una degradación moderada de su capacidad discriminativa y confirma que la generación de falsas alertas constituye su principal limitación. El modelo debe interpretarse, por tanto, como una herramienta de priorización de riesgo con elevada sensibilidad, cuyo principal margen de mejora reside en aumentar la precision sin comprometer sustancialmente la cobertura de retrasos reales.